# StreamConMamba Multilingual Distillation — Kaggle 2×T4

## Research-backed refactor: StreamConMamba multilingual KD

This revision combines the useful parts of the supplied papers while preserving
strict causal inference and the 2×T4/edge budget:

- [Fast Streaming Transducer ASR with Whisper KD](https://aclanthology.org/2024.findings-emnlp.976/)
  motivates sequence-level Whisper pseudo labels, hallucination/rate filtering,
  and training across several chunk sizes.  This notebook keeps those data gates,
  cycles 2/4/8-second chunk targets for newly generated caches, and softly weights
  accepted CTC labels by Whisper confidence.
- [DuplexMamba](https://arxiv.org/html/2502.11123v3) motivates CNN→ConMamba speech
  encoding, slice-based streaming, and fixed-size Mamba state.  Its encoder is
  bidirectional and becomes streaming only through 3-second slicing; this student
  instead keeps every temporal operator causal and reports a bounded-left-context
  streaming benchmark.
- [Samba-ASR](https://arxiv.org/html/2501.02832v2) supports the log-mel→CNN→Mamba
  encoder pattern, but its Mamba cross-decoder is insufficiently specified for a
  safe edge implementation.  The notebook retains encoder-only CTC decoding.
- The original [ConMamba study](https://arxiv.org/html/2407.09732) found the
  18-layer, 256-wide, 40-ms-token encoder-only CTC model competitive at ~31.6M
  parameters, while warning that ASR speed gains appear mainly for long inputs.
- [Multilingual DistilWhisper](https://arxiv.org/html/2311.01070) shows that joint
  ASR+KD and small language-specific routes help the multilingual capacity gap.
  Here that idea becomes a tiny **frame-causal, automatically predicted**
  language router plus rank-16 residual experts—no oracle language is required.

The final objective is multi-depth masked cosine feature KD + confidence-weighted
CTC + auxiliary language identification.  Whisper cache values are encoder hidden
states rather than logits, so applying softmax/KL or JS to them would be invalid.


In [1]:
# Verify Kaggle GPU environment + capture key versions for the rest of the cells.
import torch
import subprocess
import sys
import os

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cc = f"sm_{p.major}{p.minor}"
    print(f"  GPU {i}: {p.name:20s}  {p.total_memory/1e9:5.1f} GB  {cc}")

# nvcc — needed for source-build of mamba-ssm / causal-conv1d.
try:
    out = subprocess.check_output(["nvcc", "--version"], text=True)
    nvcc_line = [l for l in out.splitlines() if "release" in l.lower()]
    print(f"nvcc: {nvcc_line[0].strip() if nvcc_line else out.splitlines()[-1]}")
except Exception as e:
    print(f"nvcc: NOT FOUND ({e}). Source-build of mamba kernels will fail; "
          "fall back to slow Python path or use prebuilt wheels.")

if torch.cuda.device_count() > 0:
    cc = torch.cuda.get_device_capability(0)
    assert cc[0] >= 7, (
        f"GPU compute capability {
            cc[0]}.{
            cc[1]} is too old for mamba-ssm CUDA kernels. "
        "Switch the Kaggle accelerator to 'GPU T4 x2' (sm_75) instead of P100 (sm_60)."
    )


Python: 3.12.12
PyTorch: 2.10.0+cu128
CUDA (torch): 12.8
cuDNN: 91002
GPU count: 2
  GPU 0: Tesla T4               15.6 GB  sm_75
  GPU 1: Tesla T4               15.6 GB  sm_75
nvcc: Cuda compilation tools, release 12.8, V12.8.93


## Install requirements

In [2]:
!apt-get install -y -qq ffmpeg libavcodec-extra > /dev/null 2>&1
# Don't pin transformers — the Kaggle default (currently v5.x) works with
# the mamba-ssm fast-path kernels. A hard pin gets silently overridden by
# huggingface_hub / openai-whisper anyway, so it just adds confusion.
!pip install -q -U transformers
!pip install -q "datasets[audio]<4.0.0"
!pip install -q sentencepiece jiwer evaluate
!pip install -q WeTextProcessing
!pip install -q -U openai-whisper
!pip install -q --upgrade huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 110.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that ar

In [3]:
# ── Mamba CUDA kernels — auto-resolve for the actual env ─────────
import sys, torch, importlib, subprocess

cc_major = torch.cuda.get_device_capability(0)[0]
assert cc_major >= 7, (
    f"GPU compute capability {cc_major}.x is too old for mamba-ssm CUDA kernels. "
    "Switch the Kaggle accelerator to 'GPU T4 x2' (sm_75) instead of P100 (sm_60)."
)

# --no-build-isolation
!pip install -q --no-build-isolation ninja packaging wheel
!pip install -q --no-build-isolation causal-conv1d
!pip install -q --no-build-isolation mamba-ssm
!pip install -q speechbrain
#   !pip install -q "speechbrain==1.0.3"


def _check():
    failures = []
    try:
        from causal_conv1d import causal_conv1d_fn, causal_conv1d_update  # noqa
    except Exception as e:
        failures.append(f"causal_conv1d import failed: {e}")
    try:
        from mamba_ssm.ops.selective_scan_interface import (
            selective_scan_fn, mamba_inner_fn,
        )  # noqa
        from mamba_ssm.ops.triton.selective_state_update import selective_state_update  # noqa
    except Exception as e:
        failures.append(f"mamba_ssm import failed: {e}")
    try:
        import speechbrain  # noqa
    except Exception as e:
        failures.append(f"speechbrain import failed: {e}")
    return failures

problems = _check()
if problems:
    print("Fast-path NOT available. Reasons:")
    for p in problems:
        print(" -", p)
    raise RuntimeError("Mamba CUDA kernels did not import; fix before continuing.")
print("mamba-ssm + causal-conv1d kernels importable — fast path enabled")

  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.7/767.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 20.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver 

In [4]:
# Should not raise ImportError
from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn, selective_scan_fn
from mamba_ssm.ops.triton.selective_state_update import selective_state_update


In [5]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
wandb_key = user_secrets.get_secret("wandb_api_key")
login(token=hf_token)


## Config pipeline

In [ ]:
import os
import gc
import json
import math
import time
import random
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List
import torch
import torchaudio
import torch
import functools
from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperProcessor, AutoModelForSpeechSeq2Seq, AutoProcessor, MambaForCausalLM, MambaConfig
from datasets import load_dataset, Audio
import datasets
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

print = functools.partial(print, flush=True)
print(datasets.__version__)

# Hardware-aware helpers
if torch.cuda.device_count() > 0:
    NUM_GPUS = torch.cuda.device_count()
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    IS_T4 = "T4" in torch.cuda.get_device_name(0)


_KAGGLE_INPUT_ROOT = "/kaggle/input"
_KAGGLE_DATASET_OWNER = "leviettrieu369"

# Explicit read-only checkpoint bundle used to resume this run.
_RESUME_CHECKPOINT_ROOT = (
    "/kaggle/input/datasets/leviettrieu369/distillation-checkpoint/edge_asr"
)


def _kaggle_dataset_roots(slug):
    """Return supported mount roots, newest Kaggle layout first."""
    return [
        os.path.join(_KAGGLE_INPUT_ROOT, "datasets", _KAGGLE_DATASET_OWNER, slug),
        os.path.join(_KAGGLE_INPUT_ROOT, slug),
        os.path.join(_KAGGLE_INPUT_ROOT, _KAGGLE_DATASET_OWNER, slug),
    ]


def _resolve_kaggle_dir(slug, *relative_layouts):
    """Resolve a dataset directory without ever creating it under /kaggle/input."""
    layouts = relative_layouts or ("",)
    candidates = [
        os.path.normpath(os.path.join(root, relative))
        for root in _kaggle_dataset_roots(slug)
        for relative in layouts
    ]
    for candidate in candidates:
        if os.path.isdir(candidate):
            return candidate
    # Keep a deterministic expected path for diagnostics when the dataset is
    # not attached. Consumers fail with the full path instead of silently
    # falling back to a writable directory containing different data.
    return candidates[0]


def _resolve_kaggle_prefix(slug, *relative_prefixes):
    """Resolve a file prefix such as SentencePiece's path without `.model`."""
    candidates = [
        os.path.normpath(os.path.join(root, relative))
        for relative in relative_prefixes
        for root in _kaggle_dataset_roots(slug)
    ]
    for candidate in candidates:
        if os.path.isfile(candidate) or os.path.isfile(candidate + ".model"):
            return candidate
    return candidates[0]


def _dir_has_suffix(directory, suffix):
    if not os.path.isdir(directory):
        return False
    try:
        return any(entry.is_file() and entry.name.endswith(suffix)
                   for entry in os.scandir(directory))
    except OSError:
        return False


def _prefer_populated_dir(work_dir, input_dir, suffix):
    """Prefer newly generated artifacts only when they are actually present."""
    return work_dir if _dir_has_suffix(work_dir, suffix) else input_dir


def _cache_tree_has_shards(directory):
    if not os.path.isdir(directory):
        return False
    try:
        for entry in os.scandir(directory):
            if entry.is_dir() and _dir_has_suffix(entry.path, ".npz"):
                return True
    except OSError:
        pass
    return False


def _select_cache_pair(work_teacher, work_mel, input_teacher, input_mel):
    """Never mix one side of a generated cache with one side of an input cache."""
    if (_cache_tree_has_shards(work_teacher) and
            _cache_tree_has_shards(work_mel)):
        return work_teacher, work_mel
    return input_teacher, input_mel


@dataclass
class Config:
    # Storage (Kaggle working dir, persists across saves)
    root: str = "/kaggle/working/edge_asr"

    # Teacher
    teacher_id: str = "openai/whisper-large-v3"
    # T4 does not support flash_attention_2
    teacher_attn: str = "sdpa"
    teacher_d: int = 1280

    # Pseudo Labeling
    psuedo_batch: int = 8
    max_label_len: int = 128
    max_samples_per_dataset: int = 30_000

    # Filtering config
    # loop filter
    max_repeats: int = 3

    # glitch filter
    max_word_len: int = 20

    # speed filter (words per second)
    min_wps: float = 1.0
    max_wps: float = 4.0

    # WER/CER gate (percentage)
    wer_threshold: float = 10.0

    # all-caps minimum length
    allcaps_min_len: int = 5

    # Tokenizer
    spm_vocab: int = 5000
    spm_coverage: float = 0.9995
    vocab_size: int = 5001     # SP pieces + 1 CTC blank; re-derived from the tokenizer at load time
    force_retokenize_targets: bool = True
    allow_tokenizer_rebuild: bool = True

    # Student
    mamba_pretrained: str = "state-spaces/mamba-130m-hf"
    mamba_d: int = 768
    cnn_ch: int = 768
    cnn_ks: int = 5
    n_mels: int = 80
    sr: int = 16000

    # StreamConMamba encoder: unidirectional, T4-sized, trained from scratch.
    # 18 x 256 mirrors the strongest encoder-only CTC setting in ConMamba while
    # remaining around the practical edge-size target.  All temporal operators
    # are causal; do not enable the removed bidirectional path for this recipe.
    cm_d_model: int = 256
    cm_layers: int = 18
    cm_d_ffn: int = 1024
    cm_d_state: int = 16
    cm_expand: int = 2
    cm_d_conv: int = 4
    cm_kernel: int = 31          # causal Conformer conv-module kernel (odd)
    cm_bidirectional: bool = False

    # Multi-depth representation transfer.  Early/middle/final student states
    # independently regress Whisper's final encoder state.  The last projection
    # retains the historical `kl_head` name so an old KD checkpoint can seed the
    # new model; auxiliary projections start fresh.
    kd_tap_layers: tuple = (5, 11, 17)
    kd_tap_weights: tuple = (0.20, 0.30, 0.50)

    # Tiny automatic language router.  It predicts language per causal frame and
    # selects low-rank residual experts without requiring an oracle language at
    # deployment.  `num_languages`/`language_names` are derived from the cache.
    num_languages: int = 1
    language_names: tuple = ('<unk>',)
    lang_expert_rank: int = 16
    lang_expert_scale: float = 0.5
    language_balance_alpha: float = 0.5  # p(lang) proportional to n_lang**alpha
    a_lang: float = 0.05

    # Whisper avg_logprob is converted to confidence in [0, 1].  Filtering is
    # still the primary quality gate; this only softens CTC gradients from the
    # noisier accepted pseudo labels.
    confidence_floor: float = 0.35
    confidence_power: float = 0.5

    # Mel path: cached 100 fps log-mel -> 4x causal subsample -> ~25 fps.
    n_fft: int = 400
    hop: int = 160
    win: int = 400

    # Chunk cache is CTC-safe only if the collator pads, never crops.  New cache
    # builds cycle through several chunk targets (paper-style multi-chunk data),
    # while existing fixed-8 s caches remain valid for training.
    use_chunk_cache: bool = True
    chunk_seconds: float = 8.0       # legacy/fallback scalar
    chunk_seconds_choices: tuple = (2.0, 4.0, 8.0)
    chunk_overlap: float = 1.0
    allow_audio_crop_for_ctc: bool = False

    # Bounded-memory deployment simulation.  Values are multiples of the 4x
    # subsampling factor so block boundaries remain frame-aligned.
    stream_chunk_ms: int = 1280
    stream_left_context_ms: int = 2560
    stream_eval_samples: int = 200

    # Use the scratch recipe and resume scratch_kd_latest.pt when present.
    # This flag selects the lineage; it does not force random initialization.
    train_from_scratch: bool = True
    # Require the attached checkpoint bundle before launching training.
    resume_from_checkpoint: bool = True

    # Default objective weights; StageSpec owns the actual curriculum.
    a_kl: float = 1.0
    a_ctc: float = 0.30
    ctc_blank_bias_init: float = 0.0
    # temp: float = 2.0

    # Stage names are legacy; CTC alignment is learned in the chunk stage.
    fr_epochs: int = 0
    fr_lr: float = 3e-4
    fr_bs: int = 16  # batch size
    fr_ga: int = 2
    max_s1: float = 10.0

    # Unfrozen stage
    un_epochs: int = 8
    un_lr: float = 5e-5
    un_bs: int = 4
    un_ga: int = 8
    gc_norm: float = 1.0
    warmup: int = 500
    max_s2: float = 12.0   # safety limit only; no CTC audio cropping

    # Datasets
    datasets: List[tuple] = field(default_factory=lambda: [
        # ('facebook/multilingual_librispeech', 'french',  'train', 'transcript', 'french',  True),
        # ('facebook/multilingual_librispeech', 'spanish', 'train', 'transcript', 'spanish', True),
        # ('facebook/multilingual_librispeech', 'german', 'train', 'transcript', 'german',  True),
        # ('fsicoli/common_voice_22_0', 'vi', 'train', 'sentence', 'vietnamese', True),
        # ('fsicoli/common_voice_22_0', 'ja', 'train', 'sentence', 'japanese',   True),
        # ('fsicoli/common_voice_22_0', 'ko', 'train', 'sentence', 'korean',     True),
        # ('fsicoli/common_voice_22_0', 'zh-CN', 'train', 'sentence', 'chinese', True),
        # ('fsicoli/common_voice_22_0', 'en', 'train', 'sentence', 'english', True),
        # ('nguyendv02/ViMD_Dataset', 'default', 'train', 'text', 'vietnamese', True),
        # ('pnnbao-ump/VieNeu-TTS', 'default', 'train', 'text', 'vietnamese', True)
    ])

    # Regularization
    dropout: float = 0.1   # applied between Mamba output and heads

    # Early stopping (per-stage, monitors val/loss after each epoch)
    es_enabled: bool = True
    es_patience: int = 3    # epochs without improvement before stopping
    es_min_delta: float = 1e-4  # minimum drop in val metric to count as improvement
    es_metric: str = 'loss'     # 'loss' (val total loss) or 'wer'

    # The shipped unfrozen baseline is known-bad: its CTC head collapsed to a
    # frequent non-blank token after the old KD+CTC/-5 blank-bias recipe. On a
    # fresh Kaggle session, ignore only that read-only unfrozen checkpoint and
    # restart Stage 2 from frozen_latest with a clean CTC head; same-session
    # writable unfrozen checkpoints still resume normally.
    ignore_readonly_unfrozen_baseline: bool = True

    seed: int = 42
    workers: int = 2      # Kaggle has fewer CPU cores than Colab HM
    log_every: int = 50

    def __post_init__(self):
        # Immutable Kaggle inputs. Support both the owner-qualified mount
        # layout and the classic /kaggle/input/<dataset-slug> layout.
        self.pseudo_dir = _resolve_kaggle_dir('pseudo-labeling')
        self.filter_dir = _resolve_kaggle_dir('filtering-transcription')
        self.cache_dir = _resolve_kaggle_dir('teacher-cache', 'teacher_cache')
        # Companion 80-bin log-mel cache (shards 1:1 aligned with teacher cache).
        self.mel_dir = _resolve_kaggle_dir('mel-cache', 'mel_cache')

        # Writable staging trees for fresh chunk generation. Never point these
        # at /kaggle/input: Kaggle input datasets are immutable mounts.
        self.chunk_pseudo_input_dir = self.pseudo_dir
        self.chunk_filter_input_dir = self.filter_dir
        self.chunk_pseudo_work_dir = os.path.join(self.root, 'chunk_pseudo')
        self.chunk_filter_work_dir = os.path.join(self.root, 'chunk_filter')
        self.chunk_teacher_work_dir = os.path.join(
            self.root, 'chunk_teacher_cache', 'teacher_cache')
        self.chunk_mel_work_dir = os.path.join(
            self.root, 'chunk_mel_cache', 'mel_cache')
        # Backward-compatible names used in older notebook commentary/cells.
        self.chunk_pseudo_dir = self.chunk_pseudo_work_dir
        self.chunk_filter_dir = self.chunk_filter_work_dir

        # Prefer the tokenizer shipped with the checkpoint bundle so the
        # model and tokenizer travel together. Fall back to the standalone
        # tokenizer dataset for older Kaggle attachments.
        _bundle_spm_prefix = os.path.join(
            _RESUME_CHECKPOINT_ROOT, 'tokenizer', f'spm_{self.spm_vocab}')
        if (os.path.isfile(_bundle_spm_prefix + '.model') or
                os.path.isfile(_bundle_spm_prefix)):
            self.spm_prefix = _bundle_spm_prefix
        else:
            self.spm_prefix = _resolve_kaggle_prefix(
                'tokenizer', f'spm_{self.spm_vocab}', 'spm')

        # Prefer the explicit attached bundle; older mount layouts remain
        # fallbacks, but cannot shadow the requested source.
        self.ckpt_dataset_root = _RESUME_CHECKPOINT_ROOT
        self.ckpt_dataset_roots = [
            _RESUME_CHECKPOINT_ROOT,
            *_kaggle_dataset_roots('distillation-checkpoint'),
        ]
        self.ckpt_load_dir = os.path.join(
            _RESUME_CHECKPOINT_ROOT, 'checkpoints')
        self.ckpt_dir = f'{self.root}/checkpoints'

        # Loss log
        self.loss_log_path = f'{self.root}/training_log.jsonl'

        self.onnx_path = f'{self.root}/student.onnx'

    def safe_name(self, i):
        n, c = self.datasets[i][0].split('/')[-1], self.datasets[i][1]
        return f'{n}_{c}'


C = Config()


def _discover_checkpoint_load_dirs(configured):
    """Find prior-session checkpoints across known Kaggle mount layouts.

    A saved Version may be mounted as either edge_asr/checkpoints or
    results/edge_asr/checkpoints. Keep all populated matches: compatibility
    checks later can skip a stale recipe in one source and continue to another.
    """
    dataset_roots = list(getattr(C, 'ckpt_dataset_roots', []))
    dataset_roots.insert(0, getattr(C, 'ckpt_dataset_root', None))
    candidates = [configured]
    for root in dataset_roots:
        if not root:
            continue
        candidates.extend([
            root,
            os.path.join(root, 'checkpoints'),
            os.path.join(root, 'edge_asr', 'checkpoints'),
            os.path.join(root, 'results', 'edge_asr', 'checkpoints'),
            os.path.join(root, 'checkpoints'),
        ])

    populated = []
    seen = set()
    for path in candidates:
        normalized = os.path.normpath(path)
        if normalized in seen:
            continue
        seen.add(normalized)
        if not os.path.isdir(normalized):
            continue
        pt_names = [name for name in os.listdir(normalized) if name.endswith('.pt')]
        if pt_names:
            populated.append((normalized, set(pt_names)))

    # A current recover_kd checkpoint is the most valuable resume source.
    # File count breaks ties without opening several hundred MB checkpoints.
    populated.sort(
        key=lambda item: (
            'scratch_joint_latest.pt' in item[1],
            'scratch_joint_best.pt' in item[1],
            'scratch_kd_latest.pt' in item[1],
            'recover_kd_latest.pt' in item[1],
            'recover_kd_best.pt' in item[1],
            'recover_ctc_latest.pt' in item[1],
            len(item[1]),
        ),
        reverse=True,
    )
    return [path for path, _ in populated]


C.ckpt_load_dirs = _discover_checkpoint_load_dirs(C.ckpt_load_dir)
if C.ckpt_load_dirs:
    # Backward-compatible primary source for tokenizer and W&B lookup. Model
    # checkpoint loading uses the full list and therefore cannot be shadowed.
    C.ckpt_load_dir = C.ckpt_load_dirs[0]

# Only mkdir writable destinations; /kaggle/input is read-only.


def _is_writable_path(p):
    if not p:
        return False
    path = os.path.abspath(os.path.normpath(os.fspath(p)))
    input_root = os.path.abspath(os.path.normpath(_KAGGLE_INPUT_ROOT))
    try:
        return os.path.commonpath([path, input_root]) != input_root
    except ValueError:
        return True


for d in [
    C.root,
    C.ckpt_dir,
    os.path.dirname(C.loss_log_path),
    # C.chunk_pseudo_dir,
    # C.chunk_filter_dir,
    # os.path.dirname(C.chunk_teacher_work_dir),
    # os.path.dirname(C.chunk_mel_work_dir),
]:
    if _is_writable_path(d):
        os.makedirs(d, exist_ok=True)

# Sanity report on checkpoint source availability.
if C.ckpt_load_dirs:
    print(f'Checkpoint source(s), in priority order: {len(C.ckpt_load_dirs)}')
    for _source_dir in C.ckpt_load_dirs:
        _existing = sorted(
            f for f in os.listdir(_source_dir) if f.endswith('.pt'))
        print(f'  {_source_dir}')
        print(
            f'    Found {len(_existing)} .pt file(s): '
            f'{_existing[:5]}{"..." if len(_existing) > 5 else ""}')
else:
    print('Checkpoint source not found in either supported Kaggle layout:')
    print(f'  configured={C.ckpt_load_dir}')
    print('  Training can start only if the requested stage does not require a source.')
print(f'Checkpoint destination (writable): {C.ckpt_dir}')
print(f'Loss log path: {C.loss_log_path}')

random.seed(C.seed)


def flush():
    """Free GPU memory on ALL devices."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        for i in range(torch.cuda.device_count()):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            total = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f'  [GPU {i}] {alloc:.1f}/{total:.0f} GB')


print(f'{len(C.datasets)} datasets, root: {C.root}')
if torch.cuda.device_count() > 0:
    print(f'Hardware : {NUM_GPUS}× {torch.cuda.get_device_name(0)}')
print(f'Precision: float16  |  Attention: {C.teacher_attn}')
print(f'Frozen: bs={C.fr_bs} × GA={C.fr_ga} = {C.fr_bs*C.fr_ga} eff')
print(f'Unfrozen: bs={C.un_bs} × GA={C.un_ga} = {C.un_bs*C.un_ga} eff')


## True multi-chunk cache generation

Run these cells to build transcript-aligned streaming examples. New manifests
cycle `C.chunk_seconds_choices = (2, 4, 8)` seconds while keeping Whisper
timestamp segments indivisible; existing fixed-8-second cache pairs remain
load-compatible.

Order:
1. Generate timestamped Whisper large-v3 pseudo labels in `C.chunk_pseudo_work_dir`.
2. Build filtered, CTC-valid manifests in `C.chunk_filter_work_dir`.
3. Build aligned teacher/mel shards in `C.chunk_teacher_work_dir` and
   `C.chunk_mel_work_dir`.
4. Upload both trees as immutable Kaggle datasets and point `C.cache_dir` and
   `C.mel_dir` to the matching pair.


### Chunk timestamp pseudo labels

Runs Whisper large-v3 once per original audio file and stores timestamped segments. This is source-level labeling, not training data yet; the next cell groups whole timestamped segments into true chunk examples.


In [7]:
# Chunk-level timestamped pseudo labels.
# This reruns Whisper because the old pseudo labels did not store timestamps.
# Outputs are source-level JSONL files; the next cell turns them into chunk rows.

import math
from pathlib import Path
from tqdm.auto import tqdm

RUN_CHUNK_PSEUDO = False
CHUNK_PSEUDO_SESSION_HOURS = 10.5
CHUNK_PSEUDO_OVERWRITE = False
CHUNK_PSEUDO_OUT = C.chunk_pseudo_work_dir
if RUN_CHUNK_PSEUDO:
    if not _is_writable_path(CHUNK_PSEUDO_OUT):
        raise RuntimeError(f'CHUNK_PSEUDO_OUT must be writable: {CHUNK_PSEUDO_OUT}')
    os.makedirs(CHUNK_PSEUDO_OUT, exist_ok=True)
else:
    print(f'RUN_CHUNK_PSEUDO=False; writable output would be {CHUNK_PSEUDO_OUT}')

_CHUNK_LANG_CODES = {
    'english': 'en',
    'french': 'fr',
    'spanish': 'es',
    'german': 'de',
    'vietnamese': 'vi',
    'japanese': 'ja',
    'korean': 'ko',
    'chinese': 'zh',
    'en': 'en',
    'fr': 'fr',
    'es': 'es',
    'de': 'de',
    'vi': 'vi',
    'ja': 'ja',
    'ko': 'ko',
    'zh': 'zh',
    'zh-CN': 'zh',
}


def _chunk_whisper_model_name(teacher_id):
    name = teacher_id.split('/')[-1]
    return name.replace('whisper-', '')


def _chunk_lang_code(lang):
    return _CHUNK_LANG_CODES.get(
        str(lang), _CHUNK_LANG_CODES.get(str(lang).lower(), None))


def _safe_float(v, default=None):
    try:
        if v is None:
            return default
        out = float(v)
        if math.isnan(out):
            return default
        return out
    except Exception:
        return default


def _segment_payload(seg):
    text = str(seg.get('text', '')).strip()
    return {
        'start': _safe_float(seg.get('start'), 0.0),
        'end': _safe_float(seg.get('end'), 0.0),
        'text': text,
        'avg_logprob': _safe_float(seg.get('avg_logprob'), None),
        'no_speech_prob': _safe_float(seg.get('no_speech_prob'), None),
        'compression_ratio': _safe_float(seg.get('compression_ratio'), None),
    }


if RUN_CHUNK_PSEUDO:
    import whisper

    _chunk_pseudo_deadline = time.monotonic() + CHUNK_PSEUDO_SESSION_HOURS * 3600
    _chunk_pseudo_stopped = False

    _whisper_name = _chunk_whisper_model_name(C.teacher_id)
    _whisper_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Loading openai-whisper model: {_whisper_name} on {_whisper_device}')
    chunk_whisper_model = whisper.load_model(_whisper_name, device=_whisper_device)

    for di, (ds_name, ds_cfg, ds_split, text_col, lang, _) in enumerate(C.datasets):
        safe = C.safe_name(di)
        jsonl = os.path.join(CHUNK_PSEUDO_OUT, f'{safe}.jsonl')
        done_flag = os.path.join(CHUNK_PSEUDO_OUT, f'{safe}.done')

        if CHUNK_PSEUDO_OVERWRITE:
            for p in [jsonl, done_flag]:
                if os.path.exists(p):
                    os.remove(p)

        if os.path.exists(done_flag):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: already done')
            continue

        done_indices = set()
        if os.path.exists(jsonl):
            with open(jsonl, 'r', encoding='utf-8') as fin:
                for line in fin:
                    try:
                        done_indices.add(int(json.loads(line)['idx']))
                    except Exception:
                        pass
            print(
                f'[{di+1}/{len(C.datasets)}] {safe}: resume with {len(done_indices)} sources already written')
        else:
            print(f'[{di+1}/{len(C.datasets)}] {safe}: starting')

        ds_stream = load_dataset(
            ds_name, ds_cfg, split=ds_split,
            streaming=True, trust_remote_code=True,
        )
        ds_stream = ds_stream.select_columns(['audio', text_col])
        ds_stream = ds_stream.cast_column('audio', Audio(sampling_rate=C.sr))
        if C.max_samples_per_dataset:
            ds_stream = ds_stream.take(C.max_samples_per_dataset)

        lang_code = _chunk_lang_code(lang)
        written, skipped, errors = 0, 0, 0
        with open(jsonl, 'a', encoding='utf-8', buffering=1) as fout:
            pbar = tqdm(
                enumerate(ds_stream),
                total=C.max_samples_per_dataset,
                desc=f'chunk-pseudo:{safe}',
                unit='src',
            )
            for idx, sample in pbar:
                if time.monotonic() >= _chunk_pseudo_deadline:
                    _chunk_pseudo_stopped = True
                    print('Session deadline reached; JSONL is flushed and resumable.')
                    break
                if idx in done_indices:
                    continue

                try:
                    audio = sample['audio']
                    raw = np.asarray(audio['array'], dtype=np.float32)
                    duration = float(len(raw) / C.sr)
                    if raw.size == 0:
                        skipped += 1
                        continue

                    result = chunk_whisper_model.transcribe(
                        raw,
                        language=lang_code,
                        task='transcribe',
                        fp16=torch.cuda.is_available(),
                        verbose=False,
                        temperature=0.0,
                        condition_on_previous_text=False,
                    )

                    segments = [
                        _segment_payload(s)
                        for s in result.get('segments', [])
                        if str(s.get('text', '')).strip()
                    ]
                    whisper_text = str(result.get('text', '')).strip()
                    if not whisper_text or not segments:
                        skipped += 1
                        continue

                    obj = {
                        'idx': int(idx),
                        'source_id': f'{safe}:{idx}',
                        'original': str(sample.get(text_col, '') or ''),
                        'lang': lang,
                        'duration': round(duration, 3),
                        'whisper': whisper_text,
                        'segments': segments,
                    }
                    fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                    written += 1

                    if (idx + 1) % 100 == 0:
                        pbar.set_postfix_str(
                            f'written={written} skipped={skipped} errors={errors}'
                        )

                except RuntimeError as e:
                    errors += 1
                    if torch.cuda.is_available() and 'out of memory' in str(e).lower():
                        torch.cuda.empty_cache()
                    print(f'  [ERR] {safe} idx={idx}: {e}')
                    if errors > 20:
                        raise RuntimeError(f'Too many Whisper errors for {safe}') from e
                except Exception as e:
                    errors += 1
                    print(f'  [ERR] {safe} idx={idx}: {e}')
                    if errors > 20:
                        raise RuntimeError(f'Too many Whisper errors for {safe}') from e

        if _chunk_pseudo_stopped:
            print(
                f'  {safe}: partial run wrote {written}; rerun this cell to resume -> {jsonl}')
            break
        if errors:
            print(
                f'  {safe}: {errors} sources failed; leaving incomplete so the next run retries them')
            continue
        Path(done_flag).touch()
        print(f'  {safe}: wrote {written}, skipped {skipped}, errors {errors} -> {jsonl}')
else:
    print('RUN_CHUNK_PSEUDO=False; skipping timestamped pseudo-label generation.')


RUN_CHUNK_PSEUDO=False; writable output would be /kaggle/working/edge_asr/chunk_pseudo
RUN_CHUNK_PSEUDO=False; skipping timestamped pseudo-label generation.


### Chunk manifest and filtering

Groups only whole Whisper timestamp segments into chunks. The overlap repeats complete segments in the next chunk, so every chunk transcript still matches the audio span used later by the cache builder.


In [ ]:
# Build true chunk-level manifests from timestamped source pseudo labels.
# Each output line is one CTC training example: audio span + transcript + token IDs.

import collections
import math
import re
import unicodedata
from pathlib import Path
from tqdm.auto import tqdm

RUN_CHUNK_FILTER = False
# Rebuild stale manifests after the old source-level gate.
CHUNK_FILTER_OVERWRITE = True
CHUNK_PSEUDO_IN = _prefer_populated_dir(
    C.chunk_pseudo_work_dir, C.chunk_pseudo_input_dir, '.jsonl')
CHUNK_FILTER_OUT = C.chunk_filter_work_dir

if RUN_CHUNK_FILTER:
    if not _is_writable_path(CHUNK_FILTER_OUT):
        raise RuntimeError(f'CHUNK_FILTER_OUT must be writable: {CHUNK_FILTER_OUT}')
    os.makedirs(CHUNK_FILTER_OUT, exist_ok=True)
    if 'sp' not in globals() or sp.GetPieceSize() != int(C.spm_vocab):
        sp = spm.SentencePieceProcessor()
        if not sp.Load(C.spm_prefix + '.model'):
            raise FileNotFoundError(f'Tokenizer not found: {C.spm_prefix}.model')
    if sp.GetPieceSize() != int(C.spm_vocab):
        raise RuntimeError(
            f'Chunk filter requires {C.spm_vocab} SentencePiece tokens, but '
            f'{C.spm_prefix}.model contains {sp.GetPieceSize()}. Attach the '
            'matching tokenizer dataset or build it in the writable work tree.')
    BLANK = sp.GetPieceSize()
    C.vocab_size = BLANK + 1
    print(f'Chunk filter input: {CHUNK_PSEUDO_IN}')
    print(f'Chunk filter tokenizer: {sp.GetPieceSize()} SP tokens, blank={BLANK}')

try:
    import jiwer
except Exception:
    jiwer = None

_MULTI_SPACE = re.compile(r'\s+')
_PUNCT = re.compile(
    r'[\u3000-\u303F\uFF00-\uFFEF\u2000-\u206F\u2E00-\u2E7F!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~]')
_HAS_CJK = re.compile(
    r'[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF\u3040-\u30FF\u31F0-\u31FF\uAC00-\uD7AF\u1100-\u11FF\u3130-\u318F]')
_CJK_LANGS = {'chinese', 'japanese', 'korean', 'zh', 'ja', 'ko', 'zh-CN'}


def _norm_text(text):
    text = unicodedata.normalize('NFKC', str(text or '')).lower()
    text = _PUNCT.sub(' ', text)
    return _MULTI_SPACE.sub(' ', text).strip()


def _is_cjk(lang, text=''):
    return str(lang) in _CJK_LANGS or bool(_HAS_CJK.search(str(text or '')))


def _edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                prev[j] + 1,
                cur[j - 1] + 1,
                prev[j - 1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]


def _wer(ref, hyp):
    ref, hyp = _norm_text(ref), _norm_text(hyp)
    if not ref:
        return 0.0 if not hyp else 1.0
    if jiwer is not None:
        return float(jiwer.wer(ref, hyp))
    return _edit_distance(ref.split(), hyp.split()) / max(1, len(ref.split()))


def _cer(ref, hyp):
    ref, hyp = _norm_text(ref), _norm_text(hyp)
    if not ref:
        return 0.0 if not hyp else 1.0
    if jiwer is not None:
        return float(jiwer.cer(ref, hyp))
    return _edit_distance(list(ref), list(hyp)) / max(1, len(ref))


def _max_consecutive_repeats(text, cjk=False):
    if cjk:
        tokens = list(_norm_text(text).replace(' ', ''))
    else:
        tokens = _norm_text(text).split()
    if len(tokens) <= 1:
        return len(tokens)
    best = cur = 1
    for i in range(1, len(tokens)):
        if tokens[i] == tokens[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best


def _speech_rate_ok(text, duration_s, cjk=False):
    if duration_s <= 0:
        return False
    if cjk:
        cps = len(_norm_text(text).replace(' ', '')) / duration_s
        return 1.0 <= cps <= 12.0
    wps = len(_norm_text(text).split()) / duration_s
    return C.min_wps <= wps <= C.max_wps


def _has_long_word(text):
    words = _norm_text(text).split()
    return bool(words) and max(len(w) for w in words) > C.max_word_len


def _student_enc_len_from_mel_len(mel_len):
    lens = int(max(1, mel_len))
    k, p, s = C.cnn_ks, C.cnn_ks // 2, 2
    for _ in range(2):
        lens = ((lens + 2 * p - k) // s) + 1
    return max(1, lens)


def _ctc_required_len(token_ids):
    repeats = sum(1 for a, b in zip(token_ids, token_ids[1:]) if a == b)
    return len(token_ids) + repeats


def _safe_float(v, default=None):
    try:
        if v is None:
            return default
        out = float(v)
        if math.isnan(out):
            return default
        return out
    except Exception:
        return default


def _valid_segments(source_obj):
    out = []
    source_duration = _safe_float(source_obj.get('duration'), None)
    for seg in source_obj.get('segments', []):
        text = str(seg.get('text', '')).strip()
        start = _safe_float(seg.get('start'), None)
        end = _safe_float(seg.get('end'), None)
        if source_duration is not None:
            # Whisper timestamps can overshoot the decoded file duration,
            # especially on short Common Voice clips. Clamp at filter time so
            # duration/rate/CTC checks describe the audio we can actually load.
            end = min(end, source_duration)
        if not text or start is None or end is None or end <= start:
            continue
        out.append({
            'start': max(0.0, start),
            'end': max(0.0, end),
            'text': text,
            'avg_logprob': _safe_float(seg.get('avg_logprob'), None),
            'no_speech_prob': _safe_float(seg.get('no_speech_prob'), None),
            'compression_ratio': _safe_float(seg.get('compression_ratio'), None),
        })
    return sorted(out, key=lambda s: (s['start'], s['end']))


def _segment_groups(segments, target_s, overlap_s, max_s):
    """Build transcript-aligned chunks at several streaming resolutions.

    `target_s` may be a scalar (old caches) or a sequence.  Cycling choices is
    deterministic and keeps each Whisper segment indivisible, so every emitted
    transcript still describes exactly the selected audio span.  A single long
    teacher segment may exceed the current target but never `max_s`.
    """
    targets = (target_s,) if isinstance(target_s, (int, float)) else tuple(target_s)
    targets = tuple(float(x) for x in targets if 0 < float(x) <= float(max_s))
    if not targets:
        raise ValueError(f'No valid chunk target in {target_s!r} for max_s={max_s}')

    groups = []
    i = 0
    while i < len(segments):
        current_target = targets[len(groups) % len(targets)]
        start = segments[i]['start']
        end = segments[i]['end']
        j = i
        while j + 1 < len(segments):
            cand_end = max(end, segments[j + 1]['end'])
            if cand_end - start > current_target:
                break
            if cand_end - start > max_s:
                break
            j += 1
            end = cand_end
        group = segments[i:j + 1]
        groups.append(group)

        overlap_start = end - max(0.0, overlap_s)
        next_i = j + 1
        if overlap_s > 0:
            for k in range(i, j + 1):
                if segments[k]['end'] > overlap_start:
                    next_i = k
                    break
        if next_i <= i:
            next_i = j + 1
        i = next_i
    return groups


def _chunk_from_group(source_obj, group, chunk_idx):
    text = _MULTI_SPACE.sub(' ', ' '.join(s['text'].strip() for s in group)).strip()
    start = min(s['start'] for s in group)
    end = max(s['end'] for s in group)
    duration_s = end - start
    avg_logs = [s['avg_logprob'] for s in group if s['avg_logprob'] is not None]
    no_speech = [s['no_speech_prob'] for s in group if s['no_speech_prob'] is not None]
    comp = [s['compression_ratio'] for s in group if s['compression_ratio'] is not None]
    mean_avg_logprob = float(np.mean(avg_logs)) if avg_logs else None
    teacher_conf = float(min(1.0, math.exp(mean_avg_logprob))
                         ) if mean_avg_logprob is not None else None
    source_id = source_obj.get('source_id') or f'source:{source_obj.get("idx")}'
    return {
        'idx': int(source_obj['idx']),
        'chunk_id': f'{source_id}:{chunk_idx:04d}',
        'source_id': source_id,
        'chunk_start_s': round(float(start), 3),
        'chunk_end_s': round(float(end), 3),
        'duration_s': round(float(duration_s), 3),
        'text': text,
        'lang': source_obj.get('lang', 'unknown'),
        'original': source_obj.get('original', ''),
        'full_whisper': source_obj.get('whisper', ''),
        'teacher_confidence': teacher_conf,
        'avg_logprob': mean_avg_logprob,
        'max_no_speech_prob': max(no_speech) if no_speech else None,
        'max_compression_ratio': max(comp) if comp else None,
        'segment_count': len(group),
    }


def _reject_reason(chunk):
    text = chunk['text']
    lang = chunk['lang']
    duration_s = float(chunk['duration_s'])
    cjk = _is_cjk(lang, text)

    if not text.strip():
        return 'empty'
    if duration_s <= 0 or duration_s > C.max_s2:
        return 'duration'
    if chunk.get('avg_logprob') is not None and chunk['avg_logprob'] < -1.0:
        return 'confidence'
    if chunk.get(
            'max_no_speech_prob') is not None and chunk['max_no_speech_prob'] > 0.6:
        return 'no_speech'
    if chunk.get(
            'max_compression_ratio') is not None and chunk['max_compression_ratio'] > 2.4:
        return 'compression'
    if _max_consecutive_repeats(text, cjk=cjk) >= C.max_repeats:
        return 'repeat'
    if not cjk and _has_long_word(text):
        return 'long_word'
    if not _speech_rate_ok(text, duration_s, cjk=cjk):
        return 'speech_rate'

    token_ids = sp.EncodeAsIds(text)
    if not token_ids:
        return 'empty_tokens'
    if max(token_ids) >= BLANK or min(token_ids) < 0:
        return 'token_range'

    mel_len = max(1, int(round(duration_s * C.sr)) // C.hop)
    enc_len = _student_enc_len_from_mel_len(mel_len)
    required = _ctc_required_len(token_ids)
    if required > enc_len:
        return 'ctc_len'

    chunk['token_ids'] = [int(x) for x in token_ids]
    chunk['tok_len'] = len(token_ids)
    chunk['ctc_required_len'] = required
    chunk['mel_len_est'] = mel_len
    chunk['enc_len_est'] = enc_len
    return None


if RUN_CHUNK_FILTER:
    all_stats = {}
    for di, (_ds_name, _ds_cfg, _ds_split, _text_col, lang, _) in enumerate(C.datasets):
        safe = C.safe_name(di)
        print(safe)
        src = os.path.join(CHUNK_PSEUDO_IN, f'{safe}.jsonl')
        print(src)
        dst = os.path.join(CHUNK_FILTER_OUT, f'{safe}.jsonl')
        done_flag = os.path.join(CHUNK_FILTER_OUT, f'{safe}.done')

        if CHUNK_FILTER_OVERWRITE:
            for p in [dst, done_flag]:
                if os.path.exists(p):
                    os.remove(p)

        if os.path.exists(done_flag):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: already filtered')
            continue
        if not os.path.exists(src):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: missing pseudo file {src}; skip')
            continue

        stats = collections.Counter()
        dst_tmp = dst + '.tmp'
        with open(src, 'r', encoding='utf-8') as fin, open(dst_tmp, 'w', encoding='utf-8') as fout:
            for line in tqdm(fin, desc=f'chunk-filter:{safe}', unit='src'):
                stats['sources'] += 1
                source_obj = json.loads(line)
                original = source_obj.get('original', '')
                full_whisper = source_obj.get('whisper', '')
                if original and full_whisper:
                    if _is_cjk(lang, original + full_whisper):
                        source_score = 100.0 * _cer(original, full_whisper)
                    else:
                        source_score = 100.0 * _wer(original, full_whisper)
                    # Source-level WER/CER is only a diagnostic for chunked data.
                    # The full dataset transcript and Whisper's timestamped pass can
                    # differ in casing, elisions, accents, or partial-book boundaries.
                    # Hard-rejecting the whole source here was wiping every chunk for
                    # French/Spanish; rely on chunk-local confidence, no-speech,
                    # compression, rate, token-range, and CTC-length gates instead.
                    if source_score >= C.wer_threshold:
                        stats['source_high_error'] += 1
                else:
                    source_score = None

                segments = _valid_segments(source_obj)
                if not segments:
                    stats['rejected_no_segments'] += 1
                    continue

                for chunk_idx, group in enumerate(_segment_groups(
                    segments,
                    target_s=getattr(C, 'chunk_seconds_choices', C.chunk_seconds),
                    overlap_s=C.chunk_overlap,
                    max_s=C.max_s2,
                )):
                    stats['chunks_total'] += 1
                    chunk = _chunk_from_group(source_obj, group, chunk_idx)
                    chunk['source_error_rate'] = source_score
                    reason = _reject_reason(chunk)
                    if reason is not None:
                        stats[f'rejected_{reason}'] += 1
                        continue
                    fout.write(json.dumps(chunk, ensure_ascii=False) + '\n')
                    stats['kept'] += 1

        os.replace(dst_tmp, dst)
        Path(done_flag).touch()
        all_stats[safe] = dict(stats)
        print(
            f'[{di+1}/{len(C.datasets)}] {safe}: kept {stats["kept"]}/{stats["chunks_total"]} chunks')
        rejected = {k: v for k, v in stats.items() if k.startswith('rejected_')}
        print(f'  rejected: {rejected}')

    summary_path = os.path.join(CHUNK_FILTER_OUT, 'chunk_filter_summary.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(all_stats, f, ensure_ascii=False, indent=2)
    print(f'Chunk filtering summary -> {summary_path}')
else:
    print('RUN_CHUNK_FILTER=False; skipping chunk manifest/filtering.')


### Combined chunk teacher and mel cache

Replays the source audio once, slices each accepted chunk span, and writes the teacher-cache and mel-cache shards in the same order. This is the cache pair used by the refactored CTC training loader.


In [ ]:
# Build aligned chunk-level teacher and mel caches from the filtered chunk manifest.
# The two output trees must be uploaded as Kaggle datasets after this cell finishes:
#   C.chunk_teacher_work_dir -> chunk-teacher-cache/teacher_cache
#   C.chunk_mel_work_dir     -> chunk-mel-cache/mel_cache

from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperFeatureExtractor
from tqdm.auto import tqdm

RUN_CHUNK_CACHE_BUILD = False
CHUNK_CACHE_OVERWRITE = False
CHUNK_CACHE_SHARD_SIZE = 128
CHUNK_CACHE_SESSION_HOURS = 10.5

CHUNK_FILTER_IN = _prefer_populated_dir(
    C.chunk_filter_work_dir, C.chunk_filter_input_dir, '.jsonl')
CHUNK_TEACHER_OUT = C.chunk_teacher_work_dir
CHUNK_MEL_OUT = C.chunk_mel_work_dir

if RUN_CHUNK_CACHE_BUILD:
    for _output_dir in [CHUNK_TEACHER_OUT, CHUNK_MEL_OUT]:
        if not _is_writable_path(_output_dir):
            raise RuntimeError(f'Chunk cache output must be writable: {_output_dir}')
        os.makedirs(_output_dir, exist_ok=True)
    if 'sp' not in globals() or sp.GetPieceSize() != int(C.spm_vocab):
        sp = spm.SentencePieceProcessor()
        if not sp.Load(C.spm_prefix + '.model'):
            raise FileNotFoundError(f'Tokenizer not found: {C.spm_prefix}.model')
    if sp.GetPieceSize() != int(C.spm_vocab):
        raise RuntimeError(
            f'Chunk cache build requires {C.spm_vocab} SentencePiece tokens, but '
            f'{C.spm_prefix}.model contains {sp.GetPieceSize()}.')
    BLANK = sp.GetPieceSize()
    C.vocab_size = BLANK + 1
    print(f'Chunk manifest input: {CHUNK_FILTER_IN}')


def chunk_quantize_to_int8(arr):
    arr = np.asarray(arr, dtype=np.float32)
    fmin, fmax = float(arr.min()), float(arr.max())
    if fmax - fmin < 1e-8:
        return np.zeros_like(arr, dtype=np.int8), np.float32(1.0), np.float32(0.0)
    scale = np.float32((fmax - fmin) / 254.0)
    zero = np.float32(np.round(-127.0 - (fmin / scale)))
    q = np.clip(np.round(arr / scale + zero), -127, 127).astype(np.int8)
    return q, scale, zero


def _existing_npz_count(lang_dir):
    shards = sorted(f for f in os.listdir(lang_dir) if f.endswith(
        '.npz')) if os.path.isdir(lang_dir) else []
    n = 0
    for sf in shards:
        with np.load(os.path.join(lang_dir, sf), allow_pickle=True) as data:
            n += len(data['scales'])
    return n, shards


def _save_chunk_shard(buf, teacher_dir, mel_dir, shard_idx):
    hidden_chunks, h_scales, h_zeros, h_offsets = [], [], [], [0]
    mel_chunks, m_scales, m_zeros, m_offsets = [], [], [], [0]
    mel_lens, texts, source_ids, langs = [], [], [], []
    chunk_start_s, chunk_end_s, duration_s, teacher_conf = [], [], [], []
    token_ids_flat, token_offsets, token_lens = [], [0], []

    for item in buf:
        hq, hs, hz = chunk_quantize_to_int8(item['teacher_h'])
        mq, ms, mz = chunk_quantize_to_int8(item['mel'])

        hidden_chunks.append(hq)
        h_scales.append(hs)
        h_zeros.append(hz)
        h_offsets.append(h_offsets[-1] + hq.shape[0])

        mel_chunks.append(mq)
        m_scales.append(ms)
        m_zeros.append(mz)
        m_offsets.append(m_offsets[-1] + mq.shape[0])

        ids = [int(x) for x in item['token_ids']]
        assert ids and max(ids) < BLANK and min(ids) >= 0
        token_ids_flat.extend(ids)
        token_offsets.append(token_offsets[-1] + len(ids))
        token_lens.append(len(ids))

        mel_lens.append(int(item['mel_len']))
        texts.append(item['text'])
        source_ids.append(item['source_id'])
        langs.append(item['lang'])
        chunk_start_s.append(float(item['chunk_start_s']))
        chunk_end_s.append(float(item['chunk_end_s']))
        duration_s.append(float(item['duration_s']))
        teacher_conf.append(np.nan if item.get('teacher_confidence')
                            is None else float(item['teacher_confidence']))

    teacher_path = os.path.join(teacher_dir, f'shard_{shard_idx:04d}.npz')
    mel_path = os.path.join(mel_dir, f'shard_{shard_idx:04d}.npz')

    teacher_tmp = teacher_path + '.tmp.npz'
    mel_tmp = mel_path + '.tmp.npz'

    np.savez_compressed(
        teacher_tmp,
        hidden_cat=np.concatenate(hidden_chunks, axis=0),
        offsets=np.array(h_offsets, dtype=np.int32),
        scales=np.array(h_scales, dtype=np.float32),
        zeros=np.array(h_zeros, dtype=np.float32),
        mel_lens=np.array(mel_lens, dtype=np.int32),
        texts=np.array(texts, dtype=object),
        token_ids=np.array(token_ids_flat, dtype=np.int32),
        token_offsets=np.array(token_offsets, dtype=np.int32),
        token_lens=np.array(token_lens, dtype=np.int32),
        source_ids=np.array(source_ids, dtype=object),
        chunk_start_s=np.array(chunk_start_s, dtype=np.float32),
        chunk_end_s=np.array(chunk_end_s, dtype=np.float32),
        duration_s=np.array(duration_s, dtype=np.float32),
        langs=np.array(langs, dtype=object),
        teacher_confidence=np.array(teacher_conf, dtype=np.float32),
    )
    np.savez_compressed(
        mel_tmp,
        mel_cat=np.concatenate(mel_chunks, axis=0),
        offsets=np.array(m_offsets, dtype=np.int32),
        scales=np.array(m_scales, dtype=np.float32),
        zeros=np.array(m_zeros, dtype=np.float32),
        mel_lens=np.array(mel_lens, dtype=np.int32),
    )
    # Publish only complete files. If Kaggle stops between these two replaces,
    # the next run promotes the surviving temporary counterpart below.
    os.replace(teacher_tmp, teacher_path)
    os.replace(mel_tmp, mel_path)
    return os.path.getsize(teacher_path), os.path.getsize(mel_path)


def _slice_chunk_audio(raw_audio, start_s, end_s):
    s = max(0, int(round(float(start_s) * C.sr)))
    e = min(len(raw_audio), int(round(float(end_s) * C.sr)))
    if e <= s:
        raise ValueError(
            f'Invalid chunk slice: start={start_s}, end={end_s}, samples={
                len(raw_audio)}')
    return np.asarray(raw_audio[s:e], dtype=np.float32)


def _chunk_teacher_encode(raw_audio):
    inputs = chunk_feat_ext(
        raw_audio,
        sampling_rate=C.sr,
        return_tensors='pt',
        padding='max_length',
        max_length=480000,
    )
    mel_len = max(1, min(len(raw_audio) // C.hop, 3000))
    with torch.inference_mode():
        h = chunk_encoder(
            inputs.input_features.to(chunk_device, dtype=chunk_teacher_dtype)
        ).last_hidden_state
    h_np = h.cpu().float().numpy()[0]
    enc_len = max(1, mel_len // 2)
    return h_np[:enc_len].astype(np.float16), mel_len


chunk_mel_tf = torchaudio.transforms.MelSpectrogram(
    sample_rate=C.sr,
    n_fft=C.n_fft,
    win_length=C.win,
    hop_length=C.hop,
    n_mels=C.n_mels,
    power=2.0,
    pad_mode='constant',
)


def _chunk_mel_sample(raw_audio, expected_mel_len):
    # torchaudio's STFT uses reflect padding when center=True, which requires
    # the waveform to be longer than n_fft//2. Whisper timestamps can produce
    # very short edge chunks, so pad with silence for the transform while keeping
    # mel_len based on the real audio length.
    raw_audio = np.asarray(raw_audio, dtype=np.float32)
    real_samples = len(raw_audio)
    if real_samples < C.n_fft:
        raw_for_mel = np.pad(raw_audio, (0, C.n_fft - real_samples), mode='constant')
    else:
        raw_for_mel = raw_audio

    wav = torch.as_tensor(raw_for_mel, dtype=torch.float32).unsqueeze(0)
    m = chunk_mel_tf(wav)[0]
    m = torch.log(m.clamp(min=1e-10)).transpose(0, 1).contiguous()
    mel_len = max(1, min(real_samples // C.hop, 3000))
    if mel_len != int(expected_mel_len):
        raise ValueError(
            f'mel_len mismatch teacher={expected_mel_len} student={mel_len}')
    m = m[:mel_len]
    if m.shape[0] < mel_len:
        m = torch.cat([m, m.new_zeros(mel_len - m.shape[0], m.shape[1])], 0)
    return m.numpy().astype(np.float16), mel_len


if RUN_CHUNK_CACHE_BUILD:
    _chunk_cache_deadline = time.monotonic() + CHUNK_CACHE_SESSION_HOURS * 3600
    _chunk_cache_stopped = False
    for _name in ['chunk_whisper_model']:
        if _name in globals():
            del globals()[_name]
    flush()

    chunk_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    chunk_teacher_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(
        f'Loading teacher encoder for chunk cache: {
            C.teacher_id} ({chunk_teacher_dtype})')
    chunk_teacher = WhisperForConditionalGeneration.from_pretrained(
        C.teacher_id,
        torch_dtype=chunk_teacher_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
        attn_implementation=C.teacher_attn,
    ).to(chunk_device)
    chunk_teacher.eval()
    chunk_encoder = chunk_teacher.get_encoder()
    chunk_feat_ext = WhisperFeatureExtractor.from_pretrained(C.teacher_id)
    print(
        f'Teacher encoder loaded on {chunk_device}; Whisper mels={
            chunk_feat_ext.feature_size}')

    total_teacher_bytes, total_mel_bytes = 0, 0
    for di, (ds_name, ds_cfg, ds_split, _text_col, lang, _) in enumerate(C.datasets):
        safe = C.safe_name(di)
        manifest = os.path.join(CHUNK_FILTER_IN, f'{safe}.jsonl')
        if not os.path.exists(manifest):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: missing manifest {manifest}; skip')
            continue

        teacher_dir = os.path.join(CHUNK_TEACHER_OUT, safe)
        mel_dir = os.path.join(CHUNK_MEL_OUT, safe)
        os.makedirs(teacher_dir, exist_ok=True)
        os.makedirs(mel_dir, exist_ok=True)
        teacher_done = os.path.join(teacher_dir, '.done')
        mel_done = os.path.join(mel_dir, '.done')

        if CHUNK_CACHE_OVERWRITE:
            for root in [teacher_dir, mel_dir]:
                for name in os.listdir(root):
                    p = os.path.join(root, name)
                    if os.path.isfile(p):
                        os.remove(p)

        entries = [json.loads(line) for line in open(manifest, 'r', encoding='utf-8')]
        if not entries:
            print(f'[{di+1}/{len(C.datasets)}] {safe}: empty manifest')
            continue

        if os.path.exists(teacher_done) and os.path.exists(mel_done):
            print(f'[{di+1}/{len(C.datasets)}] {safe}: already cached ({len(entries)} chunks)')
            continue

        # Recover if interruption happened after publishing one side of a pair.
        for shard_name in set(os.listdir(teacher_dir)) | set(os.listdir(mel_dir)):
            if not shard_name.endswith('.npz.tmp.npz'):
                continue
            final_name = shard_name[:-8]
            teacher_tmp = os.path.join(teacher_dir, shard_name)
            mel_tmp = os.path.join(mel_dir, shard_name)
            teacher_final = os.path.join(teacher_dir, final_name)
            mel_final = os.path.join(mel_dir, final_name)
            if os.path.exists(teacher_final) and os.path.exists(
                    mel_tmp) and not os.path.exists(mel_final):
                os.replace(mel_tmp, mel_final)
            if os.path.exists(mel_final) and os.path.exists(
                    teacher_tmp) and not os.path.exists(teacher_final):
                os.replace(teacher_tmp, teacher_final)

        teacher_n, teacher_shards = _existing_npz_count(teacher_dir)
        mel_n, mel_shards = _existing_npz_count(mel_dir)
        if teacher_n != mel_n or len(teacher_shards) != len(mel_shards):
            raise RuntimeError(
                f'Partial chunk cache mismatch for {safe}: '
                f'teacher={teacher_n}/{len(teacher_shards)} mel={mel_n}/{len(mel_shards)}'
            )

        n_done = teacher_n
        shard_idx = len(teacher_shards)
        if n_done >= len(entries):
            Path(teacher_done).touch()
            Path(mel_done).touch()
            print(f'[{di+1}/{len(C.datasets)}] {safe}: all cached ({shard_idx} shards)')
            continue

        needed_by_idx = {}
        for pos, entry in enumerate(entries):
            if pos < n_done:
                continue
            needed_by_idx.setdefault(int(entry['idx']), []).append((pos, entry))
        max_idx = max(needed_by_idx) if needed_by_idx else -1

        ds_stream = load_dataset(
            ds_name, ds_cfg, split=ds_split,
            streaming=True, trust_remote_code=True,
        )
        ds_stream = ds_stream.select_columns(['audio'])
        ds_stream = ds_stream.cast_column('audio', Audio(sampling_rate=C.sr))

        buf = []
        n_written = n_done
        lang_teacher_bytes, lang_mel_bytes = 0, 0
        pbar = tqdm(
            enumerate(ds_stream),
            total=max_idx + 1,
            desc=f'chunk-cache:{safe}',
            unit='src',
        )
        for stream_idx, sample in pbar:
            if time.monotonic() >= _chunk_cache_deadline:
                _chunk_cache_stopped = True
                print('Session deadline reached; flushing the current cache shard.')
                break
            if stream_idx > max_idx:
                break
            if stream_idx not in needed_by_idx:
                continue

            raw_full = np.asarray(sample['audio']['array'], dtype=np.float32)
            for _pos, entry in needed_by_idx[stream_idx]:
                req_start_s = float(entry['chunk_start_s'])
                req_end_s = float(entry['chunk_end_s'])
                source_duration_s = len(raw_full) / C.sr
                eff_start_s = min(max(0.0, req_start_s), source_duration_s)
                eff_end_s = min(max(eff_start_s, req_end_s), source_duration_s)
                chunk_audio = _slice_chunk_audio(raw_full, eff_start_s, eff_end_s)

                expected_samples = max(1, int(round(float(entry['duration_s']) * C.sr)))
                requested_overshoots_source = req_end_s > source_duration_s + \
                    (1.0 / C.sr)
                # A tiny slice for a normal-duration manifest entry usually means
                # the pseudo JSONL was generated from a different dataset order or
                # config than this load_dataset stream. Common Voice/Whisper edge
                # timestamps can run past the actual clip end; those are safe to
                # clamp because the chunk starts in the right source audio.
                if (
                    expected_samples >= C.n_fft
                    and len(chunk_audio) < max(C.n_fft, int(0.90 * expected_samples))
                    and not requested_overshoots_source
                ):
                    raise RuntimeError(
                        f'{safe} pos={_pos} source_id={
                            entry.get("source_id")} audio slice too short: '
                        f'got {
                            len(chunk_audio)} samples, expected ~{expected_samples}; '
                        f'chunk=[{
                            entry["chunk_start_s"]:.3f}, {
                            entry["chunk_end_s"]:.3f}], '
                        f'source_samples={len(raw_full)}. Check that the pseudo JSONL '
                        f'matches ds_name={
                            ds_name!r}, ds_cfg={
                            ds_cfg!r}, split={
                            ds_split!r}.'
                    )
                teacher_h, mel_len = _chunk_teacher_encode(chunk_audio)
                mel, mel_len_2 = _chunk_mel_sample(chunk_audio, mel_len)
                if mel_len != mel_len_2:
                    raise RuntimeError(f'chunk mel mismatch at {safe} pos={_pos}')

                token_ids = [int(x) for x in entry.get('token_ids', [])]
                if not token_ids or max(token_ids) >= BLANK or min(token_ids) < 0:
                    raise ValueError(
                        f'Bad token_ids in {safe} pos={_pos}: {token_ids[:20]}')

                buf.append({
                    'teacher_h': teacher_h,
                    'mel': mel,
                    'mel_len': mel_len,
                    'text': entry['text'],
                    'token_ids': token_ids,
                    'source_id': entry['source_id'],
                    'chunk_start_s': round(float(eff_start_s), 3),
                    'chunk_end_s': round(float(eff_end_s), 3),
                    'duration_s': round(float(len(chunk_audio) / C.sr), 3),
                    'lang': entry.get('lang', lang),
                    'teacher_confidence': entry.get('teacher_confidence'),
                })
                n_written += 1

                if len(buf) >= CHUNK_CACHE_SHARD_SIZE:
                    tb, mb = _save_chunk_shard(buf, teacher_dir, mel_dir, shard_idx)
                    total_teacher_bytes += tb
                    total_mel_bytes += mb
                    lang_teacher_bytes += tb
                    lang_mel_bytes += mb
                    shard_idx += 1
                    buf.clear()
                    pbar.set_postfix_str(
                        f'{n_written}/{len(entries)} chunks shard={shard_idx}')

        pbar.close()

        if buf:
            tb, mb = _save_chunk_shard(buf, teacher_dir, mel_dir, shard_idx)
            total_teacher_bytes += tb
            total_mel_bytes += mb
            lang_teacher_bytes += tb
            lang_mel_bytes += mb
            shard_idx += 1
            buf.clear()

        if _chunk_cache_stopped:
            print(
                f'[{di+1}/{len(C.datasets)}] {safe}: partial cache {n_written}/{len(entries)}; rerun to resume')
            break
        if n_written != len(entries):
            raise RuntimeError(
                f'{safe}: cached {n_written}/{len(entries)} chunks; source stream ended early')

        Path(teacher_done).touch()
        Path(mel_done).touch()
        print(
            f'[{di+1}/{len(C.datasets)}] {safe}: {len(entries)} chunks, '
            f'{shard_idx} shards, teacher={lang_teacher_bytes/1e9:.2f}GB, '
            f'mel={lang_mel_bytes/1e9:.2f}GB'
        )

    print(
        f'Chunk teacher cache: {CHUNK_TEACHER_OUT} ({
            total_teacher_bytes /
            1e9:.2f}GB written this run)')
    print(
        f'Chunk mel cache:     {CHUNK_MEL_OUT} ({
            total_mel_bytes /
            1e9:.2f}GB written this run)')
    print('Upload those two folders as Kaggle datasets, then rerun training with C.use_chunk_cache=True.')
else:
    print('RUN_CHUNK_CACHE_BUILD=False; skipping chunk cache build.')


### Chunk NPZ cache sanity check

Fast structural check for the freshly generated NPZ shards before uploading them as Kaggle datasets.


In [10]:
# # Lightweight NPZ sanity check before upload.

# if 'sp' not in globals():
#     sp = spm.SentencePieceProcessor()
#     sp.Load(C.spm_prefix + '.model')

# def _first_npz(root):
#     for lang_dir in sorted(os.listdir(root)):
#         path = os.path.join(root, lang_dir)
#         if not os.path.isdir(path):
#             continue
#         shards = sorted(f for f in os.listdir(path) if f.endswith('.npz'))
#         if shards:
#             return os.path.join(path, shards[0])
#     return None


# _teacher_shard = _first_npz(C.chunk_teacher_work_dir) if os.path.isdir(C.chunk_teacher_work_dir) else None
# if _teacher_shard is None:
#     print(f'No generated chunk teacher shards found yet at {C.chunk_teacher_work_dir}')
# else:
#     _mel_shard = _teacher_shard.replace(C.chunk_teacher_work_dir, C.chunk_mel_work_dir, 1)
#     print(f'Checking teacher shard: {_teacher_shard}')
#     print(f'Checking mel shard:     {_mel_shard}')
#     with np.load(_teacher_shard, allow_pickle=True) as td, np.load(_mel_shard, allow_pickle=True) as md:
#         required_teacher = {
#             'hidden_cat', 'offsets', 'scales', 'zeros', 'mel_lens', 'texts',
#             'token_ids', 'token_offsets', 'source_ids', 'chunk_start_s',
#             'chunk_end_s', 'duration_s', 'langs', 'teacher_confidence',
#         }
#         required_mel = {'mel_cat', 'offsets', 'scales', 'zeros', 'mel_lens'}
#         missing_teacher = sorted(required_teacher - set(td.files))
#         missing_mel = sorted(required_mel - set(md.files))
#         assert not missing_teacher, f'Missing teacher fields: {missing_teacher}'
#         assert not missing_mel, f'Missing mel fields: {missing_mel}'
#         assert np.array_equal(td['mel_lens'], md['mel_lens']), 'teacher/mel mel_lens mismatch'
#         assert len(td['texts']) == len(td['scales']) == len(td['source_ids'])
#         assert len(td['token_offsets']) == len(td['texts']) + 1
#         assert td['token_ids'].size == int(td['token_offsets'][-1])
#         assert td['token_ids'].size == 0 or int(td['token_ids'].max()) < sp.GetPieceSize()
#         n_show = min(3, len(td['texts']))
#         print(f'OK: {len(td["texts"])} chunks in shard, mel_lens aligned.')
#         for i in range(n_show):
#             ts, te = int(td['token_offsets'][i]), int(td['token_offsets'][i + 1])
#             print(
#                 f'  {i}: mel_len={int(td["mel_lens"][i])} '
#                 f'tok_len={te-ts} lang={td["langs"][i]} '
#                 f'source={td["source_ids"][i]} '
#                 f'chunk=({float(td["chunk_start_s"][i]):.2f}, {float(td["chunk_end_s"][i]):.2f}) '
#                 f'text={str(td["texts"][i])[:100]}'
#             )


## Distillation  

## ConMamba encoder modules (vendored architecture; SpeechBrain utilities)

The ConMamba layer code is vendored in the notebook. Small activation,
normalization, and dynamic-chunk utility classes still come from `speechbrain`,
which is installed in the environment cell.


In [ ]:
# ConMamba encoder, unidirectional/causal via stock mamba_ssm.Mamba.
# Vendored from .kdedit/conmamba-src/modules/Conmamba.py; keep the
# ConMambaStudent fp32 encoder island for selective-scan stability.
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from speechbrain.nnet.activations import Swish
from speechbrain.nnet.attention import PositionalwiseFeedForward
from speechbrain.nnet.normalization import LayerNorm
from speechbrain.utils.dynamic_chunk_training import DynChunkTrainConfig

from mamba_ssm import Mamba   # unidirectional / causal SSM block

# Fail loudly if the removed bidirectional path is re-enabled.


class _BiMambaRemoved:
    def __init__(self, *a, **k):
        raise RuntimeError(
            'Bidirectional ConMamba was removed (vendored Vim bimamba dropped for '
            'stock mamba_ssm). Set Config.cm_bidirectional=False, or restore the '
            'bimamba / selective_scan_interface blocks to use it.')


BiMamba = _BiMambaRemoved


class ConvolutionModule(nn.Module):
    """This is an implementation of convolution module in Conmamba.
    """

    def __init__(
        self,
        input_size,
        kernel_size=31,
        bias=True,
        activation=Swish,
        dropout=0.0,
        causal=False,
        dilation=1,
    ):
        super().__init__()

        self.kernel_size = kernel_size
        self.causal = causal
        self.dilation = dilation

        if self.causal:
            self.padding = (kernel_size - 1) * 2 ** (dilation - 1)
        else:
            self.padding = (kernel_size - 1) * 2 ** (dilation - 1) // 2

        self.layer_norm = nn.LayerNorm(input_size)
        self.bottleneck = nn.Sequential(
            # pointwise
            nn.Conv1d(
                input_size, 2 * input_size, kernel_size=1, stride=1, bias=bias
            ),
            nn.GLU(dim=1),
        )
        # depthwise
        self.conv = nn.Conv1d(
            input_size,
            input_size,
            kernel_size=kernel_size,
            stride=1,
            padding=self.padding,
            dilation=dilation,
            groups=input_size,
            bias=bias,
        )

        # BatchNorm in the original Conformer replaced with a LayerNorm due to
        # https://github.com/speechbrain/speechbrain/pull/1329
        # see discussion
        # https://github.com/speechbrain/speechbrain/pull/933#issuecomment-1033367884

        self.after_conv = nn.Sequential(
            nn.LayerNorm(input_size),
            activation(),
            # pointwise
            nn.Linear(input_size, input_size, bias=bias),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        dynchunktrain_config: Optional[DynChunkTrainConfig] = None,
    ):
        """Applies the convolution to an input tensor `x`.
        """

        if dynchunktrain_config is not None:
            # chances are chunking+causal is unintended; i don't know where it
            # may make sense, but if it does to you, feel free to implement it.
            assert (
                not self.causal
            ), "Chunked convolution not supported with causal padding"

            assert (
                self.dilation == 1
            ), "Current DynChunkTrain logic does not support dilation != 1"

            # in a causal convolution, which is not the case here, an output
            # frame would never be able to depend on a input frame from any
            # point in the future.

            # but with the dynamic chunk convolution, we instead use a "normal"
            # convolution but where, for any output frame, the future beyond the
            # "current" chunk gets masked.
            # see the paper linked in the documentation for details.

            chunk_size = dynchunktrain_config.chunk_size
            batch_size = x.shape[0]

            # determine the amount of padding we need to insert at the right of
            # the last chunk so that all chunks end up with the same size.
            if x.shape[1] % chunk_size != 0:
                final_right_padding = chunk_size - (x.shape[1] % chunk_size)
            else:
                final_right_padding = 0

            # -> [batch_size, t, in_channels]
            out = self.layer_norm(x)

            # -> [batch_size, in_channels, t] for the CNN
            out = out.transpose(1, 2)

            # -> [batch_size, in_channels, t] (pointwise)
            out = self.bottleneck(out)

            # -> [batch_size, in_channels, lc+t+final_right_padding]
            out = F.pad(out, (self.padding, final_right_padding), value=0)

            # now, make chunks with left context.
            # as a recap to what the above padding and this unfold do, consider
            # each a/b/c letter represents a frame as part of chunks a, b, c.
            # consider a chunk size of 4 and a kernel size of 5 (padding=2):
            #
            # input seq: 00aaaabbbbcc00
            # chunk #1:  00aaaa
            # chunk #2:      aabbbb
            # chunk #3:          bbcc00
            #
            # a few remarks here:
            # - the left padding gets inserted early so that the unfold logic
            #   works trivially
            # - the right 0-padding got inserted as the number of time steps
            #   could not be evenly split in `chunk_size` chunks

            # -> [batch_size, in_channels, num_chunks, lc+chunk_size]
            out = out.unfold(2, size=chunk_size + self.padding, step=chunk_size)

            # as we manually disable padding in the convolution below, we insert
            # right 0-padding to the chunks, e.g. reusing the above example:
            #
            # chunk #1:  00aaaa00
            # chunk #2:      aabbbb00
            # chunk #3:          bbcc0000

            # -> [batch_size, in_channels, num_chunks, lc+chunk_size+rpad]
            out = F.pad(out, (0, self.padding), value=0)

            # the transpose+flatten effectively flattens chunks into the batch
            # dimension to be processed into the time-wise convolution. the
            # chunks will later on be unflattened.

            # -> [batch_size, num_chunks, in_channels, lc+chunk_size+rpad]
            out = out.transpose(1, 2)

            # -> [batch_size * num_chunks, in_channels, lc+chunk_size+rpad]
            out = out.flatten(start_dim=0, end_dim=1)

            # TODO: experiment around reflect padding, which is difficult
            # because small chunks have too little time steps to reflect from

            # let's keep backwards compat by pointing at the weights from the
            # already declared Conv1d.
            #
            # still reusing the above example, the convolution will be applied,
            # with the padding truncated on both ends. the following example
            # shows the letter corresponding to the input frame on which the
            # convolution was centered.
            #
            # as you can see, the sum of lengths of all chunks is equal to our
            # input sequence length + `final_right_padding`.
            #
            # chunk #1:  aaaa
            # chunk #2:      bbbb
            # chunk #3:          cc00

            # -> [batch_size * num_chunks, out_channels, chunk_size]
            out = F.conv1d(
                out,
                weight=self.conv.weight,
                bias=self.conv.bias,
                stride=self.conv.stride,
                padding=0,
                dilation=self.conv.dilation,
                groups=self.conv.groups,
            )

            # -> [batch_size * num_chunks, chunk_size, out_channels]
            out = out.transpose(1, 2)

            out = self.after_conv(out)

            # -> [batch_size, num_chunks, chunk_size, out_channels]
            out = torch.unflatten(out, dim=0, sizes=(batch_size, -1))

            # -> [batch_size, t + final_right_padding, out_channels]
            out = torch.flatten(out, start_dim=1, end_dim=2)

            # -> [batch_size, t, out_channels]
            if final_right_padding > 0:
                out = out[:, :-final_right_padding, :]
        else:
            out = self.layer_norm(x)
            out = out.transpose(1, 2)
            out = self.bottleneck(out)
            out = self.conv(out)

            if self.causal:
                # chomp
                out = out[..., : -self.padding]

            out = out.transpose(1, 2)
            out = self.after_conv(out)

        if mask is not None:
            out.masked_fill_(mask, 0.0)

        return out


class ConmambaEncoderLayer(nn.Module):
    """This is an implementation of Conmamba encoder layer.
    """

    def __init__(
        self,
        d_model,
        d_ffn,
        kernel_size=31,
        activation=Swish,
        bias=True,
        dropout=0.0,
        causal=False,
        mamba_config=None
    ):
        super().__init__()
        assert mamba_config is not None

        bidirectional = mamba_config.pop('bidirectional')
        if causal or (not bidirectional):
            self.mamba = Mamba(
                d_model=d_model,
                **mamba_config
            )
        else:
            self.mamba = BiMamba(
                d_model=d_model,
                bimamba_type='v2',
                **mamba_config
            )
        mamba_config['bidirectional'] = bidirectional

        self.convolution_module = ConvolutionModule(
            d_model, kernel_size, bias, activation, dropout, causal=causal
        )

        self.ffn_module1 = nn.Sequential(
            nn.LayerNorm(d_model),
            PositionalwiseFeedForward(
                d_ffn=d_ffn,
                input_size=d_model,
                dropout=dropout,
                activation=activation,
            ),
            nn.Dropout(dropout),
        )

        self.ffn_module2 = nn.Sequential(
            nn.LayerNorm(d_model),
            PositionalwiseFeedForward(
                d_ffn=d_ffn,
                input_size=d_model,
                dropout=dropout,
                activation=activation,
            ),
            nn.Dropout(dropout),
        )

        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(
        self,
        x,
        src_mask: Optional[torch.Tensor] = None,
        src_key_padding_mask: Optional[torch.Tensor] = None,
        pos_embs: torch.Tensor = None,
        dynchunktrain_config: Optional[DynChunkTrainConfig] = None,
    ):
        conv_mask: Optional[torch.Tensor] = None
        if src_key_padding_mask is not None:
            conv_mask = src_key_padding_mask.unsqueeze(-1)


        # ffn module
        x = x + 0.5 * self.ffn_module1(x)
        # mamba module
        skip = x
        x = self.norm1(x)
        x = self.mamba(x)
        x = x + skip
        # convolution module
        x = x + self.convolution_module(
            x, conv_mask, dynchunktrain_config=dynchunktrain_config
        )
        # ffn module
        x = self.norm2(x + 0.5 * self.ffn_module2(x))
        return x


class ConmambaEncoder(nn.Module):
    """This class implements the Conmamba encoder.
    """

    def __init__(
        self,
        num_layers,
        d_model,
        d_ffn,
        kernel_size=31,
        activation=Swish,
        bias=True,
        dropout=0.0,
        causal=False,
        mamba_config=None
    ):
        super().__init__()
        print(f'dropout={str(dropout)} is not used in Mamba.')

        self.layers = torch.nn.ModuleList(
            [
                ConmambaEncoderLayer(
                    d_model=d_model,
                    d_ffn=d_ffn,
                    dropout=dropout,
                    activation=activation,
                    kernel_size=kernel_size,
                    bias=bias,
                    causal=causal,
                    mamba_config=mamba_config,
                )
                for i in range(num_layers)
            ]
        )
        self.norm = LayerNorm(d_model, eps=1e-6)

    def forward(
        self,
        src,
        src_mask: Optional[torch.Tensor] = None,
        src_key_padding_mask: Optional[torch.Tensor] = None,
        pos_embs: Optional[torch.Tensor] = None,
        dynchunktrain_config: Optional[DynChunkTrainConfig] = None,
        return_intermediate_layers=None,
    ):
        """
        Arguments
        ----------
        src : torch.Tensor
            The sequence to the encoder layer.
        src_mask : torch.Tensor, optional
            The mask for the src sequence.
        src_key_padding_mask : torch.Tensor, optional
            The mask for the src keys per batch.
        pos_embs: torch.Tensor, torch.nn.Module,
            Module or tensor containing the input sequence positional embeddings
            If custom pos_embs are given it needs to have the shape (1, 2*S-1, E)
            where S is the sequence length, and E is the embedding dimension.
        dynchunktrain_config: Optional[DynChunkTrainConfig]
            Dynamic Chunk Training configuration object for streaming,
            specifically involved here to apply Dynamic Chunk Convolution to the
            convolution module.
        """

        requested = set(return_intermediate_layers or ())
        intermediate = []
        output = src
        for layer_idx, enc_layer in enumerate(self.layers):
            output = enc_layer(
                output,
                src_mask=src_mask,
                src_key_padding_mask=src_key_padding_mask,
                pos_embs=pos_embs,
                dynchunktrain_config=dynchunktrain_config,
            )
            if layer_idx in requested:
                intermediate.append((layer_idx, output))
        output = self.norm(output)

        return output, intermediate


In [ ]:
import sentencepiece as spm
from transformers import MambaForCausalLM, MambaConfig
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import torch
import gc
import os
import math
import time
import json
import glob
# Free leftovers from the pseudo-labelling / encoder-cache cells.
for _v in ['t_model', 'enc', 'feat_ext', 'processor', 'proc',
           'model', 'whisper_model', 'feature_extractor', 'student', 'optimizer']:
    if _v in dir():
        try:
            exec(f'del {_v}')
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.ipc_collect()


device = torch.device('cuda')

# Training-time budget. Start it in the Training process cell, *after* setup
# and the smoke gate. A one-hour go/no-go must mean one hour of optimizer
# updates, not ~50 minutes after notebook startup and smoke-test overhead.
# The wall-clock Kaggle session still bounds the kernel; this is the rolling
# checkpoint budget owned by one training-process invocation.
SESSION_HOURS = 1.0
CKPT_EVERY_MINUTES = 30      # rolling save cadence during training
SESSION_START = None
SESSION_DEADLINE = None


def start_training_budget(reset=False):
    """Start one explicit training budget after smoke/setup complete."""
    global SESSION_START, SESSION_DEADLINE
    if SESSION_START is None or reset:
        SESSION_START = time.monotonic()
        SESSION_DEADLINE = SESSION_START + SESSION_HOURS * 3600
        print(
            f'Training budget started: {SESSION_HOURS}h; '
            f'ckpt every {CKPT_EVERY_MINUTES} min')
    return SESSION_DEADLINE


def time_left_seconds():
    # Before the launch cell, expose the full configured budget so helpers
    # remain safe when inspected manually after the setup cell.
    if SESSION_DEADLINE is None:
        return SESSION_HOURS * 3600
    return max(0.0, SESSION_DEADLINE - time.monotonic())


def training_elapsed_seconds():
    if SESSION_START is None:
        return 0.0
    return max(0.0, time.monotonic() - SESSION_START)


def fmt_hms(seconds):
    seconds = int(seconds)
    return f'{seconds // 3600:d}h{(seconds % 3600) // 60:02d}m{seconds % 60:02d}s'


print(
    f'Training budget configured: {SESSION_HOURS}h, '
    'starts in the Training process cell after smoke passes')


# Use absolute C.ckpt_dir so saves do not drift into the kernel CWD.
os.makedirs(C.ckpt_dir, exist_ok=True)

# ── Weights & Biases setup: one resumed run across Kaggle sessions.
WANDB_PROJECT = "edge-asr-distillation"   # ← edit if you want a different project
_wandb_run_filename = (
    'wandb_run_scratch.json' if C.train_from_scratch else 'wandb_run.json')
WANDB_RUN_FILE = os.path.join(C.ckpt_dir, _wandb_run_filename)

# Install if missing (Kaggle base image usually has it, but be safe)
try:
    import wandb
except ImportError:
    os.system("pip install -q wandb")
    import wandb

# Pull API key from Kaggle Secrets; fall back to offline mode if absent.
# To set it: Kaggle notebook  : Add-ons  : Secrets  : add WANDB_API_KEY
if "WANDB_API_KEY" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("wandb_api_key")
        print("WANDB_API_KEY loaded from Kaggle Secrets")
    except Exception as e:
        print(f"No WANDB_API_KEY found ({type(e).__name__}). Logging in OFFLINE mode.")
        print(f"To enable online sync: add WANDB_API_KEY in Add-ons  : Secrets, then rerun.")
        print(f"Or sync afterwards with:  wandb sync /kaggle/working/wandb/<run_dir>")

# Resume from writable run metadata first, then read-only baseline metadata.
prior_run_id = None
_wandb_run_candidates = [WANDB_RUN_FILE]
for _load_dir in (getattr(C, 'ckpt_load_dirs', None) or
                  [getattr(C, 'ckpt_load_dir', None)]):
    if _load_dir:
        _wandb_run_candidates.append(os.path.join(
            _load_dir, _wandb_run_filename))

for _candidate in _wandb_run_candidates:
    if os.path.exists(_candidate):
        try:
            prior_run_id = json.load(open(_candidate))["run_id"]
            print(f"  ↻ Resuming wandb run {prior_run_id} (from {_candidate})")
            break
        except Exception:
            continue

wandb_mode = "online" if "WANDB_API_KEY" in os.environ else "offline"
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    id=prior_run_id,
    resume="allow",
    mode=wandb_mode,
    config={k: v for k, v in vars(C).items()
            if not k.startswith("_") and isinstance(v, (int, float, str, bool, list, tuple))},
    settings=wandb.Settings(start_method="thread"),
)
if prior_run_id is None:
    json.dump({"run_id": wandb_run.id}, open(WANDB_RUN_FILE, "w"))
    print(f"Started new wandb run: {wandb_run.id} (mode={wandb_mode})")
else:
    print(f"Continuing wandb run: {wandb_run.id} (mode={wandb_mode})")
print(f"  URL: {wandb_run.get_url() or '(offline — sync after training)'}")


# ── Local JSONL loss logger: append-only backup of wandb metrics.
class LossLogger:
    """Thin append-only JSONL writer keyed by wandb run + global wall clock."""

    def __init__(self, path, run_id):
        self.path = path
        self.run_id = run_id
        os.makedirs(os.path.dirname(path), exist_ok=True)
        # Open in append mode so resumes preserve history.
        # line_buffering=True flushes on every '\n' (each record is one line).
        self._fh = open(path, 'a', buffering=1, encoding='utf-8')
        n_existing = 0
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for _ in f:
                    n_existing += 1
        self._n_records = n_existing
        # Header record on session start — lets you separate sessions in post-hoc
        # analysis.
        self._fh.write(json.dumps({
            'type': 'session_start',
            'ts': time.time(),
            'run_id': run_id,
            'session_start_iso': time.strftime('%Y-%m-%dT%H:%M:%S'),
        }) + '\n')

    def log(self, record_type, **fields):
        rec = {'type': record_type, 'ts': time.time(), 'run_id': self.run_id}
        rec.update(fields)
        self._fh.write(json.dumps(rec, default=str) + '\n')
        self._n_records += 1

    def close(self):
        try:
            self._fh.flush()
            os.fsync(self._fh.fileno())
        except Exception:
            pass
        self._fh.close()


loss_log = LossLogger(C.loss_log_path, wandb_run.id)
print(f"Loss log: {C.loss_log_path} ({loss_log._n_records - 1} prior records)")

# ── T4 16 GB overrides: smaller micro-batches, same effective batch via GA.
_T4_OVERRIDES = dict(
    fr_bs=4,    # frozen-stage micro-batch
    fr_ga=8,    # frozen-stage grad-accum (effective bs = 4*8 = 32)
    un_bs=2,    # unfrozen micro-batch (was 1 — too noisy for DP)
    un_ga=16,   # unfrozen grad-accum (effective bs = 2*16 = 32)
    max_s1=6.0,  # max audio seconds during frozen stage
    max_s2=8.0,  # max audio seconds during unfrozen stage
)
print('T4 overrides applied:')
for _k, _v in _T4_OVERRIDES.items():
    _old = getattr(C, _k)
    setattr(C, _k, _v)
    print(f'  C.{_k:8s} {_old!r:>8}  ->  {_v!r}')
print(f'  effective batch (frozen)   = {C.fr_bs * C.fr_ga}')
print(f'  effective batch (unfrozen) = {C.un_bs * C.un_ga}')


# Decode + WER helpers (for validation logging)
# Lightweight install of jiwer for clean WER. Falls back to a manual
# Levenshtein implementation if jiwer can't be installed.
try:
    from jiwer import wer as _jiwer_wer
    _HAS_JIWER = True
except ImportError:
    try:
        os.system("pip install -q jiwer")
        from jiwer import wer as _jiwer_wer
        _HAS_JIWER = True
    except Exception:
        _HAS_JIWER = False
        print("    jiwer unavailable, falling back to manual WER computation")


def greedy_ctc_token_ids(ctc_logits, blank_id, lengths=None):
    """Greedy CTC argmax -> collapse repeats -> drop blanks; returns IDs."""
    preds = ctc_logits.argmax(dim=-1)  # [B, T]
    out = []
    for i, p in enumerate(preds):
        seq = p[:lengths[i]].tolist() if lengths is not None else p.tolist()
        cleaned, prev = [], -1
        for tok in seq:
            tok = int(tok)
            if tok != prev and tok != blank_id:
                cleaned.append(tok)
            prev = tok
        out.append(cleaned)
    return out


def greedy_ctc_decode(ctc_logits, sp_model, blank_id, lengths=None):
    """Greedy CTC decode: argmax -> collapse repeats -> drop blanks -> detokenize."""
    return [sp_model.DecodeIds(ids) for ids in greedy_ctc_token_ids(
        ctc_logits, blank_id, lengths)]


def _flatten_id_sequences(seqs):
    flat = []
    for seq in seqs or []:
        if isinstance(seq, torch.Tensor):
            seq = seq.detach().cpu().tolist()
        if isinstance(seq, np.ndarray):
            seq = seq.tolist()
        if isinstance(seq, (int, np.integer)):
            seq = [int(seq)]
        flat.extend(int(x) for x in seq)
    return flat


def compute_collapse_metrics(pred_ids, blank_id, frame_argmax_ids=None,
                             refs=None, hyps=None, sp_model=None,
                             blank_probability_pct=None,
                             empty_threshold=0.95,
                             blank_threshold=0.98,
                             blank_probability_threshold=0.98,
                             top_token_threshold=0.90):
    """Summarize CTC collapse from decoded IDs and raw frame predictions.

    This function inspects one model snapshot. Early CTC commonly has all-blank
    greedy output while its loss is falling, especially with a 5k-way head.
    Therefore blank collapse requires both near-total blank argmax *and* near-
    total blank probability mass. The training canary separately compares loss
    against a pre-training baseline before deciding that progress is terminal.
    """
    pred_lens = [len(seq) for seq in (pred_ids or [])]
    nonblank = [int(x) for seq in (pred_ids or []) for x in seq if int(x) != blank_id]
    frames = _flatten_id_sequences(frame_argmax_ids)
    hyps = hyps or []
    refs = refs or []

    empty_pct = float(np.mean([len(str(h).strip()) == 0 for h in hyps])) if hyps else 0.0
    avg_pred_len = float(np.mean(pred_lens)) if pred_lens else 0.0
    if refs and sp_model is not None:
        avg_ref_len = float(np.mean([len(sp_model.EncodeAsIds(str(r))) for r in refs]))
    else:
        avg_ref_len = 0.0

    top_id = None
    top_ratio = 0.0
    if nonblank:
        vals, counts = np.unique(nonblank, return_counts=True)
        top_idx = int(np.argmax(counts))
        top_id = int(vals[top_idx])
        top_ratio = float(counts[top_idx] / len(nonblank))

    blank_frame_pct = 0.0
    frame_top_id = None
    frame_top_ratio = 0.0
    if frames:
        blank_frame_pct = float(sum(1 for x in frames if x == blank_id) / len(frames))
        vals, counts = np.unique(frames, return_counts=True)
        top_idx = int(np.argmax(counts))
        frame_top_id = int(vals[top_idx])
        frame_top_ratio = float(counts[top_idx] / len(frames))

    blank_probability_pct = (
        None if blank_probability_pct is None else float(blank_probability_pct))
    blank_argmax_dominant = blank_frame_pct >= blank_threshold
    hard_blank_collapse = (
        blank_argmax_dominant and blank_probability_pct is not None and
        blank_probability_pct >= blank_probability_threshold)
    empty_blank_warning = (
        empty_pct >= empty_threshold and blank_argmax_dominant)
    # Detect non-blank mode collapse from raw frames. A post-CTC sequence with
    # one residual token per utterance is not non-blank collapse when nearly
    # every frame actually predicts BLANK.
    nonblank_frame_mode = (
        frame_top_id is not None and frame_top_id != blank_id and
        frame_top_ratio >= top_token_threshold)

    reason = 'ok'
    collapsed = False
    if hard_blank_collapse:
        collapsed, reason = True, 'blank_dominant'
    elif nonblank_frame_mode:
        collapsed, reason = True, 'nonblank_mode'
    elif empty_blank_warning:
        # Important warning, but not terminal without probability/loss evidence.
        reason = 'empty_blank_dominant'
    elif blank_argmax_dominant:
        reason = 'argmax_blank_dominant'
    elif empty_pct >= empty_threshold and not frames:
        # Preserve the fallback when raw-frame evidence is unavailable.
        collapsed, reason = True, 'empty_hypothesis'

    return {
        'collapsed': bool(collapsed),
        'reason': reason,
        'blank_frame_pct': blank_frame_pct,
        'blank_probability_pct': blank_probability_pct,
        'empty_hypothesis_pct': empty_pct,
        'empty_blank_warning': bool(empty_blank_warning),
        'top_id': top_id,
        'top_ratio': top_ratio,
        'nonblank_count': int(len(nonblank)),
        'frame_top_id': frame_top_id,
        'frame_top_ratio': frame_top_ratio,
        'frame_count': int(len(frames)),
        'avg_pred_token_len': avg_pred_len,
        'avg_ref_token_len': avg_ref_len,
    }


def detect_ctc_collapse(pred_ids, blank_id, threshold=0.80):
    """Backward-compatible wrapper for older diagnostic calls."""
    return compute_collapse_metrics(
        pred_ids, blank_id, top_token_threshold=threshold)


def _manual_wer(refs, hyps):
    """Word-level Levenshtein distance, summed over the batch / total ref words."""
    def edit_dist(r, h):
        # Standard DP; r, h are lists of words
        if not r:
            return len(h)
        if not h:
            return len(r)
        prev = list(range(len(h) + 1))
        for i, rw in enumerate(r, 1):
            curr = [i] + [0] * len(h)
            for j, hw in enumerate(h, 1):
                cost = 0 if rw == hw else 1
                curr[j] = min(curr[j-1] + 1,           # insertion
                              prev[j] + 1,             # deletion
                              prev[j-1] + cost)        # substitution / match
            prev = curr
        return prev[-1]

    total_words = total_err = 0
    for r, h in zip(refs, hyps):
        rw, hw = r.split(), h.split()
        total_words += len(rw)
        total_err += edit_dist(rw, hw)
    return total_err / max(total_words, 1)


def compute_wer(refs, hyps):
    if not refs:
        return 0.0
    if _HAS_JIWER:
        # jiwer raises on empty hyp lists; protect just in case
        try:
            return float(_jiwer_wer(refs, hyps))
        except Exception:
            return _manual_wer(refs, hyps)
    return _manual_wer(refs, hyps)


def _manual_cer(refs, hyps):
    """Char-level Levenshtein, summed over the batch / total ref chars.
    Mirrors _manual_wer but over characters (no whitespace split) - the
    meaningful metric for space-free scripts (zh/ja)."""
    def edit_dist(r, h):
        if not r:
            return len(h)
        if not h:
            return len(r)
        prev = list(range(len(h) + 1))
        for i, rc in enumerate(r, 1):
            curr = [i] + [0] * len(h)
            for j, hc in enumerate(h, 1):
                cost = 0 if rc == hc else 1
                curr[j] = min(curr[j-1] + 1, prev[j] + 1, prev[j-1] + cost)
            prev = curr
        return prev[-1]

    total_chars = total_err = 0
    for r, h in zip(refs, hyps):
        total_chars += len(r)
        total_err += edit_dist(r, h)
    return total_err / max(total_chars, 1)


def compute_cer(refs, hyps):
    if not refs:
        return 0.0
    if _HAS_JIWER:
        try:
            import jiwer
            return float(jiwer.cer(refs, hyps))
        except Exception:
            return _manual_cer(refs, hyps)
    return _manual_cer(refs, hyps)


def per_language_metrics(refs, hyps, langs):
    """Group (ref, hyp) by language tag; compute WER + CER per group.
    Returns {lang: {'wer','cer','n'}}. WER is unreliable for space-free
    scripts (zh/ja) - read CER there."""
    groups = {}
    for r, h, lang in zip(refs, hyps, langs):
        groups.setdefault(lang, ([], []))
        groups[lang][0].append(r)
        groups[lang][1].append(h)
    out = {}
    for lang, (rs, hs) in sorted(groups.items()):
        out[lang] = {'wer': compute_wer(rs, hs),
                     'cer': compute_cer(rs, hs),
                     'n': len(rs)}
    return out


# ── 2. Load or rebuild SentencePiece tokenizer ──────────────────
def _clean_text_scalar(v):
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if isinstance(v, bytes):
        v = v.decode('utf-8')
    if isinstance(v, np.generic):
        v = v.item()
    return str(v)


def _iter_cache_texts(cache_dir):
    for shard in sorted(glob.glob(os.path.join(cache_dir, '*', 'shard_*.npz'))):
        with np.load(shard, allow_pickle=True) as data:
            text_arr = data['texts'] if 'texts' in data.files else (
                data['text'] if 'text' in data.files else None)
            if text_arr is None:
                continue
            for text in text_arr:
                text = _clean_text_scalar(text).strip()
                if text:
                    yield text


def _build_sentencepiece_from_cache(cache_dir, prefix):
    os.makedirs(os.path.dirname(prefix), exist_ok=True)
    corpus_path = prefix + '_corpus.txt'
    n_text = 0
    with open(corpus_path, 'w', encoding='utf-8') as f:
        for text in _iter_cache_texts(cache_dir):
            f.write(text.replace('\n', ' ') + '\n')
            n_text += 1
    if n_text == 0:
        raise RuntimeError(f'No cached texts found under {cache_dir}; cannot build tokenizer.')
    print(f'Building {C.spm_vocab}-piece SentencePiece tokenizer from {n_text:,} cached texts...')
    spm.SentencePieceTrainer.Train(
        input=corpus_path,
        model_prefix=prefix,
        vocab_size=int(C.spm_vocab),
        character_coverage=float(C.spm_coverage),
        model_type='unigram',
        input_sentence_size=2000000,
        # Cache traversal is sorted, so disabling shuffle makes a rebuild
        # reproducible across Kaggle sessions and worker counts.
        shuffle_input_sentence=False,
        num_threads=1,
        hard_vocab_limit=False,
    )


def _load_sentencepiece_for_training():
    requested = int(getattr(C, 'spm_vocab', 0) or 0)
    work_prefix = os.path.join(C.root, 'tokenizer', f'spm_{requested}')
    checkpoint_prefixes = []
    for load_dir in (getattr(C, 'ckpt_load_dirs', None) or
                     [getattr(C, 'ckpt_load_dir', None)]):
        if load_dir:
            checkpoint_prefixes.append(os.path.join(
                os.path.dirname(load_dir), 'tokenizer', f'spm_{requested}'))
    candidates = [work_prefix, *checkpoint_prefixes, C.spm_prefix]
    seen = set()
    mismatches = []
    for prefix in candidates:
        if prefix in seen:
            continue
        seen.add(prefix)
        model_path = prefix + '.model'
        if not os.path.exists(model_path):
            continue
        cand = spm.SentencePieceProcessor()
        cand.Load(model_path)
        size = cand.GetPieceSize()
        if requested <= 0 or size == requested:
            C.spm_prefix = prefix
            return cand
        mismatches.append((prefix, size))

    if not getattr(C, 'allow_tokenizer_rebuild', True):
        raise RuntimeError(
            f'No tokenizer with {requested} pieces found. Mismatches={mismatches}')

    print(f'No matching {requested}-piece tokenizer found; mismatches={mismatches}.')
    _build_sentencepiece_from_cache(C.cache_dir, work_prefix)
    cand = spm.SentencePieceProcessor()
    cand.Load(work_prefix + '.model')
    C.spm_prefix = work_prefix
    return cand


sp = _load_sentencepiece_for_training()
BLANK = sp.GetPieceSize()  # CTC blank = last index
C.vocab_size = BLANK + 1   # SP pieces + 1 CTC blank
assert BLANK == sp.GetPieceSize(), 'BLANK should be one past SPM IDs'
assert C.vocab_size == sp.GetPieceSize() + 1, 'ctc_head must output (vocab + 1) classes'
print(f'Tokenizer: {sp.GetPieceSize()} tokens + blank={BLANK}, vocab_size={C.vocab_size}')
print(f'  tokenizer_prefix={C.spm_prefix}')

# A same-shaped CTC head is still incompatible when token IDs map to
# different pieces. Persist and validate this fingerprint in checkpoints.
def _tokenizer_fingerprint(sp_model):
    import hashlib
    h = hashlib.sha256()
    h.update(f'pieces={sp_model.GetPieceSize()}\n'.encode('utf-8'))
    for i in range(sp_model.GetPieceSize()):
        piece = sp_model.IdToPiece(i).encode('utf-8')
        h.update(len(piece).to_bytes(4, 'little'))
        h.update(piece)
    return h.hexdigest()

TOKENIZER_FINGERPRINT = _tokenizer_fingerprint(sp)
print(f'  tokenizer_sha256={TOKENIZER_FINGERPRINT[:16]}...')


def _require_checkpoint_tokenizer(state, path, allow_missing=False):
    found = state.get('tokenizer_fingerprint')
    if found is None:
        message = (
            f'Checkpoint {path} has no tokenizer fingerprint. Its CTC head '
            'cannot be proven compatible with the active tokenizer.')
        if allow_missing:
            print(f'WARNING: {message}')
            return False
        raise RuntimeError(message)
    if found != TOKENIZER_FINGERPRINT:
        raise RuntimeError(
            f'Tokenizer/checkpoint mismatch for {path}: '
            f'checkpoint={found[:16]}..., active={TOKENIZER_FINGERPRINT[:16]}.... '
            'Load the tokenizer saved with that checkpoint or reset the CTC head.')
    return True


# ── 3. Dataset (lazy-loaded int8 ragged shards) ────────────────
class KDDataset(Dataset):
    def __init__(self, cache_dir, sp_model, mel_dir=None):
        if not os.path.isdir(cache_dir):
            raise FileNotFoundError(
                f'Teacher cache directory not found: {cache_dir}. Attach the '
                'configured Kaggle dataset or update C.cache_dir.')
        if mel_dir is not None and not os.path.isdir(mel_dir):
            raise FileNotFoundError(
                f'Mel cache directory not found: {mel_dir}. Attach the companion '
                'mel-cache dataset or update C.mel_dir.')
        self.sp = sp_model
        self.cache_dir = os.path.normpath(cache_dir)
        self.mel_dir = os.path.normpath(mel_dir) if mel_dir is not None else None
        self.index = []
        # Parallel to `index`: all chunks from one recording share a group.
        # Persisted splits use this to keep overlapping audio out of val/test.
        self.group_ids = []
        # Parallel language names make both multilingual sampling and the
        # auxiliary language objective deterministic without reopening shards.
        self.language_names = []

        shard_paths = []
        for lang_dir in sorted(os.listdir(cache_dir)):
            lang_path = os.path.join(cache_dir, lang_dir)
            if not os.path.isdir(lang_path):
                continue
            shards = sorted(f for f in os.listdir(lang_path) if f.endswith('.npz'))
            for sf in shards:
                shard_paths.append(os.path.join(lang_path, sf))
            print(f'  {lang_dir}: {len(shards)} shards')

        print(f'Indexing {len(shard_paths)} shards...')
        skipped_ctc = 0
        skipped_token = 0
        retokenized = 0
        for shard in tqdm(shard_paths, desc='Indexing', unit='shard'):
            data = np.load(shard, allow_pickle=True)
            n = len(data['scales'])
            text_arr = self._optional_npz(data, 'texts', 'text')
            source_arr = self._optional_npz(data, 'source_ids', 'source_id')
            lang_arr = self._optional_npz(data, 'langs', 'lang')
            fallback_lang = os.path.basename(os.path.dirname(shard))
            for j in range(n):
                text = str(self._optional_value(text_arr, j, ''))
                cached_ids = None if getattr(C, 'force_retokenize_targets', True) else (
                    self._token_ids_from_cache(data, j))
                token_ids, did_retokenize = self._validated_target_ids(cached_ids, text)
                if did_retokenize:
                    retokenized += 1
                if not token_ids:
                    skipped_token += 1
                    continue

                mel_len = int(data['mel_lens'][j])
                enc_len = max(1, mel_len)
                k, p, stride = C.cnn_ks, C.cnn_ks // 2, 2
                for _ in range(2):
                    enc_len = ((enc_len + 2 * p - k) // stride) + 1
                enc_len = max(1, enc_len)
                repeats = sum(
                    1 for a, b in zip(token_ids, token_ids[1:]) if a == b)
                if len(token_ids) + repeats > enc_len:
                    skipped_ctc += 1
                    continue
                source_id = self._optional_value(source_arr, j, None)
                if source_id is None or not str(source_id).strip():
                    # Missing provenance must not merge unrelated examples.
                    rel = '/'.join(shard.replace('\\', '/').rsplit('/', 2)[-2:])
                    group_id = f'{rel}:{j}'
                else:
                    lang_dir = os.path.basename(os.path.dirname(shard))
                    group_id = f'{lang_dir}|{source_id}'
                sample_lang = str(self._optional_value(
                    lang_arr, j, fallback_lang) or fallback_lang)
                self.index.append((shard, j))
                self.group_ids.append(group_id)
                self.language_names.append(sample_lang)
            del data
        self.languages = tuple(sorted(set(self.language_names)))
        self.lang_to_id = {'<unk>': 0}
        self.lang_to_id.update({name: i + 1 for i, name in enumerate(self.languages)})
        print(f'Indexed {len(self.index)} samples across {len(self.languages)} languages')
        print(f'  language vocabulary: {self.lang_to_id}')
        if skipped_ctc or skipped_token:
            print(
                f'Skipped {skipped_ctc} CTC-invalid and '
                f'{skipped_token} token-invalid samples during indexing')
        if retokenized:
            print(f'Re-encoded {retokenized} sample(s) from cached text for the active tokenizer')

        self._cache, self._cache_order = {}, []
        self._max_cache = 4

    @staticmethod
    def _optional_npz(data, *names):
        for name in names:
            if name in data.files:
                return data[name]
        return None

    @staticmethod
    def _clean_scalar(v):
        if isinstance(v, np.ndarray) and v.shape == ():
            v = v.item()
        if isinstance(v, bytes):
            v = v.decode('utf-8')
        if isinstance(v, np.generic):
            v = v.item()
        return v

    @classmethod
    def _optional_value(cls, arr, local_idx, default=None):
        if arr is None:
            return default
        try:
            v = arr[local_idx]
        except Exception:
            return default
        v = cls._clean_scalar(v)
        if isinstance(v, np.ndarray) and v.shape == ():
            v = cls._clean_scalar(v.item())
        return v

    def _validated_target_ids(self, cached_ids, text):
        def _valid(ids):
            return bool(ids) and max(ids) < BLANK and min(ids) >= 0

        if cached_ids is not None and _valid(cached_ids):
            return cached_ids, False

        token_ids = self.sp.EncodeAsIds(str(text or ''))
        if _valid(token_ids):
            return [int(x) for x in token_ids], cached_ids is not None
        return None, cached_ids is not None

    @classmethod
    def _token_ids_from_cache(cls, data, local_idx):
        tok = data.get('token_ids')
        if tok is None:
            return None
        if 'token_offsets' in data:
            s = int(data['token_offsets'][local_idx])
            e = int(data['token_offsets'][local_idx + 1])
            ids = tok[s:e]
        else:
            ids = tok[local_idx]
            if 'token_lens' in data:
                ids = ids[:int(data['token_lens'][local_idx])]
        if isinstance(ids, np.ndarray):
            if ids.dtype == object and ids.shape == ():
                ids = ids.item()
            else:
                ids = ids.tolist()
        if isinstance(ids, (int, np.integer)):
            ids = [int(ids)]
        return [int(x) for x in list(ids)]

    def _get_shard(self, shard_path):
        if shard_path not in self._cache:
            if len(self._cache) >= self._max_cache:
                oldest = self._cache_order.pop(0)
                del self._cache[oldest]
            data = np.load(shard_path, allow_pickle=True)
            entry = {
                'hidden_cat': data['hidden_cat'],
                'offsets': data['offsets'],
                'scales': data['scales'],
                'zeros': data['zeros'],
                'mel_lens': data['mel_lens'],
                'texts': self._optional_npz(data, 'texts', 'text'),
            }
            for key in [
                'token_ids', 'token_offsets', 'token_lens',
                'source_ids', 'source_id', 'chunk_start_s', 'chunk_end_s',
                'duration_s', 'durations_s', 'langs', 'lang',
                'teacher_confidence', 'teacher_confidences',
            ]:
                if key in data.files:
                    entry[key] = data[key]
            if self.mel_dir is not None:
                relative_shard = os.path.relpath(shard_path, self.cache_dir)
                mel_path = os.path.join(self.mel_dir, relative_shard)
                if not os.path.isfile(mel_path):
                    raise FileNotFoundError(
                        f'Companion mel shard not found for {shard_path}: {mel_path}')
                md = np.load(mel_path, allow_pickle=True)
                assert np.array_equal(md['mel_lens'], data['mel_lens']), (
                    f"mel/teacher cache misaligned for {os.path.basename(shard_path)}: "
                    "mel_lens differ - regenerate the mel cache from the same chunk order.")
                entry['mel_cat'] = md['mel_cat']
                entry['mel_off'] = md['offsets']
                entry['mel_sc'] = md['scales']
                entry['mel_zr'] = md['zeros']
            self._cache[shard_path] = entry
            self._cache_order.append(shard_path)
        return self._cache[shard_path]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        shard_path, local_idx = self.index[idx]
        data = self._get_shard(shard_path)
        s, e = data['offsets'][local_idx], data['offsets'][local_idx + 1]
        h_i8 = data['hidden_cat'][s:e].astype(np.float32)
        h_f16 = (
            (h_i8 -
             data['zeros'][local_idx]) *
            data['scales'][local_idx]).astype(
            np.float16)

        text_arr = data.get('texts')
        text = str(
            self._optional_value(
                text_arr,
                local_idx,
                '')) if text_arr is not None else ''
        mel_len = int(data['mel_lens'][local_idx])
        cached_ids = None if getattr(C, 'force_retokenize_targets', True) else (
            self._token_ids_from_cache(data, local_idx))
        token_ids, _ = self._validated_target_ids(cached_ids, text)
        if not token_ids:
            raise ValueError(
                f'Could not produce valid target IDs for {os.path.basename(shard_path)} '
                f'idx={local_idx} with active blank={BLANK}')

        # Resolved once during indexing so IDs are stable across workers.
        lang = self.language_names[idx]
        duration = self._optional_value(
            data.get(
                'duration_s',
                data.get('durations_s')),
            local_idx,
            None)
        if duration is None:
            duration = mel_len / 100.0

        item = {
            'teacher_h': torch.from_numpy(h_f16),
            'mel_len': mel_len,
            'token_ids': torch.tensor(token_ids, dtype=torch.long),
            'text': text,
            'lang': str(lang),
            'lang_id': int(self.lang_to_id.get(str(lang), 0)),
            'source_id': self._optional_value(data.get('source_ids', data.get('source_id')), local_idx, None),
            'chunk_start_s': self._optional_value(data.get('chunk_start_s'), local_idx, None),
            'chunk_end_s': self._optional_value(data.get('chunk_end_s'), local_idx, None),
            'duration_s': float(duration) if duration is not None else None,
            'teacher_confidence': self._optional_value(
                data.get('teacher_confidence', data.get('teacher_confidences')), local_idx, None),
        }
        if self.mel_dir is not None:
            ms, me = data['mel_off'][local_idx], data['mel_off'][local_idx + 1]
            m_i8 = data['mel_cat'][ms:me].astype(np.float32)
            m = (
                (m_i8 -
                 data['mel_zr'][local_idx]) *
                data['mel_sc'][local_idx]).astype(
                np.float32)
            item['mel'] = torch.from_numpy(m).transpose(0, 1).contiguous()
        return item


# ── Shard-bucketed batch sampler ─────────────────────────────────
# Batch by shard to avoid repeated np.load on slow FUSE mounts; shuffle
# shard order and in-shard samples each epoch.
class ShardBucketBatchSampler:
    def __init__(self, dataset, batch_size, generator=None,
                 drop_last=True, shuffle=True, language_balance_alpha=None):
        """Yield shard-local batches, optionally rebalanced by language.

        Sampling whole pre-built batches preserves the one-shard-per-batch FUSE
        optimization.  Temperature balancing may repeat low-resource batches,
        but the epoch length stays equal to the natural number of batches.
        """
        self.batch_size = batch_size
        self.generator = generator
        self.drop_last = drop_last
        self.shuffle = shuffle
        self.language_balance_alpha = language_balance_alpha

        from torch.utils.data import Subset

        def _resolve(d):
            if isinstance(d, Subset):
                base, inner = _resolve(d.dataset)
                return base, [inner[i] for i in d.indices]
            return d, list(range(len(d)))

        base_dataset, idx_chain = _resolve(dataset)
        shard_to_positions = {}
        shard_to_language = {}
        for pos, real_i in enumerate(idx_chain):
            shard_path, _ = base_dataset.index[real_i]
            shard_to_positions.setdefault(shard_path, []).append(pos)
            if hasattr(base_dataset, 'language_names'):
                language = base_dataset.language_names[real_i]
            else:
                language = os.path.basename(os.path.dirname(shard_path))
            previous = shard_to_language.setdefault(shard_path, str(language))
            if previous != str(language):
                raise RuntimeError(
                    f'Shard {shard_path} mixes languages {previous!r}/{language!r}')

        ordered_shards = list(shard_to_positions)
        self.shard_buckets = [shard_to_positions[p] for p in ordered_shards]
        self.bucket_languages = [shard_to_language[p] for p in ordered_shards]
        if self.drop_last:
            self._n_batches = sum(len(b) // self.batch_size
                                  for b in self.shard_buckets)
        else:
            self._n_batches = sum((len(b) + self.batch_size - 1) // self.batch_size
                                  for b in self.shard_buckets)

    def __len__(self):
        return self._n_batches

    def _permute(self, values):
        values = list(values)
        if not self.shuffle or len(values) <= 1:
            return values
        order = torch.randperm(len(values), generator=self.generator).tolist()
        return [values[i] for i in order]

    def _bucket_batches(self, bucket):
        bucket = self._permute(bucket)
        batches = []
        for i in range(0, len(bucket), self.batch_size):
            batch = bucket[i:i + self.batch_size]
            if len(batch) < self.batch_size and self.drop_last:
                continue
            batches.append(batch)
        return batches

    def __iter__(self):
        shard_order = self._permute(range(len(self.shard_buckets)))
        if self.language_balance_alpha is None:
            for shard_idx in shard_order:
                yield from self._bucket_batches(self.shard_buckets[shard_idx])
            return

        by_language = {}
        for shard_idx in shard_order:
            language = self.bucket_languages[shard_idx]
            by_language.setdefault(language, []).extend(
                self._bucket_batches(self.shard_buckets[shard_idx]))
        by_language = {
            language: self._permute(batches)
            for language, batches in by_language.items() if batches
        }
        if not by_language:
            return

        languages = sorted(by_language)
        alpha = float(self.language_balance_alpha)
        counts = torch.tensor(
            [len(by_language[language]) for language in languages],
            dtype=torch.float64,
        )
        probabilities = counts.pow(alpha)
        probabilities /= probabilities.sum()
        draws = torch.multinomial(
            probabilities, self._n_batches, replacement=True,
            generator=self.generator,
        ).tolist()
        cursors = {language: 0 for language in languages}
        for language_idx in draws:
            language = languages[language_idx]
            batches = by_language[language]
            cursor = cursors[language]
            if cursor >= len(batches):
                batches = self._permute(batches)
                by_language[language] = batches
                cursor = 0
            yield batches[cursor]
            cursors[language] = cursor + 1


# ── Persistent train/val/test splits ─────────────────────────────
# Persist train/val/test splits across Kaggle sessions; fingerprint shards
# so stale splits fail loudly.
def _dataset_fingerprint(ds):
    """Short hash of dataset state. Changes if shards are added, removed,
    or reordered. Sampled across the dataset, not just the head."""
    import hashlib
    h = hashlib.sha256()
    n = len(ds)
    h.update(f'len={n};split=grouped_source_v1'.encode())
    step = max(1, n // 100)
    for i in range(0, n, step):
        shard_path, local_idx = ds.index[i]
        # Use last two path components (language_dir/shard_file) so the
        # fingerprint survives moving the cache to a different mount.
        parts = shard_path.replace('\\', '/').rsplit('/', 2)[-2:]
        group_id = ds.group_ids[i] if hasattr(ds, 'group_ids') else ''
        h.update(f'{"/".join(parts)}:{local_idx}:{group_id}'.encode())
    return h.hexdigest()[:16]


SPLIT_STRATEGY = 'grouped_source_v1'


def get_or_create_splits(ds, val_size, test_size, seed, splits_path):
    """Persist source-grouped train/val/test splits.

    Chunk caches contain overlapping windows from the same recording. A random
    per-chunk split leaks shared audio across train and test, so every source_id
    is allocated as one indivisible group. Actual val/test counts can exceed the
    requested targets slightly when the final source contributes many chunks.
    """
    from torch.utils.data import Subset

    fingerprint = _dataset_fingerprint(ds)
    group_ids = getattr(ds, 'group_ids', None)
    if group_ids is None or len(group_ids) != len(ds):
        raise RuntimeError(
            'KDDataset group_ids are missing/misaligned; rerun the distillation '
            'definition cell before creating splits.')

    if os.path.exists(splits_path):
        with open(splits_path) as f:
            saved = json.load(f)
        saved_strategy = saved.get('strategy')
        if saved_strategy != SPLIT_STRATEGY or saved.get('fingerprint') != fingerprint:
            raise RuntimeError(
                f'Splits in {splits_path} are stale or unsafe:\n'
                f'  saved strategy      = {saved_strategy!r}\n'
                f'  required strategy   = {SPLIT_STRATEGY!r}\n'
                f'  saved fingerprint   = {saved.get("fingerprint")}\n'
                f'  current fingerprint = {fingerprint}\n'
                'Delete only the writable splits.json and rerun this cell. Old '
                'per-chunk test results are invalid because overlapping chunks '
                'from one source could cross split boundaries.'
            )
        train_idx = saved['train_indices']
        val_idx = saved['val_indices']
        test_idx = saved['test_indices']
        train_groups = {group_ids[i] for i in train_idx}
        val_groups = {group_ids[i] for i in val_idx}
        test_groups = {group_ids[i] for i in test_idx}
        assert train_groups.isdisjoint(val_groups)
        assert train_groups.isdisjoint(test_groups)
        assert val_groups.isdisjoint(test_groups)
        print(f'Loaded persisted {SPLIT_STRATEGY} splits from {splits_path}')
        print(f'  seed={saved["seed"]}, fingerprint={fingerprint}, '
              f'train/val/test = {len(train_idx):,} / {len(val_idx):,} / '
              f'{len(test_idx):,}')
        return Subset(ds, train_idx), Subset(ds, val_idx), Subset(ds, test_idx)

    n = len(ds)
    if val_size + test_size >= n:
        raise ValueError(
            f'val_size + test_size = {val_size + test_size} >= dataset size {n}')

    groups = {}
    for idx, group_id in enumerate(group_ids):
        groups.setdefault(str(group_id), []).append(idx)
    group_keys = sorted(groups)
    gen = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(group_keys), generator=gen).tolist()

    train_idx, val_idx, test_idx = [], [], []
    for pos in order:
        rows = groups[group_keys[pos]]
        if len(val_idx) < val_size or len(test_idx) < test_size:
            # Fill the currently smaller fraction of its target. This keeps the
            # two held-out splits close in size without ever breaking a source.
            val_fill = len(val_idx) / max(val_size, 1)
            test_fill = len(test_idx) / max(test_size, 1)
            if len(val_idx) < val_size and (val_fill <= test_fill or len(test_idx) >= test_size):
                val_idx.extend(rows)
            else:
                test_idx.extend(rows)
        else:
            train_idx.extend(rows)

    if not train_idx or not val_idx or not test_idx:
        raise RuntimeError(
            f'Grouped split failed: train/val/test={len(train_idx)}/{len(val_idx)}/{len(test_idx)}')

    train_groups = {group_ids[i] for i in train_idx}
    val_groups = {group_ids[i] for i in val_idx}
    test_groups = {group_ids[i] for i in test_idx}
    assert train_groups.isdisjoint(val_groups)
    assert train_groups.isdisjoint(test_groups)
    assert val_groups.isdisjoint(test_groups)

    os.makedirs(os.path.dirname(splits_path) or '.', exist_ok=True)
    with open(splits_path, 'w') as f:
        json.dump({
            'strategy': SPLIT_STRATEGY,
            'fingerprint': fingerprint,
            'seed': seed,
            'dataset_len': n,
            'requested_val_size': val_size,
            'requested_test_size': test_size,
            'val_size': len(val_idx),
            'test_size': len(test_idx),
            'train_indices': train_idx,
            'val_indices': val_idx,
            'test_indices': test_idx,
        }, f)
    print(f'Created {SPLIT_STRATEGY} splits at {splits_path}: '
          f'train={len(train_idx):,}, val={len(val_idx):,}, '
          f'test={len(test_idx):,} (seed={seed}, groups={len(groups):,})')
    return Subset(ds, train_idx), Subset(ds, val_idx), Subset(ds, test_idx)


def collate_kd(batch, max_mel_frames=None, allow_crop=False):
    """Pad chunk-level examples without corrupting CTC alignment.

    True chunk data already pairs each audio span with the transcript spoken in
    that span. If a chunk exceeds a safety cap, raise so the data can be
    filtered/regenerated; do not crop the audio while keeping the full target.
    """
    n_mels = batch[0]['mel'].shape[0]
    raw_max_m = max(b['mel'].shape[1] for b in batch)
    if max_mel_frames is not None and raw_max_m > max_mel_frames and not allow_crop:
        offenders = [
            (i, b.get('source_id'), b['mel'].shape[1], b.get('text', '')[:80])
            for i, b in enumerate(batch) if b['mel'].shape[1] > max_mel_frames
        ]
        raise ValueError(
            f'Batch contains chunk longer than max_mel_frames={max_mel_frames}. '
            f'Do not crop audio for CTC; regenerate/filter shorter chunks. '
            f'First offenders: {offenders[:3]}')

    max_m = raw_max_m if max_mel_frames is None else min(raw_max_m, max_mel_frames)
    mel = torch.zeros(len(batch), n_mels, max_m, dtype=torch.float32)
    mel_lens = []
    for i, b in enumerate(batch):
        t = min(b['mel'].shape[1], max_m)
        mel[i, :, :t] = b['mel'][:, :t]
        mel_lens.append(t)

    max_t = max(b['teacher_h'].shape[0] for b in batch)
    if allow_crop and max_mel_frames is not None:
        max_t = min(max_t, max_m // 2)
    D = batch[0]['teacher_h'].shape[1]
    teacher_h = torch.zeros(len(batch), max_t, D, dtype=torch.float16)
    teacher_mask = torch.zeros(len(batch), max_t, dtype=torch.bool)
    for i, b in enumerate(batch):
        t = min(b['teacher_h'].shape[0], max_t)
        teacher_h[i, :t] = b['teacher_h'][:t]
        teacher_mask[i, :t] = True

    token_lists = []
    for b in batch:
        ids = b['token_ids']
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        ids = [int(x) for x in ids]
        if ids:
            assert max(
                ids) < BLANK, f'Target contains blank/invalid id: max={max(ids)}, blank={BLANK}'
            assert min(ids) >= 0, f'Target contains negative id: min={min(ids)}'
        token_lists.append(ids)

    max_tok = max((len(ids) for ids in token_lists), default=0)
    tokens = torch.full((len(batch), max_tok), BLANK, dtype=torch.long)
    tok_lens = []
    for i, ids in enumerate(token_lists):
        tl = len(ids)
        if tl:
            tokens[i, :tl] = torch.tensor(ids, dtype=torch.long)
        tok_lens.append(tl)

    confidence = torch.tensor([
        float(b.get('teacher_confidence'))
        if b.get('teacher_confidence') is not None and
        math.isfinite(float(b.get('teacher_confidence'))) else 1.0
        for b in batch
    ], dtype=torch.float32).clamp_(0.0, 1.0)

    return {
        'mel': mel,
        'mel_lens': torch.tensor(mel_lens, dtype=torch.long),
        'teacher_h': teacher_h,
        'teacher_mask': teacher_mask,
        'teacher_lens': teacher_mask.sum(dim=1, dtype=torch.long),
        'tokens': tokens,
        'tok_lens': torch.tensor(tok_lens, dtype=torch.long),
        'texts': [b['text'] for b in batch],
        'langs': [b.get('lang', 'unknown') for b in batch],
        'lang_ids': torch.tensor([b.get('lang_id', 0) for b in batch], dtype=torch.long),
        'source_ids': [b.get('source_id', None) for b in batch],
        'chunk_start_s': [b.get('chunk_start_s', None) for b in batch],
        'chunk_end_s': [b.get('chunk_end_s', None) for b in batch],
        'audio_seconds': [b.get('duration_s', None) for b in batch],
        'teacher_confidence': confidence,
    }

# --- 4. Student Model (ConMamba encoder, single mel path) ---
# Single mel path: mel -> conv -> ConMamba -> {CTC, feature KD}; keep the
# fp32 encoder island because selective-scan can overflow in fp16.


class CausalConv1d(nn.Conv1d):
    """Left-pad a strided Conv1d; weight/state-dict shapes stay unchanged."""

    def forward(self, x):
        left = (self.kernel_size[0] - 1) * self.dilation[0]
        return super().forward(F.pad(x, (left, 0)))


class CausalLanguageAdapter(nn.Module):
    """Frame-causal language routing with a very small expert bank.

    The router sees only the current causal encoder state.  Soft routing keeps
    training differentiable; zero-initialized expert outputs make this module an
    identity at initialization, which permits safe warm-starting from the older
    shared multilingual ConMamba checkpoint.
    """

    def __init__(self, d_model, n_languages, rank=16, scale=0.5):
        super().__init__()
        self.n_languages = max(1, int(n_languages))
        self.scale = float(scale)
        self.router = nn.Linear(d_model, self.n_languages)
        self.norm = nn.LayerNorm(d_model)
        self.down = nn.Linear(d_model, int(rank), bias=False)
        self.expert_up = nn.Parameter(torch.zeros(
            self.n_languages, int(rank), d_model))

    def forward(self, x, padding_mask=None):
        language_logits = self.router(x)
        probabilities = language_logits.float().softmax(dim=-1).to(x.dtype)
        z = F.silu(self.down(self.norm(x)))
        routed = torch.zeros_like(x)
        # The expert bank is intentionally tiny (rank=16 by default).  This loop
        # avoids materializing [B,T,L,R,D], which would erase its edge advantage.
        for language_idx in range(self.n_languages):
            expert = F.linear(z, self.expert_up[language_idx].transpose(0, 1))
            routed = routed + probabilities[..., language_idx:language_idx + 1] * expert
        out = x + self.scale * routed
        if padding_mask is not None:
            out = out.masked_fill(padding_mask.unsqueeze(-1), 0.0)
            language_logits = language_logits.masked_fill(
                padding_mask.unsqueeze(-1), 0.0)
        return out, language_logits


class StreamConMambaStudent(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d = config.cm_d_model

        # Front-end: two causal stride-2 convs -> 4x downsample
        # (mel 100 fps -> encoder 25 fps, i.e. 40 ms tokens).
        self.cnn = nn.Sequential(
            CausalConv1d(config.n_mels, d, config.cnn_ks, stride=2, padding=0),
            nn.GELU(),
            CausalConv1d(d, d, config.cnn_ks, stride=2, padding=0),
            nn.GELU(),
        )
        self.in_norm = nn.LayerNorm(d)

        self.encoder = ConmambaEncoder(
            num_layers=config.cm_layers, d_model=d, d_ffn=config.cm_d_ffn,
            kernel_size=config.cm_kernel, dropout=config.dropout,
            causal=not config.cm_bidirectional,
            mamba_config=dict(
                d_state=config.cm_d_state, expand=config.cm_expand,
                d_conv=config.cm_d_conv, bidirectional=config.cm_bidirectional,
            ),
        )

        taps = tuple(sorted(set(int(i) for i in config.kd_tap_layers)))
        if not taps or taps[-1] != config.cm_layers - 1:
            taps = tuple(i for i in taps if i < config.cm_layers - 1) + (
                config.cm_layers - 1,)
        if taps[0] < 0 or taps[-1] >= config.cm_layers:
            raise ValueError(
                f'kd_tap_layers={taps} invalid for cm_layers={config.cm_layers}')
        self.kd_tap_layers = taps
        self.kd_aux_heads = nn.ModuleList([
            nn.Sequential(nn.LayerNorm(d), nn.Linear(d, config.teacher_d))
            for _ in taps[:-1]
        ])

        self.language_adapter = CausalLanguageAdapter(
            d, getattr(config, 'num_languages', 1),
            rank=getattr(config, 'lang_expert_rank', 16),
            scale=getattr(config, 'lang_expert_scale', 0.5),
        )
        self.dropout = nn.Dropout(getattr(config, 'dropout', 0.0))
        self.ctc_head = nn.Linear(d, config.vocab_size)
        # Historical name retained: old KD checkpoints can initialize this head.
        self.kl_head = nn.Linear(d, config.teacher_d)

    def _subsample_len(self, lens):
        """Lengths after two stride-2 causal convs (ceil division by four)."""
        k, p, stride = self.config.cnn_ks, self.config.cnn_ks // 2, 2
        for _ in range(2):
            lens = torch.div(lens + 2 * p - k, stride, rounding_mode='floor') + 1
        return lens.clamp(min=1)

    def enable_grad_ckpt(self):
        """Class-level checkpoint patch: idempotent and DataParallel-safe."""
        from torch.utils.checkpoint import checkpoint
        if not self.encoder.layers:
            return
        block_cls = type(self.encoder.layers[0])
        if getattr(block_cls, '_grad_ckpt_patched', False):
            print(f'Gradient checkpointing already enabled on {block_cls.__name__}')
            return
        original_forward = block_cls.forward

        def ckpt_forward(self, hidden_states, *args, **kwargs):
            if not self.training:
                return original_forward(self, hidden_states, *args, **kwargs)
            return checkpoint(
                lambda h: original_forward(self, h, *args, **kwargs),
                hidden_states, use_reentrant=False,
            )

        block_cls.forward = ckpt_forward
        block_cls._grad_ckpt_patched = True
        print(f'Gradient checkpointing enabled on {len(self.encoder.layers)} '
              f'{block_cls.__name__} blocks (DataParallel-safe)')

    def _forward_impl(self, mel, mel_lens, compute_kd=True):
        x = self.cnn(mel)
        x = self.in_norm(x.transpose(1, 2))
        enc_lens = self._subsample_len(mel_lens).clamp(max=x.shape[1])
        padding_mask = (torch.arange(x.shape[1], device=x.device)[None, :]
                        >= enc_lens[:, None])

        # Critical fp32 island: selective scan can overflow fp16 on long audio.
        with torch.amp.autocast('cuda', enabled=False):
            h, tapped = self.encoder(
                x.float(), src_key_padding_mask=padding_mask,
                return_intermediate_layers=(self.kd_tap_layers if compute_kd else ()),
            )
        h = h.masked_fill(padding_mask.unsqueeze(-1), 0.0)
        h, language_logits = self.language_adapter(h, padding_mask)
        ctc_logits = self.ctc_head(self.dropout(h))

        kd_features = ()
        if compute_kd:
            tapped_by_layer = dict(tapped)
            projected = []
            for aux_idx, layer_idx in enumerate(self.kd_tap_layers[:-1]):
                tap = tapped_by_layer[layer_idx].masked_fill(
                    padding_mask.unsqueeze(-1), 0.0)
                projected.append(self.kd_aux_heads[aux_idx](self.dropout(tap)))
            projected.append(self.kl_head(self.dropout(h)))
            kd_features = tuple(projected)
        return ctc_logits, kd_features, language_logits, enc_lens

    def forward(self, mel, mel_lens):
        return self._forward_impl(mel, mel_lens, compute_kd=True)

    def inference_ctc(self, mel, mel_lens):
        """CTC/router outputs only; auxiliary KD heads are training-only."""
        ctc_logits, _, language_logits, enc_lens = self._forward_impl(
            mel, mel_lens, compute_kd=False)
        return ctc_logits, language_logits, enc_lens

    def streaming_ctc(self, mel, mel_lens, chunk_mel_frames=None,
                      left_context_mel_frames=None):
        """Bounded-memory blockwise CTC for deployment benchmarking.

        Stock `mamba_ssm` exposes recurrent cache APIs, but the surrounding
        causal Conformer convolution needs its own cache for exact stateful
        stepping.  Until that export path is implemented, reprocess a fixed
        left context and emit only the new frames.  Memory and algorithmic
        latency stay bounded, and no future frame is observed.
        """
        if mel.shape[0] != 1:
            raise ValueError('streaming_ctc currently requires batch size 1')
        chunk_mel_frames = int(chunk_mel_frames or self.config.stream_chunk_ms // 10)
        left_context_mel_frames = int(
            left_context_mel_frames
            if left_context_mel_frames is not None
            else self.config.stream_left_context_ms // 10)
        if chunk_mel_frames <= 0 or chunk_mel_frames % 4:
            raise ValueError('chunk_mel_frames must be a positive multiple of 4')
        if left_context_mel_frames < 0 or left_context_mel_frames % 4:
            raise ValueError('left_context_mel_frames must be a multiple of 4')

        total_mel = int(mel_lens[0].item())
        ctc_parts, language_parts = [], []
        for start in range(0, total_mel, chunk_mel_frames):
            end = min(total_mel, start + chunk_mel_frames)
            context_start = max(0, start - left_context_mel_frames)
            window = mel[:, :, context_start:end]
            window_lens = mel_lens.new_tensor([end - context_start])
            logits, _, language_logits, window_enc_lens = self._forward_impl(
                window, window_lens, compute_kd=False)
            context_mel = start - context_start
            if context_mel:
                context_len = int(self._subsample_len(
                    mel_lens.new_tensor([context_mel]))[0].item())
            else:
                context_len = 0
            valid_end = int(window_enc_lens[0].item())
            ctc_parts.append(logits[:, context_len:valid_end])
            language_parts.append(language_logits[:, context_len:valid_end])

        ctc_logits = torch.cat(ctc_parts, dim=1)
        language_logits = torch.cat(language_parts, dim=1)
        out_lens = mel_lens.new_tensor([ctc_logits.shape[1]])
        return ctc_logits, language_logits, out_lens


# Backward-compatible constructor name used by older diagnostic snippets.
ConMambaStudent = StreamConMambaStudent

# --- 5. Loss Functions ---


def _single_kd_feature_loss(student_feat, teacher_feat, student_lens,
                            teacher_lens=None, chunk=128):
    """One student depth against the final Whisper encoder representation."""
    batch_size, student_steps, _ = student_feat.shape
    if teacher_lens is None:
        teacher_lens = torch.full(
            (batch_size,), teacher_feat.shape[1], dtype=torch.long,
            device=teacher_feat.device)
    student_lens = student_lens.to(device=student_feat.device, dtype=torch.long)
    teacher_lens = teacher_lens.to(device=teacher_feat.device, dtype=torch.long)

    total = student_feat.new_zeros((), dtype=torch.float32)
    n_valid = 0
    for batch_idx in range(batch_size):
        student_len = max(1, min(int(student_lens[batch_idx].item()), student_steps))
        teacher_len = max(
            1, min(int(teacher_lens[batch_idx].item()), teacher_feat.shape[1]))
        target = teacher_feat[
            batch_idx:batch_idx + 1, :teacher_len].float().transpose(1, 2)
        if teacher_len != student_len:
            target = F.interpolate(
                target, size=student_len, mode='linear', align_corners=False)
        target = target.transpose(1, 2)[0]
        for start in range(0, student_len, chunk):
            end = min(start + chunk, student_len)
            cosine = F.cosine_similarity(
                student_feat[batch_idx, start:end].float(),
                target[start:end], dim=-1)
            total = total + (1.0 - cosine).sum()
        n_valid += student_len
    return total / max(n_valid, 1)


def kd_feature_loss(student_features, teacher_feat, student_lens,
                    teacher_lens=None, chunk=128, level_weights=None):
    """Weighted multi-depth cosine distillation from Whisper encoder states.

    Cached teacher tensors are hidden states, not logits, so KL/JS over a
    softmax would erase their geometry.  Every student depth is independently
    aligned from ~25 fps to the valid ~50 fps teacher span before cosine loss.
    """
    if torch.is_tensor(student_features):
        student_features = (student_features,)
    student_features = tuple(student_features)
    if not student_features:
        raise ValueError('student_features must contain at least one projection')
    if level_weights is None:
        configured = tuple(getattr(C, 'kd_tap_weights', ()))
        level_weights = configured if len(configured) == len(student_features) else (
            (1.0,) * len(student_features))
    weights = torch.as_tensor(
        level_weights, dtype=torch.float32, device=student_features[-1].device)
    if weights.numel() != len(student_features) or float(weights.sum()) <= 0:
        raise ValueError(
            f'Invalid KD level weights {level_weights} for {len(student_features)} levels')
    weights = weights / weights.sum()
    losses = [
        _single_kd_feature_loss(
            features, teacher_feat, student_lens,
            teacher_lens=teacher_lens, chunk=chunk)
        for features in student_features
    ]
    return sum(weight * loss for weight, loss in zip(weights, losses))


def language_id_loss(language_logits, language_ids, enc_lens):
    """Framewise auxiliary language ID; causal and absent from decoding inputs."""
    if language_logits.shape[-1] <= 1:
        return language_logits.new_zeros((), dtype=torch.float32)
    targets = language_ids.to(language_logits.device, dtype=torch.long)
    if targets.numel() and targets.max().item() >= language_logits.shape[-1]:
        raise ValueError(
            f'Language id {targets.max().item()} outside router size '
            f'{language_logits.shape[-1]}')
    frame_targets = targets[:, None].expand(-1, language_logits.shape[1])
    losses = F.cross_entropy(
        language_logits.float().reshape(-1, language_logits.shape[-1]),
        frame_targets.reshape(-1), reduction='none', label_smoothing=0.05,
    ).view_as(frame_targets)
    mask = (torch.arange(language_logits.shape[1], device=language_logits.device)[None, :]
            < enc_lens[:, None])
    return losses.masked_select(mask).mean()


def ctc_required_lens(tokens, tok_lens):
    """Minimum CTC frames per sample, including blank slots between repeats.

    PyTorch CTC returns inf when a target cannot be aligned. The simple
    tok_len <= enc_len check misses repeated adjacent tokens, which need an
    extra frame each (for the separating blank). Keep zero_infinity=False below
    so real data bugs stay visible, but catch them here with useful context.
    """
    req = tok_lens.to(dtype=torch.long).clone()
    if tokens.numel() == 0 or tokens.shape[1] <= 1:
        return req
    dev_lens = tok_lens.to(tokens.device)
    for pos in range(tokens.shape[1] - 1):
        valid_pair = (pos + 1) < dev_lens
        repeat_pair = tokens[:, pos] == tokens[:, pos + 1]
        req = req + (valid_pair & repeat_pair).to(req.device, dtype=torch.long)
    return req


def describe_invalid_ctc_batch(batch, enc_lens, tok_lens, tokens=None, limit=5):
    req_lens = tok_lens.to(enc_lens.device)
    if tokens is not None:
        req_lens = ctc_required_lens(tokens, tok_lens).to(enc_lens.device)
    bad_mask = req_lens > enc_lens
    bad = torch.nonzero(bad_mask, as_tuple=False).flatten().tolist()
    if bad:
        print('Invalid CTC samples in batch:')
        for bi in bad[:limit]:
            src_id = batch.get('source_ids', [None] * len(batch['texts']))[bi]
            start = batch.get('chunk_start_s', [None] * len(batch['texts']))[bi]
            end = batch.get('chunk_end_s', [None] * len(batch['texts']))[bi]
            print(
                f'  sample={bi}, enc_len={enc_lens[bi].item()}, '
                f'tok_len={tok_lens[bi].item()}, req_len={req_lens[bi].item()}, '
                f'lang={batch["langs"][bi]}, source={src_id}, '
                f'chunk=({start}, {end}), text={batch["texts"][bi][:120]}'
            )
    return bad


def teacher_confidence_weights(confidence):
    confidence = torch.as_tensor(confidence, dtype=torch.float32)
    confidence = torch.nan_to_num(confidence, nan=1.0, posinf=1.0, neginf=0.0)
    return confidence.clamp(
        min=float(getattr(C, 'confidence_floor', 0.35)), max=1.0
    ).pow(float(getattr(C, 'confidence_power', 0.5)))


def ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens, sample_weights=None):
    assert ctc_logits.shape[-1] == BLANK + 1, (
        f'CTC vocab mismatch: logits={ctc_logits.shape[-1]}, expected={BLANK + 1}'
    )
    enc_lens = enc_lens.to(dtype=torch.long)
    tok_lens = tok_lens.to(dtype=torch.long)

    if tokens.numel() > 0:
        pos = torch.arange(tokens.shape[1], device=tokens.device)[None, :]
        valid = pos < tok_lens.to(tokens.device)[:, None]
        valid_targets = tokens[valid]
        if valid_targets.numel() > 0:
            assert valid_targets.max().item() < BLANK, 'CTC targets must not contain blank ID'
            assert valid_targets.min().item() >= 0, 'CTC targets contain negative IDs'

    req_lens = ctc_required_lens(tokens, tok_lens).to(enc_lens.device)
    if not torch.all(req_lens <= enc_lens):
        bad = torch.nonzero(req_lens > enc_lens, as_tuple=False).flatten().tolist()
        raise ValueError(
            f'Invalid CTC lengths. Examples={bad[:5]}, '
            f'enc_lens={enc_lens[bad[:5]].tolist()}, '
            f'tok_lens={tok_lens[bad[:5]].tolist()}, '
            f'req_lens={req_lens[bad[:5]].tolist()}'
        )

    log_probs = F.log_softmax(ctc_logits.float(), dim=-1).transpose(0, 1)
    losses = F.ctc_loss(
        log_probs, tokens, enc_lens, tok_lens,
        blank=BLANK, zero_infinity=False, reduction='none',
    )
    # Match reduction='mean' (per-target-token normalization), then apply the
    # pseudo-label confidence weights from Whisper's segment avg_logprob.
    losses = losses / tok_lens.to(losses.device).clamp_min(1)
    if sample_weights is None:
        return losses.mean()
    weights = sample_weights.to(losses.device, dtype=losses.dtype)
    if weights.shape != losses.shape:
        raise ValueError(
            f'CTC sample_weights shape {weights.shape} != loss shape {losses.shape}')
    return (losses * weights).sum() / weights.sum().clamp_min(1e-6)


### Chunk cache KDDataset sanity-load

Runs after the distillation definitions create `KDDataset`. It checks the freshly generated working cache when present, otherwise the mounted chunk cache.


In [13]:
# KDDataset-level sanity check for chunk caches.
_sanity_teacher_dir, _sanity_mel_dir = _select_cache_pair(
    C.chunk_teacher_work_dir, C.chunk_mel_work_dir, C.cache_dir, C.mel_dir)

if not os.path.isdir(_sanity_teacher_dir) or not os.path.isdir(_sanity_mel_dir):
    print('Chunk KDDataset sanity skipped: teacher/mel cache dirs are not available yet.')
    print(f'  teacher: {_sanity_teacher_dir}')
    print(f'  mel:     {_sanity_mel_dir}')
else:
    print('Chunk KDDataset sanity loading:')
    print(f'  teacher: {_sanity_teacher_dir}')
    print(f'  mel:     {_sanity_mel_dir}')
    _chunk_sanity_ds = KDDataset(_sanity_teacher_dir, sp, _sanity_mel_dir)
    assert len(_chunk_sanity_ds) > 0, 'Chunk cache is empty.'
    for _i in range(min(3, len(_chunk_sanity_ds))):
        _item = _chunk_sanity_ds[_i]
        assert 'mel' in _item and 'teacher_h' in _item
        assert int(_item['token_ids'].numel()) > 0
        assert int(_item['token_ids'].max().item()) < BLANK
        print(
            f'  {_i}: mel={tuple(_item["mel"].shape)} '
            f'teacher={tuple(_item["teacher_h"].shape)} '
            f'tok_len={int(_item["token_ids"].numel())} '
            f'lang={_item["lang"]} source={_item["source_id"]} '
            f'chunk=({_item["chunk_start_s"]}, {_item["chunk_end_s"]}) '
            f'text={_item["text"][:100]}'
        )
    del _chunk_sanity_ds


Chunk KDDataset sanity loading:
  teacher: /kaggle/input/datasets/leviettrieu369/teacher-cache/teacher_cache
  mel:     /kaggle/input/datasets/leviettrieu369/mel-cache/mel_cache
  ViMD_Dataset_default: 57 shards
  VieNeu-TTS_default: 150 shards
  common_voice_22_0_en: 237 shards
  common_voice_22_0_ja: 123 shards
  common_voice_22_0_ko: 3 shards
  common_voice_22_0_vi: 12 shards
  common_voice_22_0_zh-CN: 227 shards
  multilingual_librispeech_french: 25 shards
  multilingual_librispeech_german: 23 shards
  multilingual_librispeech_spanish: 24 shards
Indexing 881 shards...


Indexing:   0%|          | 0/881 [00:00<?, ?shard/s]

Indexed 112246 samples
Skipped 1 CTC-invalid and 0 token-invalid samples during indexing
  0: mel=(80, 920) teacher=(460, 1280) tok_len=50 lang=vietnamese source=ViMD_Dataset_default:0 chunk=(0.0, 9.199999809265137) text=nghiên cứu học tập các ứng dụng các khoa công nghệ và những ứng dụng tiên tiến để đưa vào trong áp d
  1: mel=(80, 690) teacher=(345, 1280) tok_len=35 lang=vietnamese source=ViMD_Dataset_default:2 chunk=(0.0, 6.900000095367432) text=Trong quá trình mình làm việc để quản lý thông tin các vật tư thiết bị của công ty Điện lực Cao Bằng
  2: mel=(80, 836) teacher=(418, 1280) tok_len=44 lang=vietnamese source=ViMD_Dataset_default:2 chunk=(7.480000019073486, 15.84000015258789) text=cá nhân tôi cảm thấy phần quản lý vật tư thiết bị còn chưa khoa học, rời rạp, không tập trung, mất n


### Training configuration (run before smoke test)

Builds the source-grouped dataset splits, derives the language vocabulary,
constructs StreamConMamba, and defines checkpoint-safe training. The training
cell only launches after the smoke gate validates multi-depth KD, confidence CTC,
language routing, and bounded streaming.


In [ ]:
# ── 6. Build everything ────────────────────────────────────────
print('Loading dataset...')
ds = KDDataset(C.cache_dir, sp, C.mel_dir)
C.num_languages = len(ds.lang_to_id)
C.language_names = tuple(
    name for name, _ in sorted(ds.lang_to_id.items(), key=lambda item: item[1]))
print(f'Multilingual router: {C.num_languages} IDs -> {C.language_names}')
flush()

# Persist splits so resumed sessions never leak train samples into eval.
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
_val_size = max(50, min(2000, len(ds) // 50))   # ~2%, capped at [50, 2000]
_test_size = _val_size                           # symmetric val/test
train_ds, val_ds, test_ds = get_or_create_splits(
    ds, val_size=_val_size, test_size=_test_size,
    seed=C.seed, splits_path=SPLITS_PATH,
)
with open(SPLITS_PATH) as _f:
    ACTIVE_SPLIT_METADATA = json.load(_f)
assert ACTIVE_SPLIT_METADATA.get('strategy') == SPLIT_STRATEGY
print(f'  train: {len(train_ds):,} samples')
print(f'  val:   {len(val_ds):,} samples (held out, deterministic)')
print(f'  test:  {len(test_ds):,} samples (held out for final eval — '
      f'never touched by training)')

print('\nBuilding student model...')
student = ConMambaStudent(C).to(device)
student.enable_grad_ckpt()

# Multi-GPU: DataParallel is notebook-friendly; access internals via .module.
if torch.cuda.device_count() > 1:
    student = torch.nn.DataParallel(student)
    print(f'Wrapped student in DataParallel across {torch.cuda.device_count()} GPUs')
student_core = student.module if isinstance(student, torch.nn.DataParallel) else student

n_params = sum(p.numel() for p in student_core.parameters())
n_trainable = sum(p.numel() for p in student_core.parameters() if p.requires_grad)
print(f'Student: {n_params/1e6:.1f}M params ({n_trainable/1e6:.1f}M trainable)')
flush()


# ── 7. Checkpoint helpers ──────────────────────────────────────
def _atomic_save(state, path):
    """Write to .tmp then rename. POSIX rename is atomic, so a kill mid-write
    leaves the previous checkpoint intact instead of corrupting it."""
    # Fail early on directory/empty paths before torch.save emits opaque errors.
    if not path or path.endswith(os.sep) or os.path.isdir(path):
        raise ValueError(
            f'_atomic_save got an invalid path: {path!r}. '
            f'Expected a full file path like '
            f'{os.path.join(C.ckpt_dir, "frozen_latest.pt")!r}.'
        )
    # Recreate the writable parent if Kaggle cleanup removed it.
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    tmp = path + '.tmp'
    torch.save(state, tmp)
    os.replace(tmp, path)


def _save_path(stage_name, suffix='latest'):
    """Always writable destination — every save uses this."""
    return os.path.join(C.ckpt_dir, f'{stage_name}_{suffix}.pt')


def _load_candidates(stage_name, suffix='latest', include_readonly=True):
    """Ordered list of paths to try when loading.

    Priority:
      1. C.ckpt_dir       — writable. Wins if a prior step in THIS session
                            already saved here (within-session resume).
      2. C.ckpt_load_dirs — read-only prior-session/baseline sources across
                            supported Kaggle layouts, in priority order.
    """
    fname = f'{stage_name}_{suffix}.pt'
    paths = [os.path.join(C.ckpt_dir, fname)]
    if include_readonly:
        load_dirs = getattr(C, 'ckpt_load_dirs', None)
        if load_dirs is None:
            load_dirs = [getattr(C, 'ckpt_load_dir', None)]
        for load_dir in load_dirs:
            if load_dir and load_dir != C.ckpt_dir:
                candidate = os.path.join(load_dir, fname)
                if candidate not in paths:
                    paths.append(candidate)
    return paths


# Back-compat shims so the rest of the code (and external scripts) still work.
def _latest_path(stage_name):  # treated as save path everywhere it's used
    return _save_path(stage_name, 'latest')


def _best_path(stage_name):
    return _save_path(stage_name, 'best')


def _unwrap(model):
    """Return the inner module if `model` is DataParallel, else `model`.
    Use this whenever you would call `.state_dict()` or `.load_state_dict()`
    on the model — checkpoints stay portable between single- and multi-GPU
    sessions because the `module.` prefix never enters the on-disk format.
    """
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def _normalize_state_dict(sd):
    """Strip any `module.` prefix from keys, so the dict matches an
    un-wrapped model. No-op if no keys are prefixed."""
    if not any(k.startswith('module.') for k in sd):
        return sd
    return {(k[len('module.'):] if k.startswith('module.') else k): v
            for k, v in sd.items()}


def _load_model_tolerant(model, sd, allowed_missing_prefixes=()):
    """Load `sd` into `model`, dropping any param whose shape no longer matches
    the current model, and erroring on anything else missing/unexpected.

    Migrates a checkpoint across a *deliberate* shape change — e.g. the ctc_head
    going 120001 -> 12001 after the vocab_size fix — without throwing away the
    rest of a long run. Dropped params keep their fresh init; the encoder +
    kl_head (unchanged shapes) are restored. Returns the list of dropped keys
    (truthy => a migration happened)."""
    model_sd = model.state_dict()
    dropped = [k for k, v in sd.items()
               if k in model_sd and v.shape != model_sd[k].shape]
    if dropped:
        print(f'  Dropping {len(dropped)} shape-mismatched param(s) from '
              f'checkpoint (kept fresh init): {dropped}')
        sd = {k: v for k, v in sd.items() if k not in dropped}
    incompat = model.load_state_dict(sd, strict=False)
    if incompat.unexpected_keys:
        raise RuntimeError(f'Unexpected keys in checkpoint: {incompat.unexpected_keys}')
    missing = set(incompat.missing_keys) - set(dropped)
    missing = {k for k in missing if not any(k.startswith(p) for p in allowed_missing_prefixes)}
    if missing:
        raise RuntimeError(f'Checkpoint missing expected params: {sorted(missing)}')
    return dropped


def save_full_checkpoint(path, *, model, optimizer, scheduler, scaler,
                         stage_name, epoch, batch_idx, global_step, best_loss,
                         best_val_metric=float('inf'),
                         epochs_since_improvement=0,
                         early_stopped=False,
                         last_train_loss=None,
                         resume_tag=None,
                         canary_baseline_loss=None,
                         canary_done=False,
                         evaluation_lineage_clean=False):
    """Save everything needed to resume training exactly.

    Includes early-stopping state so resumes across sessions correctly
    track epochs-without-improvement.
    """
    display_loss = best_loss
    if last_train_loss is not None and math.isfinite(float(last_train_loss)):
        display_loss = float(last_train_loss)
    state = {
        # Save unwrapped weights so checkpoints load with or without DP.
        'model':        _unwrap(model).state_dict(),
        'optimizer':    optimizer.state_dict(),
        'scheduler':    scheduler.state_dict(),
        'scaler':       scaler.state_dict(),  # preserves fp16 loss scale
        'stage_name':   stage_name,
        'resume_tag':   resume_tag,
        'epoch':        epoch,
        'batch_idx':    batch_idx,            # for within-epoch resume
        'global_step':  global_step,
        'best_loss':    best_loss,
        'loss':         display_loss,         # human-facing resume/pause metric
        'last_train_loss': last_train_loss,
        # Persist the one-shot canary state. Without this, a resumed stage
        # at step > canary_updates reruns the canary immediately and can
        # reject a checkpoint that already passed it.
        'canary_baseline_loss': canary_baseline_loss,
        'canary_done': bool(canary_done),
        # Split identity describes this run; lineage_clean describes whether
        # every ancestor was also trained with source-disjoint held-out data.
        'split_strategy': ACTIVE_SPLIT_METADATA.get('strategy'),
        'split_fingerprint': ACTIVE_SPLIT_METADATA.get('fingerprint'),
        'evaluation_lineage_clean': bool(evaluation_lineage_clean),
        # Early-stopping state (carried across session boundaries).
        'best_val_metric':           best_val_metric,
        'epochs_since_improvement':  epochs_since_improvement,
        'early_stopped':             early_stopped,
        'config':       C,
        'torch_version': torch.__version__,
        'tokenizer_fingerprint': TOKENIZER_FINGERPRINT,
        'language_names': tuple(C.language_names),
        'model_architecture': MODEL_ARCHITECTURE_TAG,
    }
    _atomic_save(state, path)


def save_best_model_only(path, model, *, stage_name, resume_tag,
                         epoch, global_step, loss, metric=None, collapse=None,
                         evaluation_lineage_clean=False):
    """Save inference weights plus lineage needed for safe selection."""
    _atomic_save({
        'model': _unwrap(model).state_dict(),
        'stage_name': stage_name,
        'resume_tag': resume_tag,
        'epoch': epoch,
        'global_step': global_step,
        'loss': loss,
        'metric': metric,
        'collapse': collapse,
        'split_strategy': ACTIVE_SPLIT_METADATA.get('strategy'),
        'split_fingerprint': ACTIVE_SPLIT_METADATA.get('fingerprint'),
        'evaluation_lineage_clean': bool(evaluation_lineage_clean),
        'tokenizer_fingerprint': TOKENIZER_FINGERPRINT,
        'language_names': tuple(C.language_names),
        'model_architecture': MODEL_ARCHITECTURE_TAG,
    }, path)


def try_load_full_checkpoint(path_or_paths):
    """Try a single path or an ordered list of candidates.

    Returns (state_dict, loaded_from_path) on success, (None, None) on failure.
    Each candidate also looks at its .tmp sibling in case a rename was
    interrupted (extremely rare with atomic os.replace).
    """
    if isinstance(path_or_paths, str):
        candidates = [path_or_paths]
    else:
        candidates = list(path_or_paths)

    for path in candidates:
        if not os.path.exists(path):
            continue
        try:
            state = torch.load(path, map_location='cpu', weights_only=False)
            return state, path
        except Exception as e:
            print(f'Could not load {path}: {e}')
            tmp = path + '.tmp'
            if os.path.exists(tmp):
                try:
                    state = torch.load(tmp, map_location='cpu', weights_only=False)
                    return state, tmp
                except Exception:
                    pass
            # Try the next candidate.

    return None, None


# ── 8. Training Loop ───────────────────────────────────────────
# Bump these whenever architecture, source-lineage rules, or loss recipes change.
# Both full and best checkpoints persist the tag; stale uploaded attempts cannot
# silently shadow the current recipe on a fresh Kaggle session.
SCRATCH_KD_RECIPE_TAG = 'scratch_kd_causal_v1'
SCRATCH_JOINT_RECIPE_TAG = 'scratch_joint_kd_v1'
LEGACY_RECOVER_KD_RECIPE_TAG = 'recover_kd_causal_v2'
RECOVER_KD_RECIPE_TAG = 'recover_kd_stream_router_v3'
RECOVER_CTC_RECIPE_TAG = 'recover_ctc_stream_router_v9'
STREAM_KD_RECIPE_TAG = 'stream_conmamba_multilevel_kd_v1'
STREAM_JOINT_RECIPE_TAG = 'stream_conmamba_multilingual_joint_v1'
MODEL_ARCHITECTURE_TAG = 'stream_conmamba_multilevel_router_v1'


def _require_checkpoint_model_metadata(state, path, allow_missing=False):
    """Protect language-expert IDs and architecture across Kaggle resumes."""
    architecture = state.get('model_architecture')
    language_names = state.get('language_names')
    if architecture is None or language_names is None:
        if allow_missing:
            print(f'WARNING: legacy checkpoint lacks model metadata: {path}')
            return False
        raise RuntimeError(
            f'Checkpoint {path} lacks architecture/language metadata and cannot '
            'safely resume the StreamConMamba optimizer state.')
    if architecture != MODEL_ARCHITECTURE_TAG:
        raise RuntimeError(
            f'Architecture mismatch for {path}: checkpoint={architecture!r}, '
            f'active={MODEL_ARCHITECTURE_TAG!r}')
    expected_languages = tuple(C.language_names)
    if tuple(language_names) != expected_languages:
        raise RuntimeError(
            f'Language-router vocabulary mismatch for {path}: '
            f'checkpoint={tuple(language_names)!r}, active={expected_languages!r}')
    return True
@dataclass
class StageSpec:
    name: str
    source_checkpoint_stage: str = None
    # Primary source stages may require an exact recipe tag. Fallback stages are
    # deliberately allowed to be legacy checkpoints because the CTC head is reset.
    source_checkpoint_tag: str = None
    fallback_source_stages: tuple = ()
    source_checkpoint_suffix: str = 'best'
    fallback_source_suffix: str = 'latest'
    require_source_checkpoint: bool = False
    lr: float = 5e-5
    ctc_head_lr: float = None
    warmup_steps: int = None
    epochs: int = 1
    bs: int = 2
    ga: int = 16
    max_seconds: float = None
    a_kl: float = 0.0
    a_lang: float = 0.0
    ctc_start_weight: float = 0.0
    ctc_end_weight: float = 0.0
    ctc_warmup_steps: int = 0
    reset_ctc_head: bool = False
    ctc_blank_bias: float = 0.0
    best_metric: str = 'wer'
    allow_resume: bool = True
    allow_readonly_resume: bool = False
    fail_fast_on_collapse: bool = False
    # True only for a fresh training lineage that never used legacy
    # per-chunk-split weights. Recovery warm-starts must remain False.
    evaluation_lineage_clean: bool = False
    # Entries may be 'stage' or ('stage', required_resume_tag). A stale
    # downstream checkpoint must never prove that an upstream stage completed.
    skip_if_checkpoint_stages: tuple = ()
    canary_updates: int = 0
    canary_min_relative_loss_drop: float = 0.05
    empty_hypothesis_threshold: float = 0.95
    blank_frame_threshold: float = 0.98
    blank_probability_threshold: float = 0.98
    top_token_threshold: float = 0.90
    resume_tag: str = None
    # Optional input-only tags for a compatible external checkpoint.
    resume_source_tags: tuple = ()
    # Only for deliberate migration from the pre-router architecture.
    allowed_missing_prefixes: tuple = ()


def _collapse_thresholds(spec):
    return {
        'empty_threshold': spec.empty_hypothesis_threshold,
        'blank_threshold': spec.blank_frame_threshold,
        'blank_probability_threshold': spec.blank_probability_threshold,
        'top_token_threshold': spec.top_token_threshold,
    }


def _is_collapsed(metrics):
    return bool(metrics.get('collapse', {}).get('collapsed', False))


def _metric_for_best(metrics, spec, avg_loss):
    if metrics is None:
        return float(avg_loss), 'train/loss'
    if spec.best_metric == 'wer':
        return float(metrics['wer']), 'val/wer'
    if spec.best_metric == 'cer':
        return float(metrics['cer']), 'val/cer'
    return float(metrics['loss']), 'val/loss'


@torch.no_grad()
def evaluate(student, val_dataset, max_seconds=None, bs=2, max_batches=None,
             a_kl=0.0, a_lang=0.0, a_ctc=1.0, collapse_thresholds=None):
    """Run validation and return loss, WER/CER, and CTC-collapse metrics."""
    student.eval()
    max_mel_frames = None if max_seconds is None else int(max_seconds * 100)
    val_sampler = ShardBucketBatchSampler(
        val_dataset, batch_size=bs, generator=None,
        drop_last=False, shuffle=False,
    )
    loader = DataLoader(
        val_dataset, batch_sampler=val_sampler,
        num_workers=C.workers, persistent_workers=(C.workers > 0),
        pin_memory=True,
        collate_fn=lambda b: collate_kd(
            b, max_mel_frames=max_mel_frames, allow_crop=False),
    )

    total_loss = total_kl = total_ctc = total_lang = 0.0
    loss_sample_count = 0
    blank_probability_sum = 0.0
    blank_probability_n = 0
    n_batches = 0
    refs, hyps, langs = [], [], []
    pred_id_seqs, frame_argmax_seqs = [], []

    for batch_idx, batch in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        mel = batch['mel'].to(device, non_blocking=True)
        teacher_h = (batch['teacher_h'].to(device, non_blocking=True)
                     if a_kl > 0 else None)
        teacher_lens = (batch['teacher_lens'].to(device, non_blocking=True)
                        if a_kl > 0 else None)
        tokens = batch['tokens'].to(device, non_blocking=True)
        tok_lens = batch['tok_lens'].to(device, non_blocking=True)
        mel_lens = batch['mel_lens'].to(device, non_blocking=True)
        lang_ids = batch['lang_ids'].to(device, non_blocking=True)
        confidence = teacher_confidence_weights(
            batch['teacher_confidence']).to(device, non_blocking=True)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            ctc_logits, kd_features, language_logits, enc_lens = student(mel, mel_lens)
            zero = ctc_logits.new_zeros((), dtype=torch.float32)
            loss_kl = (kd_feature_loss(
                kd_features, teacher_h, enc_lens, teacher_lens=teacher_lens)
                if a_kl > 0 else zero)
            loss_lang = (language_id_loss(language_logits, lang_ids, enc_lens)
                         if a_lang > 0 else zero)
            loss_ctc = (ctc_loss_fn(
                ctc_logits, tokens, enc_lens, tok_lens,
                sample_weights=confidence)
                if a_ctc > 0 else zero)
            loss = a_kl * loss_kl + a_lang * loss_lang + a_ctc * loss_ctc

        # Shard buckets produce many final batches of size one. Weight batch
        # means by sample count so those tails do not dominate model selection.
        batch_n = int(mel.shape[0])
        total_loss += loss.item() * batch_n
        total_kl += loss_kl.item() * batch_n
        total_ctc += loss_ctc.item() * batch_n
        total_lang += loss_lang.item() * batch_n
        loss_sample_count += batch_n
        n_batches += 1

        # Greedy blank argmax is common while a fresh 5k-way CTC head is still
        # calibrating. Record actual blank probability mass so the fail-fast
        # canary only stops a genuinely degenerate blank distribution.
        logits32 = ctc_logits.float()
        blank_log_probs = logits32[..., BLANK] - torch.logsumexp(logits32, dim=-1)
        valid_frame_mask = (
            torch.arange(ctc_logits.shape[1], device=ctc_logits.device)[None, :]
            < enc_lens[:, None])
        blank_probability_sum += float(
            blank_log_probs.exp().masked_select(valid_frame_mask).sum().cpu())
        blank_probability_n += int(valid_frame_mask.sum().item())

        logits_cpu = logits32.cpu()
        lengths = enc_lens.cpu().tolist()
        pred_ids = greedy_ctc_token_ids(logits_cpu, BLANK, lengths=lengths)
        frame_argmax = logits_cpu.argmax(dim=-1)
        for seq, valid_len in zip(frame_argmax, lengths):
            frame_argmax_seqs.append(seq[:valid_len].tolist())
        decoded = [sp.DecodeIds(ids) for ids in pred_ids]

        pred_id_seqs.extend(pred_ids)
        refs.extend(batch['texts'])
        hyps.extend(decoded)
        langs.extend(batch.get('langs', ['unknown'] * len(decoded)))

        del mel, teacher_h, teacher_lens, tokens, tok_lens, mel_lens
        del lang_ids, confidence, kd_features, language_logits
        del ctc_logits, logits32, blank_log_probs, valid_frame_mask
        del loss_kl, loss_lang, loss_ctc, loss, zero

    student.train()
    n = max(loss_sample_count, 1)
    collapse_thresholds = collapse_thresholds or {}
    collapse = compute_collapse_metrics(
        pred_id_seqs, BLANK,
        frame_argmax_ids=frame_argmax_seqs,
        refs=refs, hyps=hyps, sp_model=sp,
        blank_probability_pct=blank_probability_sum / max(blank_probability_n, 1),
        **collapse_thresholds,
    )
    if a_ctc <= 0:
        # A random/reset CTC head is irrelevant during KD-only warmup and
        # must not block saving the best feature-distillation checkpoint.
        collapse['collapsed'] = False
        collapse['reason'] = 'ctc_disabled'
    return {
        'loss':     total_loss / n,
        'loss_kl':  total_kl / n,
        'loss_ctc': total_ctc / n,
        'loss_lang': total_lang / n,
        'wer':      compute_wer(refs, hyps),
        'cer':      compute_cer(refs, hyps),
        'per_lang': per_language_metrics(refs, hyps, langs),
        'collapse': collapse,
        'n_samples': len(refs),
    }


def _source_checkpoint_candidates(spec):
    """Return (path, required_tag) pairs in source-preference order."""
    stages = []
    if spec.source_checkpoint_stage:
        stages.append((spec.source_checkpoint_stage, spec.source_checkpoint_tag))
    stages.extend((stage, None) for stage in (spec.fallback_source_stages or ()))
    out = []
    for stage, required_tag in stages:
        out.extend((p, required_tag) for p in _load_candidates(
            stage, suffix=spec.source_checkpoint_suffix))
        if spec.fallback_source_suffix != spec.source_checkpoint_suffix:
            out.extend((p, required_tag) for p in _load_candidates(
                stage, suffix=spec.fallback_source_suffix))
    return out


def _is_readonly_checkpoint_path(path):
    if not path:
        return False
    load_dirs = getattr(C, 'ckpt_load_dirs', None)
    if load_dirs is None:
        load_dirs = [getattr(C, 'ckpt_load_dir', None)]
    path_abs = os.path.abspath(path)
    for load_dir in load_dirs:
        if not load_dir or load_dir == C.ckpt_dir:
            continue
        try:
            load_root = os.path.abspath(load_dir)
            if os.path.commonpath([path_abs, load_root]) == load_root:
                return True
        except ValueError:
            continue
    return False


def try_load_compatible_checkpoint(path_or_paths, *, expected_tag=None,
                                   purpose='checkpoint'):
    """Try every candidate, skipping stale recipe tags instead of stopping.

    The old code loaded the first existing path, rejected its tag, and never
    tried the remaining candidates. A stale writable file could therefore hide
    a valid uploaded checkpoint (or vice versa).
    """
    candidates = [path_or_paths] if isinstance(path_or_paths, str) else list(path_or_paths)
    for path in candidates:
        state, loaded_from = try_load_full_checkpoint(path)
        if state is None:
            continue
        found_tag = state.get('resume_tag')
        if expected_tag is not None:
            expected_tags = ({expected_tag} if isinstance(expected_tag, str)
                             else set(expected_tag))
            if found_tag not in expected_tags:
                location = 'read-only' if _is_readonly_checkpoint_path(loaded_from) else 'writable'
                print(
                    f'Ignoring {location} {purpose} {loaded_from}: '
                    f'resume_tag={found_tag!r}, expected one of '
                    f'{sorted(expected_tags)!r}.')
                continue
        return state, loaded_from
    return None, None


def _load_stage_source_weights(student, spec):
    candidates = _source_checkpoint_candidates(spec)
    prev = prev_src = None
    for path, required_tag in candidates:
        prev, prev_src = try_load_compatible_checkpoint(
            path, expected_tag=required_tag,
            purpose=f'{spec.name} source checkpoint')
        if prev is not None:
            break
    if prev is None:
        msg = f'[{spec.name}] No compatible source checkpoint found in {candidates}'
        if spec.require_source_checkpoint:
            raise FileNotFoundError(
                msg + '\nAttach the expected Kaggle checkpoint dataset or run '
                'the required upstream recovery stage first.')
        print(msg + '; starting from current initialization.')
        return

    if prev.get('model_architecture') is not None:
        _require_checkpoint_model_metadata(prev, prev_src)
    sd = _normalize_state_dict(prev['model'])
    allowed_missing = ()
    if spec.reset_ctc_head:
        sd = {k: v for k, v in sd.items() if not k.startswith('ctc_head.')}
        allowed_missing = ('ctc_head.',)
    elif any(k.startswith('ctc_head.') for k in sd):
        _require_checkpoint_tokenizer(prev, prev_src)
    allowed_missing = tuple(set(allowed_missing) | set(
        getattr(spec, 'allowed_missing_prefixes', ()) or ()))
    _load_model_tolerant(
        _unwrap(student), sd, allowed_missing_prefixes=allowed_missing)
    print(f'Loaded compatible source weights for {spec.name} from {prev_src}')


def _reset_ctc_head(student, blank_bias):
    head = _unwrap(student).ctc_head
    head.reset_parameters()
    with torch.no_grad():
        head.bias.zero_()
        head.bias[BLANK] = float(blank_bias)
    print(f'Reset ctc_head to fresh init; blank bias = {blank_bias}')


def train_stage(student, dataset, spec, val_dataset=None):
    """Train one StageSpec with collapse-aware validation and best checkpoints."""
    # The launch cell starts this after smoke; retain a safe direct-call path
    # for notebook debugging and ad-hoc recovery runs.
    if SESSION_START is None:
        start_training_budget()
    stage_name = spec.name
    _core = student.module if isinstance(student, torch.nn.DataParallel) else student
    for p in _core.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in _core.parameters() if p.requires_grad)
    print(f'[{stage_name}] {trainable/1e6:.1f}M trainable parameters')
    print(f'[{stage_name}] loss = {spec.a_kl} * loss_kd + '
          f'{spec.a_lang} * loss_lang + '
          f'{spec.ctc_start_weight}->{spec.ctc_end_weight} * loss_ctc '
          f'over {spec.ctc_warmup_steps} update(s)')

    max_mel_frames = None if spec.max_seconds is None else int(spec.max_seconds * 100)
    _tmp_sampler = ShardBucketBatchSampler(
        dataset, batch_size=spec.bs, generator=None, drop_last=True,
        language_balance_alpha=getattr(C, 'language_balance_alpha', None),
    )
    steps_per_epoch = len(_tmp_sampler) // spec.ga
    del _tmp_sampler
    total_steps = max(1, spec.epochs * steps_per_epoch)

    if spec.ctc_head_lr is not None:
        ctc_params = list(_core.ctc_head.parameters())
        ctc_param_ids = {id(p) for p in ctc_params}
        backbone_params = [
            p for p in _core.parameters()
            if p.requires_grad and id(p) not in ctc_param_ids
        ]
        optimizer = torch.optim.AdamW(
            [
                {'params': backbone_params, 'lr': spec.lr},
                {'params': ctc_params, 'lr': spec.ctc_head_lr},
            ],
            weight_decay=0.01,
        )
        print(f'[{stage_name}] optimizer lr: backbone={spec.lr:g}, '
              f'ctc_head={spec.ctc_head_lr:g}')
    else:
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, student.parameters()),
            lr=spec.lr, weight_decay=0.01,
        )

    warmup_steps = (
        C.warmup if spec.warmup_steps is None else int(spec.warmup_steps)
    )

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda')

    ckpt_path = _save_path(stage_name)
    best_path = _save_path(stage_name, 'best')
    start_epoch = 0
    start_batch_idx = 0
    global_step = 0
    best_loss = float('inf')
    last_train_loss = None
    best_val_metric = float('inf')
    best_tuple = (float('inf'), float('inf'))
    epochs_since_improvement = 0
    es_enabled = getattr(C, 'es_enabled', True) and (val_dataset is not None)
    es_patience = getattr(C, 'es_patience', 3)
    es_min_delta = getattr(C, 'es_min_delta', 1e-4)
    canary_baseline_loss = None
    canary_done = False

    _resume_candidates = []
    if spec.allow_resume:
        _resume_candidates = _load_candidates(
            stage_name, include_readonly=spec.allow_readonly_resume)
    _resume_expected_tags = spec.resume_source_tags or spec.resume_tag
    ckpt, ckpt_src = try_load_compatible_checkpoint(
        _resume_candidates, expected_tag=_resume_expected_tags,
        purpose=f'{stage_name} resume checkpoint')
    if ckpt is not None:
        if spec.ctc_start_weight > 0 or spec.ctc_end_weight > 0:
            _require_checkpoint_tokenizer(ckpt, ckpt_src)
        _require_checkpoint_model_metadata(ckpt, ckpt_src)
        _load_model_tolerant(_unwrap(student), _normalize_state_dict(ckpt['model']))
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt['epoch']
        start_batch_idx = ckpt.get('batch_idx', 0)
        global_step = ckpt['global_step']
        best_loss = ckpt.get('best_loss', float('inf'))
        last_train_loss = ckpt.get('last_train_loss', ckpt.get('loss'))
        best_val_metric = ckpt.get('best_val_metric', float('inf'))
        epochs_since_improvement = ckpt.get('epochs_since_improvement', 0)
        canary_baseline_loss = ckpt.get('canary_baseline_loss')
        canary_done = bool(ckpt.get(
            'canary_done', spec.canary_updates > 0 and global_step >= spec.canary_updates))
        best_tuple = (best_val_metric, best_loss)
        _last_msg = 'n/a' if last_train_loss is None else f'{float(last_train_loss):.4f}'
        print(f'Resumed {stage_name} from {ckpt_src}')
        print(f'  epoch {start_epoch+1}/{spec.epochs}, batch_idx={start_batch_idx}, '
              f'step {global_step}, best_metric={best_val_metric:.4f}, '
              f'best_loss={best_loss:.4f}, last_train_loss={_last_msg}')
    else:
        _downstream_candidates = []
        for _entry in getattr(spec, 'skip_if_checkpoint_stages', ()) or ():
            if isinstance(_entry, (tuple, list)):
                _stage, _required_tag = _entry
            else:
                _stage, _required_tag = _entry, None
            for _suffix in ('latest', 'best'):
                _downstream_candidates.extend(
                    (p, _required_tag) for p in _load_candidates(
                        _stage, suffix=_suffix, include_readonly=True))
        _downstream = _downstream_src = None
        for _path, _required_tag in _downstream_candidates:
            _downstream, _downstream_src = try_load_compatible_checkpoint(
                _path, expected_tag=_required_tag,
                purpose=f'{stage_name} downstream proof')
            if _downstream is not None:
                break
        if _downstream is not None:
            _loss = _downstream.get('loss', _downstream.get('best_loss', 0.0))
            print(f'[{stage_name}] Found compatible downstream checkpoint '
                  f'{_downstream_src}; treating this stage as already complete.')
            return float(_loss), True

        _load_stage_source_weights(student, spec)
        if spec.reset_ctc_head:
            _reset_ctc_head(student, spec.ctc_blank_bias)

    if start_epoch >= spec.epochs:
        print(f'[{stage_name}] Already complete (epoch {start_epoch}/{spec.epochs})')
        return best_loss, True

    def _ctc_weight():
        if spec.ctc_warmup_steps > 0 and global_step < spec.ctc_warmup_steps:
            frac = global_step / max(spec.ctc_warmup_steps, 1)
            return spec.ctc_start_weight + (spec.ctc_end_weight - spec.ctc_start_weight) * frac
        return spec.ctc_end_weight

    def _checkpoint_batch_idx(batch_idx):
        completed = batch_idx + 1
        return completed - (completed % spec.ga)

    def _report_loss():
        if math.isfinite(float(best_loss)):
            return float(best_loss)
        if last_train_loss is not None and math.isfinite(float(last_train_loss)):
            return float(last_train_loss)
        return float('inf')

    def _run_validation(label, epoch_for_log):
        val_max_batches = min(len(val_dataset) // 2, 250)
        print(f'  Running {label} validation on up to {val_max_batches * 2} samples...')
        val_metrics = evaluate(
            student, val_dataset,
            max_seconds=spec.max_seconds,
            bs=2,
            max_batches=val_max_batches,
            a_kl=spec.a_kl,
            a_lang=spec.a_lang,
            a_ctc=spec.ctc_end_weight,
            collapse_thresholds=_collapse_thresholds(spec),
        )
        collapse = val_metrics['collapse']
        print(f'  {label} val - loss={val_metrics["loss"]:.4f} '
              f'wer={val_metrics["wer"]*100:.2f}% cer={val_metrics["cer"]*100:.2f}% '
              f'empty={collapse["empty_hypothesis_pct"]*100:.1f}% '
              f'blank={collapse["blank_frame_pct"]*100:.1f}% '
              f'blank_p={collapse["blank_probability_pct"]*100:.1f}% '
              f'top={collapse["top_ratio"]*100:.1f}% '
              f'collapsed={collapse["collapsed"]} ({collapse["reason"]}) '
              f'(n={val_metrics["n_samples"]})')
        payload = {
            f'{stage_name}/{label}/loss': val_metrics['loss'],
            f'{stage_name}/{label}/loss_kl': val_metrics['loss_kl'],
            f'{stage_name}/{label}/loss_ctc': val_metrics['loss_ctc'],
            f'{stage_name}/{label}/loss_lang': val_metrics['loss_lang'],
            f'{stage_name}/{label}/wer': val_metrics['wer'],
            f'{stage_name}/{label}/cer': val_metrics['cer'],
            f'{stage_name}/{label}/collapsed': int(collapse['collapsed']),
            f'{stage_name}/{label}/blank_frame_pct': collapse['blank_frame_pct'],
            f'{stage_name}/{label}/blank_probability_pct': collapse['blank_probability_pct'],
            f'{stage_name}/{label}/empty_hypothesis_pct': collapse['empty_hypothesis_pct'],
            f'{stage_name}/{label}/top_token_ratio': collapse['top_ratio'],
            f'{stage_name}/{label}/avg_pred_token_len': collapse['avg_pred_token_len'],
            f'{stage_name}/{label}/avg_ref_token_len': collapse['avg_ref_token_len'],
            f'{stage_name}/{label}/epoch': epoch_for_log,
        }
        wandb_run.log(payload)
        return val_metrics

    # Establish a like-for-like validation baseline after source loading and
    # CTC-head reset. A single all-blank snapshot is common early in CTC; only a
    # collapsed snapshot with insufficient loss improvement is fail-fast worthy.
    if (spec.canary_updates and not canary_done and val_dataset is not None and
            canary_baseline_loss is None and time_left_seconds() > 600):
        baseline = _run_validation('canary_baseline', start_epoch)
        canary_baseline_loss = float(baseline['loss'])
        print(f'  Canary baseline loss={canary_baseline_loss:.4f}')

    student.train()
    last_save = time.monotonic()
    save_interval_s = CKPT_EVERY_MINUTES * 60

    for epoch in range(start_epoch, spec.epochs):
        epoch_gen = torch.Generator().manual_seed(C.seed * 1_000_003 + epoch)
        train_sampler = ShardBucketBatchSampler(
            dataset, batch_size=spec.bs, generator=epoch_gen, drop_last=True,
            language_balance_alpha=getattr(C, 'language_balance_alpha', None),
        )
        loader = DataLoader(
            dataset, batch_sampler=train_sampler,
            num_workers=C.workers, persistent_workers=(C.workers > 0),
            pin_memory=True,
            collate_fn=lambda b: collate_kd(
                b, max_mel_frames=max_mel_frames, allow_crop=False),
        )

        skip_batches = start_batch_idx if epoch == start_epoch else 0
        if skip_batches:
            print(f'  Skipping first {skip_batches} batches of epoch {epoch+1} (resume)')
        start_batch_idx = 0

        epoch_loss = epoch_kl = epoch_ctc = epoch_lang = 0.0
        n_batches = 0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(loader, desc=f'[{stage_name}] E{epoch+1}/{spec.epochs}',
                    unit='batch', leave=True, initial=skip_batches,
                    total=len(loader))
        time_up = False

        for batch_idx, batch in enumerate(loader):
            if batch_idx < skip_batches:
                pbar.update(0)
                continue
            pbar.update(1)

            mel = batch['mel'].to(device, non_blocking=True)
            teacher_h = (batch['teacher_h'].to(device, non_blocking=True)
                         if spec.a_kl > 0 else None)
            teacher_lens = (batch['teacher_lens'].to(device, non_blocking=True)
                            if spec.a_kl > 0 else None)
            tokens = batch['tokens'].to(device, non_blocking=True)
            tok_lens = batch['tok_lens'].to(device, non_blocking=True)
            mel_lens = batch['mel_lens'].to(device, non_blocking=True)
            lang_ids = batch['lang_ids'].to(device, non_blocking=True)
            confidence = teacher_confidence_weights(
                batch['teacher_confidence']).to(device, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                ctc_logits, kd_features, language_logits, enc_lens = student(mel, mel_lens)
                zero = ctc_logits.new_zeros((), dtype=torch.float32)
                loss_kl = (kd_feature_loss(
                    kd_features, teacher_h, enc_lens, teacher_lens=teacher_lens)
                    if spec.a_kl > 0 else zero)
                loss_lang = (language_id_loss(language_logits, lang_ids, enc_lens)
                             if spec.a_lang > 0 else zero)
                ctc_w = _ctc_weight()
                if ctc_w > 0:
                    bad = describe_invalid_ctc_batch(
                        batch, enc_lens, tok_lens, tokens)
                    if bad:
                        loss_log.log('invalid_ctc_batch', stage=stage_name,
                                     epoch=epoch, batch_idx=batch_idx)
                        raise RuntimeError(
                            'Invalid CTC batch; see printed examples above.')
                    loss_ctc = ctc_loss_fn(
                        ctc_logits, tokens, enc_lens, tok_lens,
                        sample_weights=confidence)
                else:
                    loss_ctc = ctc_logits.new_zeros((), dtype=torch.float32)
                raw_loss = (
                    spec.a_kl * loss_kl + spec.a_lang * loss_lang +
                    ctc_w * loss_ctc)
                if not (torch.isfinite(loss_kl) and torch.isfinite(loss_lang) and
                        torch.isfinite(loss_ctc) and torch.isfinite(raw_loss)):
                    loss_log.log(
                        'nonfinite_loss', stage=stage_name, epoch=epoch,
                        batch_idx=batch_idx, loss=float(raw_loss.detach().float().cpu()),
                        loss_kl=float(loss_kl.detach().float().cpu()),
                        loss_ctc=float(loss_ctc.detach().float().cpu()),
                        loss_lang=float(loss_lang.detach().float().cpu()),
                        ctc_weight=ctc_w,
                    )
                    raise RuntimeError(
                        f'Non-finite loss in {stage_name}: '
                        f'loss={raw_loss.detach().float().cpu().item()}, '
                        f'kl={loss_kl.detach().float().cpu().item()}, '
                        f'ctc={loss_ctc.detach().float().cpu().item()}, '
                        f'lang={loss_lang.detach().float().cpu().item()}, '
                        f'ctc_weight={ctc_w}')
                last_train_loss = float(raw_loss.detach().float().cpu())
                loss = raw_loss / spec.ga

            scaler.scale(loss).backward()

            if (batch_idx + 1) % spec.ga == 0:
                if C.gc_norm:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(student.parameters(), C.gc_norm)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                    global_step += 1
                    step_loss = loss.item() * spec.ga
                    wandb_run.log({
                        f'{stage_name}/train/loss': step_loss,
                        f'{stage_name}/train/loss_kl': loss_kl.item(),
                        f'{stage_name}/train/loss_ctc': loss_ctc.item(),
                        f'{stage_name}/train/loss_lang': loss_lang.item(),
                        f'{stage_name}/train/ctc_weight': ctc_w,
                        f'{stage_name}/train/lr': scheduler.get_last_lr()[0],
                        f'{stage_name}/train/grad_scale': scaler.get_scale(),
                        f'{stage_name}/train/gpu_mem_gb': torch.cuda.memory_allocated() / 1e9,
                        f'{stage_name}/train/elapsed_h': training_elapsed_seconds() / 3600,
                        f'{stage_name}/train/global_step': global_step,
                        f'{stage_name}/train/epoch_frac': epoch + (batch_idx + 1) / max(len(loader), 1),
                    })
                    loss_log.log(
                        'train_step', stage=stage_name, epoch=epoch,
                        batch_idx=batch_idx, global_step=global_step,
                        loss=step_loss, loss_kl=loss_kl.item(),
                        loss_ctc=loss_ctc.item(), loss_lang=loss_lang.item(),
                        ctc_weight=ctc_w,
                        lr=scheduler.get_last_lr()[0],
                        grad_scale=scaler.get_scale(),
                        gpu_mem_gb=torch.cuda.memory_allocated() / 1e9,
                    )
                else:
                    loss_log.log('scaler_skip', stage=stage_name, epoch=epoch,
                                 batch_idx=batch_idx, scale_before=scale_before,
                                 scale_after=scaler.get_scale())
                    if batch_idx % C.log_every == 0:
                        pbar.write(f'  [scaler skip] inf/nan grads; scale '
                                   f'{scale_before:g} -> {scaler.get_scale():g}')

            epoch_loss += loss.item() * spec.ga
            epoch_kl += loss_kl.item()
            epoch_ctc += loss_ctc.item()
            epoch_lang += loss_lang.item()
            n_batches += 1

            del mel, teacher_h, teacher_lens, tokens, tok_lens, mel_lens
            del lang_ids, confidence, kd_features, language_logits
            del ctc_logits, loss_kl, loss_lang, loss_ctc, loss, zero

            now = time.monotonic()

            if (spec.canary_updates and not canary_done and val_dataset is not None and
                    global_step >= spec.canary_updates and time_left_seconds() > 300):
                canary_done = True
                canary = _run_validation('canary', epoch + (batch_idx + 1) / max(len(loader), 1))
                relative_drop = None
                if (canary_baseline_loss is not None and
                        math.isfinite(canary_baseline_loss)):
                    relative_drop = (
                        canary_baseline_loss - float(canary['loss'])) / max(
                            abs(canary_baseline_loss), 1e-8)
                has_progress = (
                    relative_drop is not None and
                    relative_drop >= spec.canary_min_relative_loss_drop)
                loss_log.log(
                    'canary', stage=stage_name, global_step=global_step,
                    collapsed=canary['collapse']['collapsed'],
                    collapse_reason=canary['collapse']['reason'],
                    val_wer=canary['wer'], val_loss=canary['loss'],
                    baseline_loss=canary_baseline_loss,
                    relative_loss_drop=relative_drop,
                )
                if _is_collapsed(canary) and has_progress:
                    pbar.write(
                        f'  Canary output is collapsed but validation loss improved '
                        f'{relative_drop*100:.1f}% from baseline; continuing.')
                elif spec.fail_fast_on_collapse and _is_collapsed(canary):
                    # Save the diagnostic state before raising; otherwise a
                    # fail-fast canary discards the only evidence from the run.
                    save_full_checkpoint(
                        ckpt_path, model=student, optimizer=optimizer,
                        scheduler=scheduler, scaler=scaler,
                        stage_name=stage_name, epoch=epoch,
                        batch_idx=_checkpoint_batch_idx(batch_idx),
                        global_step=global_step, best_loss=best_loss,
                        best_val_metric=best_val_metric,
                        epochs_since_improvement=epochs_since_improvement,
                        last_train_loss=last_train_loss,
                        resume_tag=spec.resume_tag,
                        canary_baseline_loss=canary_baseline_loss,
                        canary_done=canary_done,
                    evaluation_lineage_clean=spec.evaluation_lineage_clean)
                    raise RuntimeError(
                        f'[{stage_name}] Canary collapsed without sufficient loss '
                        f'progress (relative_drop={relative_drop}, '
                        f'metrics={canary["collapse"]}).')

            # Validation may take minutes; refresh the timestamp before the
            # periodic/deadline checks so a canary cannot run past the Kaggle
            # budget without writing a resumable checkpoint.
            now = time.monotonic()
            if now - last_save >= save_interval_s:
                save_full_checkpoint(
                    ckpt_path, model=student, optimizer=optimizer,
                    scheduler=scheduler, scaler=scaler,
                    stage_name=stage_name, epoch=epoch,
                    batch_idx=_checkpoint_batch_idx(batch_idx),
                    global_step=global_step, best_loss=best_loss,
                    best_val_metric=best_val_metric,
                    epochs_since_improvement=epochs_since_improvement,
                    last_train_loss=last_train_loss,
                    resume_tag=spec.resume_tag,
                    canary_baseline_loss=canary_baseline_loss,
                    canary_done=canary_done,
                    evaluation_lineage_clean=spec.evaluation_lineage_clean)
                last_save = now
                pbar.write(f'saved @ step {global_step} (time left: {fmt_hms(time_left_seconds())})')

            if now >= SESSION_DEADLINE:
                save_full_checkpoint(
                    ckpt_path, model=student, optimizer=optimizer,
                    scheduler=scheduler, scaler=scaler,
                    stage_name=stage_name, epoch=epoch,
                    batch_idx=_checkpoint_batch_idx(batch_idx),
                    global_step=global_step, best_loss=best_loss,
                    best_val_metric=best_val_metric,
                    epochs_since_improvement=epochs_since_improvement,
                    last_train_loss=last_train_loss,
                    resume_tag=spec.resume_tag,
                    canary_baseline_loss=canary_baseline_loss,
                    canary_done=canary_done,
                    evaluation_lineage_clean=spec.evaluation_lineage_clean)
                pbar.write(f'\nSession budget reached. Saved checkpoint to {ckpt_path}.')
                time_up = True
                break

            if batch_idx % C.log_every == 0:
                pbar.set_postfix_str(
                    f'loss={epoch_loss/max(n_batches, 1):.3f} '
                    f'mem={torch.cuda.memory_allocated()/1e9:.1f}G '
                    f'lr={scheduler.get_last_lr()[0]:.2e} '
                    f'left={fmt_hms(time_left_seconds())}')

        pbar.close()
        if time_up:
            return _report_loss(), False

        avg_loss = epoch_loss / max(n_batches, 1)
        avg_kl = epoch_kl / max(n_batches, 1)
        avg_ctc = epoch_ctc / max(n_batches, 1)
        avg_lang = epoch_lang / max(n_batches, 1)
        print(f'E{epoch + 1} avg - loss={avg_loss:.4f} kd={avg_kl:.4f} '
              f'ctc={avg_ctc:.4f} lang={avg_lang:.4f}')

        val_metrics = None
        epoch_payload = {
            f'{stage_name}/epoch_avg/loss': avg_loss,
            f'{stage_name}/epoch_avg/loss_kl': avg_kl,
            f'{stage_name}/epoch_avg/loss_ctc': avg_ctc,
            f'{stage_name}/epoch_avg/loss_lang': avg_lang,
            f'{stage_name}/epoch_avg/epoch': epoch + 1,
            f'{stage_name}/epoch_avg/global_step': global_step,
        }

        if val_dataset is not None and time_left_seconds() > 300:
            val_metrics = _run_validation('val', epoch + 1)
            collapse = val_metrics['collapse']
            epoch_payload.update({
                f'{stage_name}/val/loss': val_metrics['loss'],
                f'{stage_name}/val/loss_kl': val_metrics['loss_kl'],
                f'{stage_name}/val/loss_ctc': val_metrics['loss_ctc'],
                f'{stage_name}/val/loss_lang': val_metrics['loss_lang'],
                f'{stage_name}/val/wer': val_metrics['wer'],
                f'{stage_name}/val/cer': val_metrics['cer'],
                f'{stage_name}/val/collapsed': int(collapse['collapsed']),
                f'{stage_name}/val/blank_frame_pct': collapse['blank_frame_pct'],
                f'{stage_name}/val/blank_probability_pct': collapse['blank_probability_pct'],
                f'{stage_name}/val/empty_hypothesis_pct': collapse['empty_hypothesis_pct'],
                f'{stage_name}/val/top_token_ratio': collapse['top_ratio'],
                f'{stage_name}/val/avg_pred_token_len': collapse['avg_pred_token_len'],
                f'{stage_name}/val/avg_ref_token_len': collapse['avg_ref_token_len'],
                f'{stage_name}/val/epoch': epoch + 1,
            })
            for _lang, _m in val_metrics.get('per_lang', {}).items():
                epoch_payload[f'{stage_name}/val/cer_{_lang}'] = _m['cer']
            torch.cuda.empty_cache()

        current_metric, best_source = _metric_for_best(val_metrics, spec, avg_loss)
        current_loss = val_metrics['loss'] if val_metrics is not None else avg_loss
        collapsed = _is_collapsed(val_metrics) if val_metrics is not None else False
        improved = ((current_metric < best_tuple[0] - es_min_delta) or
                    (abs(current_metric - best_tuple[0]) <= es_min_delta and
                     current_loss < best_tuple[1] - es_min_delta))

        if es_enabled:
            if improved and not collapsed:
                epochs_since_improvement = 0
                best_val_metric = current_metric
                print(f'  early-stop: {best_source}={current_metric:.5f} '
                      f'(best so far) - counter reset')
            else:
                epochs_since_improvement += 1
                suffix = 'collapsed' if collapsed else 'no improvement'
                print(f'  early-stop: {best_source}={current_metric:.5f} '
                      f'(best={best_val_metric:.5f}) - {suffix} '
                      f'{epochs_since_improvement}/{es_patience}')
            epoch_payload[f'{stage_name}/val/best_{spec.best_metric}'] = best_val_metric
            epoch_payload[f'{stage_name}/val/epochs_without_improvement'] = epochs_since_improvement

        wandb_run.log(epoch_payload)
        loss_log.log(
            'epoch_end', stage=stage_name, epoch=epoch + 1,
            global_step=global_step, train_loss=avg_loss,
            train_loss_kl=avg_kl, train_loss_ctc=avg_ctc,
            train_loss_lang=avg_lang,
            val_loss=epoch_payload.get(f'{stage_name}/val/loss'),
            val_loss_kl=epoch_payload.get(f'{stage_name}/val/loss_kl'),
            val_loss_ctc=epoch_payload.get(f'{stage_name}/val/loss_ctc'),
            val_loss_lang=epoch_payload.get(f'{stage_name}/val/loss_lang'),
            val_wer=epoch_payload.get(f'{stage_name}/val/wer'),
            collapsed=collapsed,
        )

        if improved and not collapsed:
            best_loss = current_loss
            best_tuple = (current_metric, current_loss)
            best_val_metric = current_metric

        save_full_checkpoint(
            ckpt_path, model=student, optimizer=optimizer,
            scheduler=scheduler, scaler=scaler,
            stage_name=stage_name, epoch=epoch + 1,
            batch_idx=0, global_step=global_step,
            best_loss=best_loss,
            best_val_metric=best_val_metric,
            epochs_since_improvement=epochs_since_improvement,
                    last_train_loss=last_train_loss,
                    resume_tag=spec.resume_tag,
                    canary_baseline_loss=canary_baseline_loss,
                    canary_done=canary_done,
                    evaluation_lineage_clean=spec.evaluation_lineage_clean)
        last_save = time.monotonic()

        if improved and not collapsed:
            save_best_model_only(
                best_path, student, stage_name=stage_name,
                resume_tag=spec.resume_tag, epoch=epoch,
                global_step=global_step, loss=best_loss,
                metric={spec.best_metric: current_metric, 'source': best_source},
                collapse=val_metrics.get('collapse') if val_metrics else None,
                evaluation_lineage_clean=spec.evaluation_lineage_clean,
            )
            print(f'  * New best {best_source}: metric={current_metric:.4f}, loss={best_loss:.4f}')
        elif collapsed:
            print('  Not saving best checkpoint because validation is collapsed.')

        if spec.fail_fast_on_collapse and collapsed:
            _drop = None
            if canary_baseline_loss is not None and math.isfinite(canary_baseline_loss):
                _drop = (canary_baseline_loss - current_loss) / max(
                    abs(canary_baseline_loss), 1e-8)
            if _drop is None or _drop < spec.canary_min_relative_loss_drop:
                raise RuntimeError(
                    f'[{stage_name}] Validation collapsed without loss progress; '
                    f'relative_drop={_drop}.')
            print(f'  Collapsed greedy output, but loss improved {_drop*100:.1f}% '
                  'from baseline; continuing and letting early stopping decide.')

        if es_enabled and epochs_since_improvement >= es_patience:
            print(f'\n[{stage_name}] EARLY STOPPING at epoch {epoch+1}/{spec.epochs}.')
            save_full_checkpoint(
                ckpt_path, model=student, optimizer=optimizer,
                scheduler=scheduler, scaler=scaler,
                stage_name=stage_name, epoch=spec.epochs,
                batch_idx=0, global_step=global_step,
                best_loss=best_loss,
                best_val_metric=best_val_metric,
                epochs_since_improvement=epochs_since_improvement,
                early_stopped=True,
                last_train_loss=last_train_loss,
                resume_tag=spec.resume_tag,
                canary_baseline_loss=canary_baseline_loss,
                canary_done=canary_done,
                evaluation_lineage_clean=spec.evaluation_lineage_clean,
            )
            loss_log.log('early_stop', stage=stage_name, epoch=epoch + 1,
                         best_loss=best_loss, best_val_metric=best_val_metric)
            return best_loss, True

    print(f'[{stage_name}] Done - best_loss={best_loss:.4f}\n')
    return best_loss, True


# Set this only after the entire definition/setup cell succeeds. The launch
# cell checks it because Kaggle kernels can retain an older train_stage and
# compute_collapse_metrics after the notebook source has been edited.
TRAINING_DEFINITION_VERSION = 'stream-conmamba-multilingual-kd-v5'
print(f'Training definitions ready: {TRAINING_DEFINITION_VERSION}')


### Smoke test (run after training configuration — gates training)

Runs ~50 updates through every objective and one bounded-streaming inference.
It checks the fp32 selective-scan island, CTC lengths, finite gradients, T4
memory, router shape, and chunk-boundary output before setting `_SMOKE_PASSED`.


In [ ]:
# Smoke test: run after training config; gates the training process cell.
# Checks kernels, data loading, memory, and loss before a long run.
from torch.utils.data import Subset, DataLoader
import statistics
import time

N_SMOKE = 50
SMOKE_BS = C.un_bs
# The real recovery schedule runs KD-only warmup then joint KD+CTC.
# Smoke intentionally enables both terms so one short run validates both data
# paths and all training heads, independent of the stage curriculum.
SMOKE_A_KL = 1.0
SMOKE_A_CTC = 0.30
SMOKE_A_LANG = C.a_lang
SMOKE_BACKBONE_LR = 3e-5
SMOKE_CTC_HEAD_LR = 1e-3
SMOKE_BLANK_BIAS = 0.0

# ── Build (mirrors the training cell so the smoke reflects real conditions)
print('Loading dataset...')
ds = KDDataset(C.cache_dir, sp, C.mel_dir)
C.num_languages = len(ds.lang_to_id)
C.language_names = tuple(
    name for name, _ in sorted(ds.lang_to_id.items(), key=lambda item: item[1]))
flush()

# Reuse persisted splits so train/val/test stay stable across sessions.
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
_val_size = max(50, min(2000, len(ds) // 50))
_test_size = _val_size
train_ds, val_ds, test_ds = get_or_create_splits(
    ds, val_size=_val_size, test_size=_test_size,
    seed=C.seed, splits_path=SPLITS_PATH,
)
print(f'  train: {len(train_ds):,} samples')
print(f'  val:   {len(val_ds):,} samples (held out, deterministic)')
print(f'  test:  {len(test_ds):,} samples (held out for final eval — '
      f'NOT touched during training)')

print('\nBuilding student model...')
smoke_student = ConMambaStudent(C).to(device)
smoke_student.enable_grad_ckpt()
if torch.cuda.device_count() > 1:
    smoke_student = torch.nn.DataParallel(smoke_student)
    print(f'Smoke wrapped in DataParallel across {torch.cuda.device_count()} GPUs')
_reset_ctc_head(smoke_student, SMOKE_BLANK_BIAS)

n_params = sum(p.numel() for p in _unwrap(smoke_student).parameters())
n_trainable = sum(p.numel() for p in _unwrap(smoke_student).parameters() if p.requires_grad)
print(f'Student: {n_params/1e6:.1f}M params ({n_trainable/1e6:.1f}M trainable)')
flush()

# Mirror training shape with the full trainable model and exercise both losses.
trainable = sum(p.numel() for p in smoke_student.parameters() if p.requires_grad)
print(f"Smoke test: {N_SMOKE} steps, bs={SMOKE_BS}, "
      f"{trainable/1e6:.1f}M trainable (KD+CTC diagnostic)")
print(f"  smoke loss = {SMOKE_A_KL} * loss_kl + "
      f"{SMOKE_A_CTC} * loss_ctc + {SMOKE_A_LANG} * loss_lang")

# Self-contained subset sized to exercise several shard transitions.
smoke_size = min(2000, len(train_ds))
smoke_ds = Subset(train_ds, list(range(smoke_size)))
_smoke_core = _unwrap(smoke_student)
_smoke_ctc_params = list(_smoke_core.ctc_head.parameters())
_smoke_ctc_ids = {id(p) for p in _smoke_ctc_params}
_smoke_backbone_params = [
    p for p in _smoke_core.parameters() if id(p) not in _smoke_ctc_ids]
opt = torch.optim.AdamW([
    {'params': _smoke_backbone_params, 'lr': SMOKE_BACKBONE_LR},
    {'params': _smoke_ctc_params, 'lr': SMOKE_CTC_HEAD_LR},
], weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

max_mel_frames = None  # chunk-level CTC: pad only, never crop

# Match real training: workers + shard buckets avoid FUSE np.load bottlenecks.
smoke_sampler = ShardBucketBatchSampler(
    smoke_ds, batch_size=SMOKE_BS,
    generator=torch.Generator().manual_seed(C.seed),
    drop_last=True,
    language_balance_alpha=C.language_balance_alpha,
)
loader = DataLoader(
    smoke_ds, batch_sampler=smoke_sampler,
    num_workers=C.workers, persistent_workers=(C.workers > 0),
    pin_memory=True, prefetch_factor=2 if C.workers > 0 else None,
    collate_fn=lambda b: collate_kd(b, max_mel_frames=max_mel_frames, allow_crop=False),
)
_bucket_sizes = sorted(len(b) for b in smoke_sampler.shard_buckets)
_nonempty = [s for s in _bucket_sizes if s >= SMOKE_BS]
print(f'  ShardBucketBatchSampler: {len(smoke_sampler)} batches '
      f'across {len(smoke_sampler.shard_buckets)} shards '
      f'({len(_nonempty)} of which have >= {SMOKE_BS} samples)')
print(f'  bucket-size percentiles: '
      f'min={_bucket_sizes[0]}, p25={_bucket_sizes[len(_bucket_sizes)//4]}, '
      f'p50={_bucket_sizes[len(_bucket_sizes)//2]}, '
      f'p75={_bucket_sizes[3*len(_bucket_sizes)//4]}, max={_bucket_sizes[-1]}')

smoke_student.train()
step_times, load_times, compute_times = [], [], []
losses_kl, losses_ctc, losses_lang = [], [], []
torch.cuda.reset_peak_memory_stats()

print(f"\n{'step':>4} {'loss':>7} {'l_kd':>8} {'l_ctc':>7} {'l_lang':>7} "
      f"{'mem_GB':>7} {'load':>5} {'cmp':>5} {'tot':>5}")
print("-" * 60)

# Time worker-queue dequeue separately from GPU compute.
it = iter(loader)
t_prev = time.monotonic()
for step in range(N_SMOKE):
    # ── 1. Data load (worker queue dequeue + collate already done in workers)
    t_load_start = time.monotonic()
    try:
        batch = next(it)
    except StopIteration:
        break
    t_load_done = time.monotonic()

    mel = batch['mel'].to(device, non_blocking=True)
    teacher_h = batch['teacher_h'].to(device, non_blocking=True)
    teacher_lens = batch['teacher_lens'].to(device, non_blocking=True)
    tokens = batch['tokens'].to(device, non_blocking=True)
    tok_lens = batch['tok_lens'].to(device, non_blocking=True)
    mel_lens = batch['mel_lens'].to(device, non_blocking=True)
    lang_ids = batch['lang_ids'].to(device, non_blocking=True)
    confidence = teacher_confidence_weights(
        batch['teacher_confidence']).to(device, non_blocking=True)

    # ── 2. Compute
    t_compute_start = time.monotonic()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        ctc_logits, kd_features, language_logits, enc_lens = smoke_student(
            mel, mel_lens)
        loss_kl = kd_feature_loss(
            kd_features, teacher_h, enc_lens, teacher_lens=teacher_lens)
        loss_lang = language_id_loss(language_logits, lang_ids, enc_lens)
        bad = describe_invalid_ctc_batch(batch, enc_lens, tok_lens, tokens)
        if bad:
            raise RuntimeError(
                'Smoke failed: chunk target length exceeds encoder length.')
        loss_ctc = ctc_loss_fn(
            ctc_logits, tokens, enc_lens, tok_lens,
            sample_weights=confidence)
        loss = (SMOKE_A_KL * loss_kl + SMOKE_A_CTC * loss_ctc +
                SMOKE_A_LANG * loss_lang)

    if not torch.isfinite(loss):
        _SMOKE_PASSED = False
        raise RuntimeError(
            f"Non-finite loss at smoke step {step}: "
            f"loss={loss.item()} kd={loss_kl.item()} ctc={loss_ctc.item()} "
            f"lang={loss_lang.item()}. "
            "Mamba SSM likely overflowed fp16 — confirm the fp32 island is in forward()."
        )

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(smoke_student.parameters(), C.gc_norm)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    t_compute_done = time.monotonic()

    load_t = t_load_done - t_load_start
    compute_t = t_compute_done - t_compute_start
    total_t = t_compute_done - t_prev
    t_prev = t_compute_done

    load_times.append(load_t)
    compute_times.append(compute_t)
    step_times.append(total_t)
    losses_kl.append(loss_kl.item())
    losses_ctc.append(loss_ctc.item())
    losses_lang.append(loss_lang.item())

    if step < 5 or step % 10 == 0 or step == N_SMOKE - 1:
        mem_gb = torch.cuda.memory_allocated() / 1e9
        print(f"{step:4d} {loss.item():7.3f} {loss_kl.item():8.4f} "
              f"{loss_ctc.item():7.3f} {loss_lang.item():7.3f} {mem_gb:7.2f} "
              f"{load_t:5.2f} {compute_t:5.2f} {total_t:5.2f}")

# Exercise the actual bounded-memory deployment path once.  This catches
# chunk-boundary/subsampling regressions that a full-sequence smoke cannot see.
_stream_core = _unwrap(smoke_student)
_stream_core.eval()
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    _infer_logits, _infer_lang, _infer_lens = _stream_core.inference_ctc(
        mel[:1], mel_lens[:1])
    _stream_logits, _stream_lang, _stream_lens = _stream_core.streaming_ctc(
        mel[:1], mel_lens[:1])
assert _infer_logits.shape[1] == int(_infer_lens[0]) > 0
assert _infer_logits.shape[-1] == C.vocab_size
assert _infer_lang.shape[-1] == C.num_languages
assert _stream_logits.shape[1] == int(_stream_lens[0]) > 0
assert _stream_logits.shape[-1] == C.vocab_size
assert _stream_lang.shape[-1] == C.num_languages
print(f'Inference path: logits={tuple(_infer_logits.shape)}')
print(f'Streaming path: logits={tuple(_stream_logits.shape)}, '
      f'chunk={C.stream_chunk_ms} ms, left_context={C.stream_left_context_ms} ms')
_stream_core.train()
del _infer_logits, _infer_lang, _infer_lens
del _stream_logits, _stream_lang, _stream_lens, _stream_core

# ── Verdict ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
# Skip warmup steps for medians.
warm = max(2, len(step_times) // 10)


def med(xs):
    return statistics.median(xs[warm:]) if len(xs) > warm else (xs[-1] if xs else 0.0)


med_step = med(step_times)
med_load = med(load_times)
med_compute = med(compute_times)
peak_mem = torch.cuda.max_memory_allocated() / 1e9
kl_first = sum(losses_kl[:5]) / min(5, len(losses_kl))
kl_last = sum(losses_kl[-5:]) / min(5, len(losses_kl))
ctc_first = sum(losses_ctc[:5]) / min(5, len(losses_ctc))
ctc_last = sum(losses_ctc[-5:]) / min(5, len(losses_ctc))

print(f"Median sec/step (after warmup): {med_step:.2f}s")
print(
    f"  - data load: {med_load:.2f}s  ({100*med_load/max(med_step, 1e-3):.0f}% of step)")
print(f"  - compute  : {med_compute:.2f}s  ({100 *
                                             med_compute /
                                             max(med_step, 1e-3):.0f}% of step)")
print(f"Peak GPU memory:                {peak_mem:.2f} GB")
print(
    f"loss_kl  first 5  : last 5: {
        kl_first:.4f}  : {
            kl_last:.4f}  (Δ {
                kl_last -
                kl_first:+.4f})")
print(
    f"loss_ctc first 5  : last 5: {
        ctc_first:.3f}  : {
            ctc_last:.3f}  (Δ {
                ctc_last -
                ctc_first:+.3f})")
print()

# Thresholds are smoke-specific: small batch, no long-run optimizer state.
issues = []
# Step-time: with workers active and compute at ~0.5 s, we should be ≤ 2 s/step.
if med_step > 2.0:
    if med_load > med_compute:
        issues.append(
            f"step {med_step:.1f}s is dataloader-bound "
            f"(load {med_load:.1f}s >> compute {med_compute:.1f}s). "
            f"Try increasing C.workers (currently {C.workers}) or prefetch_factor."
        )
    else:
        issues.append(
            f"step {med_step:.1f}s with compute {med_compute:.1f}s — "
            f"Mamba slow path may be active. Re-run diagnostic_cell."
        )
# Compute floor: fast-path Mamba on T4 bs=4 T=300 frozen ≈ 0.3-0.7s.
if med_compute > 2.0:
    issues.append(
        f"compute {med_compute:.2f}s/step too high for fast-path Mamba "
        "(expected ~0.3-0.7s on T4 bs=4 frozen)"
    )
# A fresh random projection should have cosine loss near 1. A zero here means
# the KD path was skipped (the regression this smoke test is meant to catch).
if max(losses_kl, default=0.0) <= 1e-6:
    issues.append(
        'loss_kl is zero: teacher features were not compared with kl_head output.'
    )
# Require finite/moving CTC; KL is also optimized above but its short-run slope
# is deliberately not a gate because random batches make cosine loss noisy.
if ctc_last >= ctc_first - 0.01:
    issues.append(
        f"loss_ctc not decreasing enough: first5={
            ctc_first:.3f}, last5={
            ctc_last:.3f}. "
        "Run the tiny overfit cell before full training."
    )
# Memory: just sanity-check we didn't accidentally blow past T4 capacity.
if peak_mem > 14.0:
    issues.append(f"peak mem {peak_mem:.1f} GB — close to T4's 16 GB limit; "
                  "reduce bs or max_s1")

if issues:
    _SMOKE_PASSED = False
    print("  Issues detected:")
    for i in issues:
        print(f"  - {i}")
    print("\nDo NOT proceed to full training until these are resolved.")
    print("(_SMOKE_PASSED = False  — the training cell will refuse to start.)")
else:
    _SMOKE_PASSED = True
    print("✅ All checks pass — _SMOKE_PASSED = True")
    print("   Proceed to the training cell below.")

# Free only smoke state. The production `student` from cell 25 remains
# DataParallel-wrapped and untouched for the launch cell.
del smoke_student, _smoke_core, _smoke_ctc_params, _smoke_ctc_ids
del _smoke_backbone_params, opt, scaler, loader, smoke_ds, smoke_sampler, it
torch.cuda.empty_cache()


### CTC length sanity and tiny overfit

Run this before full training on a new chunk cache. It checks chunk CTC lengths and verifies the model can overfit a tiny valid subset.

In [ ]:
# ── Chunk-level CTC sanity + tiny overfit ───────────────────────
from torch.utils.data import Subset, DataLoader
import math

print('\nCTC length sanity:')
_diag_loader = DataLoader(
    val_ds if 'val_ds' in globals() else Subset(ds, list(range(min(16, len(ds))))),
    batch_size=4, shuffle=False, num_workers=0,
    collate_fn=lambda b: collate_kd(b, max_mel_frames=None, allow_crop=False),
)
_diag_model = _unwrap(student).eval() if 'student' in globals(
) else ConMambaStudent(C).to(device).eval()
_batch = next(iter(_diag_loader))
_mel = _batch['mel'].to(device)
_mel_lens = _batch['mel_lens'].to(device)
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    _ctc_logits, _kl, _enc_lens = _diag_model(_mel, _mel_lens)
for j in range(len(_batch['texts'])):
    print(
        f'[{j}] mel_len={_batch["mel_lens"][j].item()} '
        f'enc_len={_enc_lens[j].item()} '
        f'tok_len={_batch["tok_lens"][j].item()} '
        f'lang={_batch["langs"][j]} '
        f'source={_batch["source_ids"][j]} '
        f'text={_batch["texts"][j][:100]}'
    )
_bad = describe_invalid_ctc_batch(
    _batch, _enc_lens.cpu(), _batch['tok_lens'], _batch['tokens'])
if _bad:
    raise RuntimeError(
        'CTC length sanity failed; regenerate/filter shorter chunk targets before training.')


def pick_short_valid_subset(ds, n=8, max_mel_frames=800, max_tok_len=80):
    keep = []
    for i in range(len(ds)):
        item = ds[i]
        mel_len = item['mel'].shape[1]
        tok_len = len(item['token_ids'])
        approx_enc_len = math.ceil(mel_len / 4)
        if mel_len <= max_mel_frames and 0 < tok_len <= max_tok_len and tok_len < approx_enc_len:
            keep.append(i)
        if len(keep) >= n:
            break
    if len(keep) < n:
        raise RuntimeError(f'Only found {len(keep)} short valid chunks; need {n}.')
    return Subset(ds, keep)


RUN_TINY_OVERFIT = globals().get('RUN_TINY_OVERFIT', True)
_TINY_OVERFIT_PASSED = False
if RUN_TINY_OVERFIT:
    print('\nTiny CTC overfit:')
    _base_train = train_ds if 'train_ds' in globals() else ds
    tiny_ds = pick_short_valid_subset(_base_train, n=8)
    tiny_loader = DataLoader(
        tiny_ds, batch_size=4, shuffle=True, num_workers=0,
        collate_fn=lambda b: collate_kd(b, max_mel_frames=None, allow_crop=False),
    )
    tiny_student = ConMambaStudent(C).to(device)
    tiny_student.enable_grad_ckpt()
    tiny_student.train()
    opt = torch.optim.AdamW(tiny_student.parameters(), lr=1e-4)
    losses = []
    it = iter(tiny_loader)
    for step in range(300):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(tiny_loader)
            batch = next(it)
        mel = batch['mel'].to(device)
        mel_lens = batch['mel_lens'].to(device)
        tokens = batch['tokens'].to(device)
        tok_lens = batch['tok_lens'].to(device)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            ctc_logits, _kd_features, _language_logits, enc_lens = tiny_student(mel, mel_lens)
            bad = describe_invalid_ctc_batch(batch, enc_lens, tok_lens, tokens)
            if bad:
                raise RuntimeError('Tiny overfit failed: invalid CTC lengths.')
            loss = ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tiny_student.parameters(), 1.0)
        opt.step()
        losses.append(float(loss.item()))
        if step % 50 == 0 or step == 299:
            pred_ids = greedy_ctc_token_ids(
                ctc_logits.float().detach().cpu(), BLANK,
                lengths=enc_lens.cpu().tolist(),
            )
            decoded = [sp.DecodeIds(ids) for ids in pred_ids]
            collapse = detect_ctc_collapse(pred_ids, BLANK)
            print(f'step={step:03d} loss={loss.item():.4f} collapse={collapse}')
            print('REF:', batch['texts'][0])
            print('HYP:', decoded[0] if decoded[0] else '(empty)')
    first = sum(losses[:20]) / min(20, len(losses))
    last = sum(losses[-20:]) / min(20, len(losses))
    if not last < first:
        raise RuntimeError(
            f'Tiny overfit failed: loss did not decrease ({first:.4f} -> {last:.4f}).')
    _TINY_OVERFIT_PASSED = True
    print(f'Tiny overfit passed: loss {first:.4f} -> {last:.4f}')
    del tiny_student, opt, tiny_loader, tiny_ds
    torch.cuda.empty_cache()
else:
    print('RUN_TINY_OVERFIT=False; skipped tiny overfit loop.')


### Training process (run after smoke passes)

The new `stream_kd → stream_joint` lineage cannot resume an old optimizer state.
With `C.resume_from_checkpoint=True`, an old compatible `scratch_kd` checkpoint
may seed the unchanged CNN/encoder/final KD head as **weights only**; the router
and auxiliary KD heads start fresh. Set it to `False` for a completely new run.
The smoke guard remains mandatory and all new checkpoints write under
`/kaggle/working/edge_asr/checkpoints`.


In [ ]:
# Pre-flight: require _SMOKE_PASSED unless SKIP_SMOKE_GUARD=True.
if not globals().get('SKIP_SMOKE_GUARD', False):
    assert globals().get('_SMOKE_PASSED', False), (
        "Run the smoke-test cell FIRST and confirm it passes "
        "(it sets _SMOKE_PASSED=True). To bypass: SKIP_SMOKE_GUARD = True"
    )


# Fail before allocating GPU time if this Kaggle kernel still holds stale
# checkpoint-lineage or snapshot-only collapse logic.
_EXPECTED_TRAINING_DEFINITION_VERSION = 'stream-conmamba-multilingual-kd-v5'
assert globals().get('TRAINING_DEFINITION_VERSION') == _EXPECTED_TRAINING_DEFINITION_VERSION, (
    'Training definitions are stale or incomplete. Rerun the Training '
    'configuration cell, then rerun the smoke test before launching training. '
    f'Expected {_EXPECTED_TRAINING_DEFINITION_VERSION!r}, got '
    f'{globals().get("TRAINING_DEFINITION_VERSION")!r}.'
)

# Setup and smoke do not consume this budget. Reset on each explicit launch
# so a saved checkpoint can receive a full fresh budget when this cell reruns.
start_training_budget(reset=True)


# Resume preflight: fail before allocating GPU time if the selected
# read-only edge_asr bundle is not attached or has the wrong layout.
# Input mounts are intentionally read-only; only C.ckpt_dir is writable.
if getattr(C, 'resume_from_checkpoint', True):
    # Prefer a same-architecture resume; accept the old final-KD model as
    # a weights-only migration source for the new router/auxiliary heads.
    _resume_probe_stages = (
        ('stream_kd', 'scratch_kd') if C.train_from_scratch
        else ('recover_ctc', 'recover_kd'))
    _resume_probe_paths = []
    for _resume_probe_stage in _resume_probe_stages:
        for _suffix in ('latest', 'best'):
            _resume_probe_paths.extend(
                _load_candidates(_resume_probe_stage, suffix=_suffix,
                                 include_readonly=True))

    _resume_found = [
        _path for _path in _resume_probe_paths
        if os.path.isfile(_path)
    ]
    if not _resume_found:
        _expected_root = (
            '/kaggle/input/datasets/leviettrieu369/'
            'distillation-checkpoint/edge_asr'
        )
        _searched_dirs = sorted(set(
            os.path.dirname(_path) for _path in _resume_probe_paths
        ))
        raise FileNotFoundError(
            f'Resume checkpoint preflight failed: none of '
            f'{_resume_probe_stages!r} had a latest/best checkpoint.\\n'
            f'Expected the attached read-only dataset under: {_expected_root}\\n'
            'Expected layout: <edge_asr>/checkpoints/<stage>_latest.pt\\n'
            f'Searched checkpoint directories: {_searched_dirs}\\n'
            'Attach the distillation-checkpoint Kaggle dataset and rerun '
            'the configuration cell. To intentionally start a fresh lineage, '
            'set C.resume_from_checkpoint = False before launching. '
            f'New checkpoints are written only under {C.ckpt_dir}.'
        )

    print(f'Resume checkpoint preflight passed for {_resume_probe_stage}:')
    for _path in _resume_found:
        _size_mb = os.path.getsize(_path) / (1024 ** 2)
        print(f'  {_path} ({_size_mb:.1f} MiB)')
fresh_specs = [
    StageSpec(
        name='stream_kd',
        # A legacy scratch_kd checkpoint can seed the unchanged CNN/encoder and
        # final kl_head.  The new router and auxiliary KD heads start as an
        # identity/random projection and are fully trained in this stage.
        source_checkpoint_stage='scratch_kd',
        source_checkpoint_tag=SCRATCH_KD_RECIPE_TAG,
        source_checkpoint_suffix='latest',
        fallback_source_suffix='best',
        lr=C.fr_lr,
        epochs=1,
        bs=C.fr_bs,
        ga=C.fr_ga,
        max_seconds=None,
        a_kl=1.0,
        a_lang=C.a_lang,
        ctc_start_weight=0.0,
        ctc_end_weight=0.0,
        reset_ctc_head=True,
        ctc_blank_bias=0.0,
        best_metric='loss',
        allow_resume=True,
        allow_readonly_resume=True,
        require_source_checkpoint=False,
        resume_tag=STREAM_KD_RECIPE_TAG,
        evaluation_lineage_clean=True,
        allowed_missing_prefixes=('kd_aux_heads.', 'language_adapter.'),
        skip_if_checkpoint_stages=(
            ('stream_joint', STREAM_JOINT_RECIPE_TAG),),
        fail_fast_on_collapse=False,
    ),
    StageSpec(
        name='stream_joint',
        source_checkpoint_stage='stream_kd',
        source_checkpoint_tag=STREAM_KD_RECIPE_TAG,
        source_checkpoint_suffix='latest',
        fallback_source_suffix='best',
        lr=3e-5,
        ctc_head_lr=1e-3,
        warmup_steps=200,
        epochs=C.un_epochs,
        bs=C.un_bs,
        ga=C.un_ga,
        max_seconds=None,
        a_kl=1.0,
        a_lang=C.a_lang,
        ctc_start_weight=0.10,
        ctc_end_weight=0.30,
        ctc_warmup_steps=1000,
        reset_ctc_head=True,
        ctc_blank_bias=0.0,
        best_metric='wer',
        allow_resume=True,
        allow_readonly_resume=True,
        require_source_checkpoint=True,
        resume_tag=STREAM_JOINT_RECIPE_TAG,
        evaluation_lineage_clean=True,
        fail_fast_on_collapse=True,
        canary_updates=300,
        canary_min_relative_loss_drop=0.05,
    ),
]


recovery_specs = [
    StageSpec(
        name='recover_kd',
        # Prefer the recovery checkpoint from a prior Kaggle session. If a
        # a tag-compatible downstream recover_ctc checkpoint is already in the
        # uploaded bundle, recover_kd is complete and train_stage will skip it.
        source_checkpoint_stage='recover_kd',
        source_checkpoint_tag=(
            RECOVER_KD_RECIPE_TAG, LEGACY_RECOVER_KD_RECIPE_TAG),
        fallback_source_stages=('recover_ctc', 'frozen', 'unfrozen', 'chunk_ctc'),
        source_checkpoint_suffix='best',
        fallback_source_suffix='latest',
        lr=C.fr_lr,
        epochs=1,
        bs=C.fr_bs,
        ga=C.fr_ga,
        max_seconds=None,
        a_kl=1.0,
        a_lang=C.a_lang,
        ctc_start_weight=0.0,
        ctc_end_weight=0.0,
        ctc_warmup_steps=0,
        reset_ctc_head=True,
        ctc_blank_bias=0.0,
        best_metric='loss',
        allow_resume=True,
        allow_readonly_resume=True,
        require_source_checkpoint=True,
        allowed_missing_prefixes=('kd_aux_heads.', 'language_adapter.'),
        resume_tag=RECOVER_KD_RECIPE_TAG,
        # Do not spend another multi-hour KD epoch when a downstream CTC
        # checkpoint proves this stage already completed in an earlier session.
        skip_if_checkpoint_stages=(
            ('recover_ctc', RECOVER_CTC_RECIPE_TAG),),
        fail_fast_on_collapse=False,
    ),
    StageSpec(
        name='recover_ctc',
        source_checkpoint_stage='recover_kd',
        source_checkpoint_tag=(
            RECOVER_KD_RECIPE_TAG, LEGACY_RECOVER_KD_RECIPE_TAG),
        fallback_source_stages=('recover_ctc', 'frozen', 'unfrozen', 'chunk_ctc'),
        source_checkpoint_suffix='latest',
        fallback_source_suffix='best',
        # Protect the KD-aligned backbone while the randomly initialized
        # 5k-way projection learns CTC. The loss weight below scales the head
        # gradient, so its nominal LR is raised to preserve a useful effective
        # update without letting CTC dominate every backbone layer.
        lr=3e-5,
        ctc_head_lr=1e-3,
        warmup_steps=200,
        epochs=C.un_epochs,
        bs=C.un_bs,
        ga=C.un_ga,
        max_seconds=None,
        # Joint KD+CTC (design doc stage 2), not pure CTC. Cosine feature
        # supervision anchors the encoder while the CTC weight ramps gradually.
        a_kl=1.0,
        a_lang=C.a_lang,
        ctc_start_weight=0.10,
        ctc_end_weight=0.30,
        ctc_warmup_steps=1000,
        reset_ctc_head=True,
        # A -5 blank bias did not prevent collapse; it instead creates a
        # violent nonblank-to-blank transient. Neutral init is the tested
        # tiny-overfit path, while temporal canary logic watches real progress.
        ctc_blank_bias=0.0,
        best_metric='wer',
        allow_resume=True,
        allow_readonly_resume=True,
        require_source_checkpoint=True,
        allowed_missing_prefixes=('kd_aux_heads.', 'language_adapter.'),
        # The v9 tag invalidates stale pre-router/pure-CTC attempts and
        # guarantees a fresh head plus a metadata-checked KD backbone.
        resume_tag=RECOVER_CTC_RECIPE_TAG,
        # Optimizer/scaler resume is new-architecture only. A legacy
        # recover_ctc checkpoint may still enter through fallback_source_stages,
        # where only compatible weights are loaded and the CTC head is reset.
        resume_source_tags=(RECOVER_CTC_RECIPE_TAG,),
        fail_fast_on_collapse=True,
        # At 300 updates, fail only if a probability-confirmed collapse also
        # fails to improve validation loss by at least 5% from step zero.
        canary_updates=300,
        canary_min_relative_loss_drop=0.05,
    ),
]

if C.train_from_scratch:
    stage_specs = fresh_specs
    print('Training mode: FROM SCRATCH. Legacy recovery checkpoints are ignored.')
else:
    stage_specs = recovery_specs
    print('Training mode: LEGACY RECOVERY. Source checkpoints are required.')

stage_results = {}
all_done = True
for spec in stage_specs:
    if time_left_seconds() < 600:
        print(f'\nSession has < 10 min left. Save Version and run {spec.name} next session.')
        loss_log.log('session_pause', stage=spec.name, completed=False)
        all_done = False
        break

    print('\n' + '=' * 60)
    print(f'STAGE: {spec.name}  (time left: {fmt_hms(time_left_seconds())})')
    print('=' * 60)
    stage_loss, stage_done = train_stage(student, train_ds, spec, val_dataset=val_ds)
    stage_results[spec.name] = {'loss': stage_loss, 'done': stage_done}
    if not stage_done:
        print(f'\n{spec.name} paused. Save Version, reopen, and run again to resume.')
        loss_log.log('session_pause', stage=spec.name, completed=False, last_loss=stage_loss)
        all_done = False
        break

if all_done:
    print(f'\nTraining complete - {stage_results}')
    for _name, _result in stage_results.items():
        wandb_run.summary[f'final/{_name}_loss'] = _result['loss']
    loss_log.log('training_complete', stage_results=stage_results)

loss_log.close()
wandb_run.finish(exit_code=0, quiet=True)


### Test-set evaluation (run after training completes)

Loads the best recipe-compatible `stream_joint` checkpoint and evaluates the
source-disjoint test split. Pass 1 reports overall/per-language WER and CER.
Pass 2 benchmarks both full-chunk inference and bounded-memory blockwise
streaming (`C.stream_chunk_ms`, `C.stream_left_context_ms`) with synchronized
GPU timing and RTF. Mel extraction is excluded from both latency numbers.

Share `splits.json`, `predictions_<stage>.jsonl`, and `results_<stage>.json` for
fair comparisons on identical audio.


In [ ]:
# ── Test-set evaluation: WER + CER + Latency ──────────────────────
# Independent of the training cell's in-memory state — rebuilds the
# student from disk so it works even after a kernel restart.
import transformers as _transformers_mod
import json
import time
import numpy as np
import jiwer
from torch.utils.data import Subset, DataLoader

# 1. Load splits (test indices).
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
assert os.path.exists(SPLITS_PATH), (
    f'No splits.json at {SPLITS_PATH}. Run the training cell at least '
    'once first — it creates the splits.'
)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

# 2. Ensure dataset is loaded (might not be if kernel was restarted).
if 'ds' not in globals():
    print('Re-loading dataset...')
    ds = KDDataset(C.cache_dir, sp, C.mel_dir)
C.num_languages = len(ds.lang_to_id)
C.language_names = tuple(
    name for name, _ in sorted(ds.lang_to_id.items(), key=lambda item: item[1]))
_active_dataset_fingerprint = _dataset_fingerprint(ds)
if (splits.get('strategy') != SPLIT_STRATEGY or
        splits.get('fingerprint') != _active_dataset_fingerprint):
    raise RuntimeError(
        'Evaluation splits do not match the active cache: '
        f'saved={splits.get("strategy")}/{splits.get("fingerprint")}, '
        f'active={SPLIT_STRATEGY}/{_active_dataset_fingerprint}.')
test_ds = Subset(ds, splits['test_indices'])
print(f'Test set: {len(test_ds):,} samples '
      f'(fingerprint={splits.get("fingerprint")})')

# 3. Find the best compatible checkpoint. Prefer validated `best` weights;
# fall back to `latest` only while a stage has not produced a best snapshot.
# Search the writable session directory first within each suffix, then the
# read-only uploaded checkpoint dataset so evaluation survives a kernel restart.
# Evaluation may inspect a checkpoint produced by the immediately
# preceding CTC recipe. This compatibility exception is intentionally local
# to evaluation: training resume still requires the exact current tag.
if C.train_from_scratch:
    _candidate_specs = [
        ('stream_joint', 'best', STREAM_JOINT_RECIPE_TAG),
        ('stream_joint', 'latest', STREAM_JOINT_RECIPE_TAG),
    ]
else:
    # Legacy checkpoints are migration inputs, not measurements of this model.
    # Evaluate only a checkpoint saved after the router/architecture refactor.
    _candidate_specs = [
        ('recover_ctc', 'best', RECOVER_CTC_RECIPE_TAG),
        ('recover_ctc', 'latest', RECOVER_CTC_RECIPE_TAG),
    ]
stage = best_path = None
_state = None
for _stage, _suffix, _required_tag in _candidate_specs:
    for _path in _load_candidates(_stage, suffix=_suffix, include_readonly=True):
        _candidate, _loaded_from = try_load_compatible_checkpoint(
            _path, expected_tag=_required_tag,
            purpose=f'{_stage} evaluation checkpoint')
        if _candidate is None:
            continue
        if _candidate.get('tokenizer_fingerprint') != TOKENIZER_FINGERPRINT:
            print(f'Ignoring evaluation checkpoint {_loaded_from}: tokenizer '
                  'fingerprint is missing or incompatible.')
            continue
        try:
            _require_checkpoint_model_metadata(_candidate, _loaded_from)
        except RuntimeError as _metadata_error:
            print(f'Ignoring evaluation checkpoint {_loaded_from}: {_metadata_error}')
            continue
        stage, best_path, _state = _stage, _loaded_from, _candidate
        break
    if _state is not None:
        break
if best_path is None:
    # A session-budget pause is a normal resumable state, not an evaluation
    # failure. In particular, a KD-only checkpoint has a deliberately random
    # CTC head, so evaluating it would produce meaningless WER. Papermill should
    # finish cleanly and let the next Kaggle Version resume training instead.
    EVALUATION_SKIPPED = True
    print(
        '\nEvaluation skipped: training has not produced a recipe- and '
        'tokenizer-compatible CTC checkpoint yet.'
    )
    print(
        'The KD-only checkpoint remains resumable. Save this Kaggle Version, '
        'attach its output checkpoint dataset, and rerun the training cell '
        'to continue the active KD-to-joint curriculum before evaluation.'
    )
else:
    EVALUATION_SKIPPED = False
    print(f'Using checkpoint: {best_path} (stage={stage})')
    if (_state.get('split_strategy') != splits.get('strategy') or
            _state.get('split_fingerprint') != splits.get('fingerprint')):
        raise RuntimeError(
            'Checkpoint/test-split mismatch: '
            f'checkpoint={_state.get("split_strategy")}/'
            f'{_state.get("split_fingerprint")}, '
            f'active={splits.get("strategy")}/{splits.get("fingerprint")}.')

    # 4. Build a fresh student and load weights.
    # Always load into the un-wrapped model; strip DataParallel `module.` prefix
    # from saved state-dict if present.
    test_student = ConMambaStudent(C).to(device)
    _require_checkpoint_tokenizer(_state, best_path)
    _require_checkpoint_model_metadata(_state, best_path)
    _sd = _state['model']
    # Normalize the on-disk state-dict to no-prefix form. New checkpoints
    # (post-v7) are already in this form, so this is a no-op then. Legacy
    # DP-saved checkpoints have `module.` prefixes which we strip. The old
    # inline comprehension had a bug — it filtered out non-prefixed keys,
    # which would silently zero-init half the model on mixed dicts.
    if any(k.startswith('module.') for k in _sd):
        _sd = {(k[len('module.'):] if k.startswith('module.') else k): v
               for k, v in _sd.items()}
    _dropped = _load_model_tolerant(test_student, _sd)
    if _dropped:
        raise RuntimeError(
            f'Evaluation refused to use shape-mismatched weights: {_dropped}')
    test_student.eval()
    print(f'  checkpoint meta: epoch={_state.get("epoch")}, '
          f'global_step={_state.get("global_step")}, '
          f'loss={_state.get("loss", _state.get("best_loss", float("nan"))):.4f}')
    evaluation_lineage_clean = bool(_state.get('evaluation_lineage_clean', False))
    if not evaluation_lineage_clean:
        print('WARNING: this checkpoint descends from legacy per-chunk-split '
              'weights. Accuracy is diagnostic only, not an unbiased benchmark. '
              'Train a fresh source-grouped lineage for publishable test metrics.')

    # 5. Set up output paths.
    results_dir = os.path.join(C.ckpt_dir, 'test_results')
    os.makedirs(results_dir, exist_ok=True)
    pred_path = os.path.join(results_dir, f'predictions_{stage}.jsonl')
    results_path = os.path.join(results_dir, f'results_{stage}.json')

    # ── Pass 1: accuracy (WER + CER over full test set, bs=2) ───────
    print(f'\nPass 1/2: WER + CER on {len(test_ds):,} test samples...')
    max_seconds = None
    max_mel_frames = None

    acc_sampler = ShardBucketBatchSampler(
        test_ds, batch_size=2, drop_last=False, shuffle=False,
    )
    acc_loader = DataLoader(
        test_ds, batch_sampler=acc_sampler,
        num_workers=C.workers, persistent_workers=False,
        pin_memory=True,
        collate_fn=lambda b: collate_kd(b, max_mel_frames=max_mel_frames, allow_crop=False),
    )

    refs, hyps, audio_seconds_list, langs, pred_id_seqs, frame_argmax_seqs = [], [], [], [], [], []
    with open(pred_path, 'w') as pf, torch.no_grad():
        for batch in tqdm(acc_loader, desc='accuracy', unit='batch'):
            mel = batch['mel'].to(device, non_blocking=True)
            mel_lens = batch['mel_lens'].to(device, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                ctc_logits, _lang, enc_lens = test_student.inference_ctc(mel, mel_lens)

            logits_cpu = ctc_logits.float().cpu()
            lengths = enc_lens.cpu().tolist()
            pred_ids = greedy_ctc_token_ids(logits_cpu, BLANK, lengths=lengths)
            frame_argmax = logits_cpu.argmax(dim=-1)
            for seq, valid_len in zip(frame_argmax, lengths):
                frame_argmax_seqs.append(seq[:valid_len].tolist())
            decoded = [sp.DecodeIds(ids) for ids in pred_ids]
            pred_id_seqs.extend(pred_ids)
            batch_secs = batch.get('audio_seconds', [None] * len(decoded))
            batch_langs = batch.get('langs', ['unknown'] * len(decoded))
            for ref, hyp, sec, lang in zip(
                    batch['texts'], decoded, batch_secs, batch_langs):
                refs.append(ref)
                hyps.append(hyp)
                langs.append(lang)
                sec_val = float(sec) if sec is not None else None
                if sec_val is not None:
                    audio_seconds_list.append(sec_val)
                pf.write(json.dumps(
                    {'ref': ref, 'hyp': hyp, 'lang': lang, 'audio_seconds': sec_val},
                    ensure_ascii=False,
                ) + '\n')

    # Compute WER and CER. Same jiwer text-normalization is applied to both.
    wer = jiwer.wer(refs, hyps)
    cer = jiwer.cer(refs, hyps)
    collapse_metrics = compute_collapse_metrics(
        pred_id_seqs, BLANK, frame_argmax_ids=frame_argmax_seqs,
        refs=refs, hyps=hyps, sp_model=sp,
    )
    empty_hyp_pct = collapse_metrics['empty_hypothesis_pct']
    pred_lens = [len(ids) for ids in pred_id_seqs]
    ref_lens = [len(sp.EncodeAsIds(r)) for r in refs]
    print(f'  WER = {wer*100:.2f}%   CER = {cer*100:.2f}%   (n={len(refs)})')
    print(f'  Collapse: {collapse_metrics}  empty_hyp={empty_hyp_pct*100:.1f}%')

    # Per-language breakdown. WER is unreliable for space-free scripts (zh/ja);
    # CER is the trustworthy column there.
    per_language = {}
    for _lang in sorted(set(langs)):
        _idx = [i for i, l in enumerate(langs) if l == _lang]
        _r = [refs[i] for i in _idx]
        _h = [hyps[i] for i in _idx]
        try:
            _w = float(jiwer.wer(_r, _h))
        except Exception:
            _w = float('nan')
        try:
            _c = float(jiwer.cer(_r, _h))
        except Exception:
            _c = float('nan')
        per_language[_lang] = {'wer': _w, 'cer': _c, 'n': len(_idx)}
    print('  Per-language:')
    for _lang, _m in per_language.items():
        print(
            f'    {
                _lang:32s} CER={
                _m["cer"] *
                100:6.2f}%  WER={
                    _m["wer"] *
                    100:7.2f}%  (n={
                        _m["n"]})'
            )

    # ── Pass 2: full and bounded-streaming latency at bs=1 ──────────
    N_LAT = min(int(getattr(C, 'stream_eval_samples', 200)), len(test_ds))
    print(f'\nPass 2/2: full + streaming latency over {N_LAT} samples '
          f'(bs=1, fp16, chunk={C.stream_chunk_ms} ms, '
          f'left={C.stream_left_context_ms} ms)...')

    lat_subset = Subset(test_ds, list(range(N_LAT)))
    lat_loader = DataLoader(
        lat_subset, batch_size=1, shuffle=False, num_workers=2,
        pin_memory=True, drop_last=False,
        collate_fn=lambda b: collate_kd(b, max_mel_frames=max_mel_frames, allow_crop=False),
    )

    # Warmup (first few forwards trigger Triton JIT / kernel caching).
    with torch.no_grad():
        for i, batch in enumerate(lat_loader):
            if i >= 10:
                break
            mel = batch['mel'].to(device, non_blocking=True)
            mel_lens = batch['mel_lens'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                test_student.inference_ctc(mel, mel_lens)
                test_student.streaming_ctc(mel, mel_lens)
            torch.cuda.synchronize()

    # Measure — torch.cuda.synchronize() before and after to exclude
    # data-load + queue time from the measurement.
    latencies_ms, streaming_latencies_ms, sample_audio_secs = [], [], []
    stream_refs, stream_hyps, stream_langs = [], [], []
    stream_pred_ids, stream_frame_argmax = [], []
    with torch.no_grad():
        for batch in tqdm(lat_loader, desc='latency', unit='sample'):
            mel = batch['mel'].to(device, non_blocking=True)
            mel_lens = batch['mel_lens'].to(device, non_blocking=True)
            sec = batch.get('audio_seconds', [None])
            sec_val = float(sec[0]) if sec and sec[0] is not None else None

            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                test_student.inference_ctc(mel, mel_lens)
            torch.cuda.synchronize()
            latencies_ms.append((time.perf_counter() - t0) * 1000)

            torch.cuda.synchronize()
            stream_t0 = time.perf_counter()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                stream_logits, _stream_lang_logits, stream_lens = (
                    test_student.streaming_ctc(mel, mel_lens))
            torch.cuda.synchronize()
            streaming_latencies_ms.append(
                (time.perf_counter() - stream_t0) * 1000)
            sample_audio_secs.append(sec_val)

            stream_logits_cpu = stream_logits.float().cpu()
            stream_lengths = stream_lens.cpu().tolist()
            batch_stream_ids = greedy_ctc_token_ids(
                stream_logits_cpu, BLANK, lengths=stream_lengths)
            stream_pred_ids.extend(batch_stream_ids)
            frame_ids = stream_logits_cpu.argmax(dim=-1)
            for seq, valid_len in zip(frame_ids, stream_lengths):
                stream_frame_argmax.append(seq[:valid_len].tolist())
            stream_refs.extend(batch['texts'])
            stream_hyps.extend(sp.DecodeIds(ids) for ids in batch_stream_ids)
            stream_langs.extend(batch.get('langs', ['unknown']))

    lat_arr = np.array(latencies_ms)
    stream_lat_arr = np.array(streaming_latencies_ms)

    # Real-time factor: latency_seconds / audio_seconds. <1.0 is faster than
    # real-time. Reported only if audio_seconds is available.
    rtf_stats = None
    if all(s is not None for s in sample_audio_secs) and sample_audio_secs:
        sec_arr = np.array(sample_audio_secs)
        rtf = (lat_arr / 1000.0) / sec_arr
        streaming_rtf = (stream_lat_arr / 1000.0) / sec_arr
        rtf_stats = {
            'full': {
                'mean': float(rtf.mean()),
                'p50': float(np.percentile(rtf, 50)),
                'p95': float(np.percentile(rtf, 95)),
            },
            'streaming': {
                'mean': float(streaming_rtf.mean()),
                'p50': float(np.percentile(streaming_rtf, 50)),
                'p95': float(np.percentile(streaming_rtf, 95)),
            },
        }

    stream_wer = float(jiwer.wer(stream_refs, stream_hyps))
    stream_cer = float(jiwer.cer(stream_refs, stream_hyps))
    stream_collapse = compute_collapse_metrics(
        stream_pred_ids, BLANK, frame_argmax_ids=stream_frame_argmax,
        refs=stream_refs, hyps=stream_hyps, sp_model=sp,
    )
    stream_per_language = {}
    for _lang in sorted(set(stream_langs)):
        _idx = [i for i, value in enumerate(stream_langs) if value == _lang]
        _r = [stream_refs[i] for i in _idx]
        _h = [stream_hyps[i] for i in _idx]
        stream_per_language[_lang] = {
            'wer': float(jiwer.wer(_r, _h)),
            'cer': float(jiwer.cer(_r, _h)),
            'n': len(_idx),
        }

    # Assemble result blob.
    results = {
        'stage':           stage,
        'checkpoint_path': best_path,
        'checkpoint_meta': {
            'epoch':       _state.get('epoch'),
            'global_step': _state.get('global_step'),
            'loss':        float(_state.get('loss', _state.get('best_loss', float('nan')))),
        },
        'n_samples_accuracy': len(refs),
        'n_samples_latency':  len(latencies_ms),
        'wer': float(wer),
        'cer': float(cer),
        'per_language': per_language,
        'collapse': collapse_metrics,
        'bounded_streaming_accuracy': {
            'n': len(stream_refs),
            'wer': stream_wer,
            'cer': stream_cer,
            'per_language': stream_per_language,
            'collapse': stream_collapse,
        },
        'evaluation_valid_for_unbiased_comparison': evaluation_lineage_clean,
        'checkpoint_split_strategy': _state.get('split_strategy'),
        'checkpoint_split_fingerprint': _state.get('split_fingerprint'),
        'active_split_strategy': splits.get('strategy'),
        'active_split_fingerprint': splits.get('fingerprint'),
        'empty_hypothesis_pct': empty_hyp_pct,
        'avg_pred_token_len': float(np.mean(pred_lens)) if pred_lens else 0.0,
        'avg_ref_token_len': float(np.mean(ref_lens)) if ref_lens else 0.0,
        'latency_ms': {
            'full_chunk': {
                'mean': float(lat_arr.mean()),
                'std': float(lat_arr.std()),
                'p50': float(np.percentile(lat_arr, 50)),
                'p95': float(np.percentile(lat_arr, 95)),
                'p99': float(np.percentile(lat_arr, 99)),
                'min': float(lat_arr.min()),
                'max': float(lat_arr.max()),
            },
            'bounded_streaming': {
                'mean': float(stream_lat_arr.mean()),
                'std': float(stream_lat_arr.std()),
                'p50': float(np.percentile(stream_lat_arr, 50)),
                'p95': float(np.percentile(stream_lat_arr, 95)),
                'p99': float(np.percentile(stream_lat_arr, 99)),
                'min': float(stream_lat_arr.min()),
                'max': float(stream_lat_arr.max()),
            },
        },
        'system': {
            'gpu_name':              torch.cuda.get_device_name(0),
            'precision':             'fp16 (autocast)',
            'torch_version':         torch.__version__,
            'transformers_version':  _transformers_mod.__version__,
        },
        'config': {
            'batch_size_accuracy':  2,
            'batch_size_latency':   1,
            'max_audio_seconds':    max_seconds,
            'student_params_M':     sum(p.numel() for p in test_student.parameters()) / 1e6,
            'stream_chunk_ms':      C.stream_chunk_ms,
            'stream_left_context_ms': C.stream_left_context_ms,
            'note': ('Latency = student forward only; mel precomputed. '
                     'bounded_streaming reprocesses fixed left context.'),
        },
        'splits': {
            'splits_path':  SPLITS_PATH,
            'fingerprint':  splits.get('fingerprint'),
            'seed':         splits.get('seed'),
            'test_size':    splits.get('test_size'),
        },
    }
    if rtf_stats is not None:
        results['rtf'] = rtf_stats

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    # ── Print summary ──
    print(f'\n{"=" * 60}')
    print(f'TEST SET EVALUATION ({stage})')
    print(f'{"=" * 60}')
    print(f'  WER          : {results["wer"]*100:7.2f}%')
    print(f'  CER          : {results["cer"]*100:7.2f}%')
    print(f'  Stream WER   : {stream_wer*100:7.2f}% (n={len(stream_refs)})')
    print(f'  Stream CER   : {stream_cer*100:7.2f}%')
    print(f'  Unbiased eval: {results["evaluation_valid_for_unbiased_comparison"]}')
    _full_lat = results['latency_ms']['full_chunk']
    _stream_lat = results['latency_ms']['bounded_streaming']
    print(f'  Full latency : mean={_full_lat["mean"]:7.1f} ms '
          f'p95={_full_lat["p95"]:7.1f} ms')
    print(f'  Stream latency: mean={_stream_lat["mean"]:7.1f} ms '
          f'p95={_stream_lat["p95"]:7.1f} ms')
    if rtf_stats is not None:
        print(f'  Full RTF     : {rtf_stats["full"]["mean"]:7.3f}')
        print(f'  Stream RTF   : {rtf_stats["streaming"]["mean"]:7.3f}  '
              '(student forward only; <1.0 = faster than real-time)')
    print(f'\nArtifacts saved (use these to compare other methods on the same set):')
    print(f'  splits.json      : {SPLITS_PATH}')
    print(f'  predictions.jsonl: {pred_path}')
    print(f'  results.json     : {results_path}')

    del test_student
    torch.cuda.empty_cache()


### CTC output diagnostic (run if WER stays at 100%)

After training has saved at least one checkpoint, this cell loads it, runs the model on 10 validation samples, and prints the per-frame argmax distribution. WER=100% has four possible causes — this output tells you which:

- **Blank-dominant collapse** (>95% of frames argmax to BLANK): expected during frozen stage. The CTC head can't overcome the trivial "predict blank everywhere" local minimum without backbone gradients. Not a bug — move to unfrozen stage.
- **Mode collapse** on a single non-blank token: training instability, often a vocab/temperature issue.
- **Diverse outputs but wrong text**: model is learning, just hasn't converged yet on alignment.
- **Tokenizer mismatch**: token IDs look right but decoded text doesn't match — bug in the encode/decode round-trip.


In [ ]:
# ── CTC output diagnostic ────────────────────────────────────────
# Set RUN_CTC_DIAGNOSTIC=True and rerun this cell if WER/CER collapse. The
# diagnostic loads a fresh model so it never mutates the in-memory trainer.
RUN_CTC_DIAGNOSTIC = globals().get('RUN_CTC_DIAGNOSTIC', False)

if not RUN_CTC_DIAGNOSTIC:
    print('RUN_CTC_DIAGNOSTIC=False; enable it only after a stream checkpoint exists.')
else:
    from collections import Counter
    from torch.utils.data import DataLoader, Subset

    _diag_candidates = []
    for _stage, _suffix in (
        ('stream_joint', 'best'),
        ('stream_joint', 'latest'),
        ('stream_kd', 'best'),
        ('stream_kd', 'latest'),
    ):
        _diag_candidates.extend(_load_candidates(_stage, _suffix))
    _state, ckpt_path = try_load_full_checkpoint(_diag_candidates)
    if _state is None:
        raise FileNotFoundError(
            'No StreamConMamba checkpoint found. Complete at least part of '
            'stream_joint (or stream_kd) first.')

    _require_checkpoint_tokenizer(_state, ckpt_path, allow_missing=False)
    _require_checkpoint_model_metadata(_state, ckpt_path, allow_missing=False)
    diag = ConMambaStudent(C).to(device)
    _load_model_tolerant(diag, _normalize_state_dict(_state['model']))
    diag.eval()
    print(
        f'Loaded {ckpt_path}: stage={_state.get("stage_name")}, '
        f'epoch={_state.get("epoch")}, step={_state.get("global_step")}')
    print(
        f'Vocabulary: SentencePiece={sp.GetPieceSize()}, BLANK={BLANK}, '
        f'ctc_outputs={C.vocab_size}')
    assert BLANK == sp.GetPieceSize()
    assert C.vocab_size == BLANK + 1

    _diag_count = min(10, len(test_ds))
    if _diag_count == 0:
        raise RuntimeError('The held-out test split is empty.')
    loader = DataLoader(
        Subset(test_ds, list(range(_diag_count))),
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=lambda b: collate_kd(
            b, max_mel_frames=None, allow_crop=False),
    )

    all_argmax = Counter()
    all_pred_ids, all_frame_ids = [], []
    all_hyps, all_refs = [], []
    feature_variances = []
    for i, batch in enumerate(loader):
        mel = batch['mel'].to(device)
        mel_lens = batch['mel_lens'].to(device)
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
            ctc_logits, kd_features, language_logits, enc_lens = diag(
                mel, mel_lens)

        valid_len = int(enc_lens[0].item())
        frame_ids = ctc_logits[0, :valid_len].float().argmax(dim=-1).cpu().tolist()
        counter = Counter(frame_ids)
        all_argmax.update(counter)
        all_frame_ids.append(frame_ids)
        pred_ids = greedy_ctc_token_ids(
            ctc_logits.float().cpu(), BLANK, lengths=[valid_len])[0]
        ref = batch['texts'][0]
        hyp = sp.DecodeIds(pred_ids)
        all_pred_ids.append(pred_ids)
        all_refs.append(ref)
        all_hyps.append(hyp)

        final_kd = kd_features[-1][0, :valid_len].float()
        feature_variances.append(final_kd.var(dim=0).mean().item())
        valid_language_logits = language_logits[0, :valid_len].float().mean(dim=0)
        routed_id = int(valid_language_logits.argmax().item())
        routed_name = C.language_names[routed_id]
        blank_pct = 100.0 * counter.get(BLANK, 0) / max(valid_len, 1)
        print(
            f'[{i}] frames={valid_len} blank={blank_pct:5.1f}% '
            f'router={routed_name!r}\n  REF: {ref[:100]}\n'
            f'  HYP: {hyp[:100] if hyp else "(empty — only blanks)"}')

    metrics = compute_collapse_metrics(
        all_pred_ids,
        BLANK,
        frame_argmax_ids=all_frame_ids,
        refs=all_refs,
        hyps=all_hyps,
        sp_model=sp,
    )
    print('\nCTC diagnosis:')
    print(
        f'  collapsed={metrics["collapsed"]} reason={metrics["reason"]} '
        f'blank_frames={metrics["blank_frame_pct"] * 100:.1f}% '
        f'empty_hypotheses={metrics["empty_hypothesis_pct"] * 100:.1f}%')
    print(
        f'  final KD-head frame variance='
        f'{sum(feature_variances) / len(feature_variances):.8e}')
    if metrics['reason'] == 'blank_dominant':
        print('  Action: reset the CTC head and restart the stream_joint stage.')
    elif metrics['reason'] == 'nonblank_mode':
        print('  Action: inspect tokenizer IDs, confidence weights, and CTC LR.')
    elif metrics['reason'] == 'empty_hypothesis':
        print('  Action: inspect CTC lengths and blank bias before more training.')
    else:
        print('  Alignment is forming; compare held-out WER/CER across checkpoints.')

    del diag
    torch.cuda.empty_cache()
